<a href="https://colab.research.google.com/github/ghadirymaryam1-coder/PFMS-Net-Research2026/blob/main/PFMSPARALLEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch, os, shutil

print("Python environment ready")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("Local free space (GB):", round(shutil.disk_usage("/content").free / 1e9, 2))
print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))

Python environment ready
PyTorch: 2.11.0+cu130
CUDA: 13.0
GPU: Tesla T4
Local free space (GB): 75.31
Drive mounted: False


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

paths = {
    "GT CSV": "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv",
    "Vocabulary": "/content/drive/MyDrive/idpl_vocab.pt",
    "Images": "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl",
}

for name, path in paths.items():
    print(f"{name}: {'FOUND' if os.path.exists(path) else 'MISSING'}")

GT CSV: FOUND
Vocabulary: FOUND
Images: FOUND


In [7]:
%pip install --no-cache-dir --no-deps --force-reinstall "causal-conv1d==1.6.2.post1+cu.13.0.torch.2.11" --index-url https://wheels.astral.sh/simple/cu130/

Looking in indexes: https://wheels.astral.sh/simple/cu130/
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.6/205.6 MB 155.5 MB/s  0:00:01


In [8]:
%pip install --no-cache-dir --no-deps --force-reinstall "mamba-ssm==2.3.2.post1+cu.13.0.torch.2.11" --index-url https://wheels.astral.sh/simple/cu130/

Looking in indexes: https://wheels.astral.sh/simple/cu130/
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.4/349.4 MB 186.3 MB/s  0:00:01


In [10]:
%pip install --no-cache-dir --no-deps mambavision==1.2.0

In [11]:
from mambavision import create_model
import torch
import causal_conv1d
import mamba_ssm

model = create_model("mamba_vision_T", pretrained=False).cuda().eval()

with torch.no_grad():
    y = model(torch.randn(1, 3, 224, 224, device="cuda"))

print("MambaVision: OK")
print("GPU:", torch.cuda.get_device_name(0))
print("Output shape:", y.shape)
print("Finite output:", torch.isfinite(y).all().item())

del model, y
torch.cuda.empty_cache()

/usr/local/lib/python3.13/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


MambaVision: OK
GPU: Tesla T4
Output shape: torch.Size([1, 1000])
Finite output: True


In [12]:
import os

paths = {
    "GT": "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv",
    "Vocabulary": "/content/drive/MyDrive/idpl_vocab.pt",
    "Images": "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl",
}

for name, path in paths.items():
    print(f"{name}: {'OK' if os.path.exists(path) else 'NOT FOUND'}")

GT: OK
Vocabulary: OK
Images: OK


In [16]:
# ============================================================
# PFMS-Net Ablation | Cell 2: Spatial feature-map verification
# Real MambaVision-T + original IDPL-PFOD images
# ============================================================
import os
import torch
import pandas as pd
import numpy as np
from PIL import Image
from collections import Counter
from mambavision import create_model

GT_PATH = "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv"
VOCAB_PATH = "/content/drive/MyDrive/idpl_vocab.pt"
OUT_DIR = "/content/PFMS_ABLATION"
os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gt = pd.read_csv(GT_PATH)
train_df = gt[gt["split"].astype(str).str.lower() == "train"].reset_index(drop=True)
test_df = gt[gt["split"].astype(str).str.lower() == "test"].reset_index(drop=True)

# Inspect the saved vocabulary without assuming its internal format.
vocab_obj = torch.load(VOCAB_PATH, map_location="cpu", weights_only=False)
print("Saved vocabulary object type:", type(vocab_obj).__name__)

if isinstance(vocab_obj, dict):
    print("Vocabulary keys (first 30):", list(vocab_obj.keys())[:30])
    for k, v in list(vocab_obj.items())[:10]:
        if isinstance(v, dict):
            print(f"  {k}: dict, entries={len(v)}")
        elif isinstance(v, (list, tuple)):
            print(f"  {k}: {type(v).__name__}, length={len(v)}")
        else:
            print(f"  {k}: {type(v).__name__}", str(v)[:120])
elif isinstance(vocab_obj, (list, tuple)):
    print("Vocabulary length:", len(vocab_obj))
else:
    print("Vocabulary preview:", str(vocab_obj)[:500])

print(f"\nTrain rows: {len(train_df)} | Test rows: {len(test_df)}")
print("Unique training characters:", len(set("".join(train_df["text"].astype(str)))))

# Load a real image and inspect dimensions.
sample_path = str(train_df.iloc[0]["image_path"])
if not os.path.isfile(sample_path):
    raise FileNotFoundError(f"Image path from GT does not exist: {sample_path}")

with Image.open(sample_path) as im:
    im = im.convert("RGB")
    print("\nSample image:", os.path.basename(sample_path))
    print("Original image size (W,H):", im.size)

# Build the actual MambaVision-T backbone.
mamba = create_model("mamba_vision_T", pretrained=False).to(device).eval()
captured = {}

def capture_stage(name):
    def _hook(module, inputs, output):
        captured[name] = output
    return _hook

handles = []
for i, stage in enumerate(mamba.levels):
    handles.append(stage.register_forward_hook(capture_stage(f"stage_{i+1}")))

# Probe with the original aspect ratio scaled down only for this shape test.
# This is not training and does not resize any dataset files on disk.
probe_h, probe_w = 64, 448
x = torch.randn(1, 3, probe_h, probe_w, device=device)

with torch.no_grad():
    _ = mamba(x)

for h in handles:
    h.remove()

def describe_output(obj):
    if torch.is_tensor(obj):
        return f"Tensor{tuple(obj.shape)}"
    if isinstance(obj, (tuple, list)):
        return type(obj).__name__ + "[" + ", ".join(describe_output(z) for z in obj) + "]"
    return type(obj).__name__

print("\nMambaVision internal stage outputs:")
for name, obj in captured.items():
    print(name, "->", describe_output(obj))

# Choose a spatial tensor from the deepest stage output.
last_output = captured[f"stage_{len(mamba.levels)}"]
candidates = []
def collect_spatial(obj):
    if torch.is_tensor(obj) and obj.ndim == 4:
        candidates.append(obj)
    elif isinstance(obj, (tuple, list)):
        for z in obj:
            collect_spatial(z)
collect_spatial(last_output)

if not candidates:
    raise RuntimeError(
        "No 4D spatial feature map was exposed by the last stage. "
        "Do not train using the pooled 640-D classifier features."
    )

feature_map = max(candidates, key=lambda z: z.shape[1])
print("\nSelected spatial feature map:", tuple(feature_map.shape))
print("Spatial width (sequence axis):", feature_map.shape[-1])
print("Spatial height:", feature_map.shape[-2])
print("Feature channels:", feature_map.shape[1])

# Save only a small text report locally.
report_path = os.path.join(OUT_DIR, "mambavision_feature_probe.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write(f"Train rows: {len(train_df)}\n")
    f.write(f"Test rows: {len(test_df)}\n")
    f.write(f"Probe input: {tuple(x.shape)}\n")
    f.write(f"Selected feature map: {tuple(feature_map.shape)}\n")
    f.write(f"Vocabulary type: {type(vocab_obj).__name__}\n")

print("\nProbe report saved locally:", report_path)
print("No checkpoint or feature cache was written to Google Drive.")

Saved vocabulary object type: dict
Vocabulary keys (first 30): ['char2idx', 'idx2char', 'characters', 'PAD_IDX', 'BOS_IDX', 'EOS_IDX', 'UNK_IDX', 'vocab_size']
  char2idx: dict, entries=169
  idx2char: dict, entries=169
  characters: list, length=165
  PAD_IDX: int 0
  BOS_IDX: int 1
  EOS_IDX: int 2
  UNK_IDX: int 3
  vocab_size: int 169

Train rows: 23054 | Test rows: 4066
Unique training characters: 164

Sample image: 00490.tif
Original image size (W,H): (700, 50)

MambaVision internal stage outputs:
stage_1 -> Tensor(1, 160, 8, 56)
stage_2 -> Tensor(1, 320, 4, 28)
stage_3 -> Tensor(1, 640, 2, 14)
stage_4 -> Tensor(1, 640, 2, 14)

Selected spatial feature map: (1, 640, 2, 14)
Spatial width (sequence axis): 14
Spatial height: 2
Feature channels: 640

Probe report saved locally: /content/PFMS_ABLATION/mambavision_feature_probe.txt
No checkpoint or feature cache was written to Google Drive.


In [17]:
# ============================================================
# PFMS-Net Ablation | Cell 3: OCR data pipeline + CTC sanity test
# ============================================================
import os, gc, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from mambavision import create_model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda")
GT_PATH = "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv"
VOCAB_PATH = "/content/drive/MyDrive/idpl_vocab.pt"
OUT_DIR = "/content/PFMS_ABLATION"
os.makedirs(OUT_DIR, exist_ok=True)

gt = pd.read_csv(GT_PATH)
train_df = gt[gt["split"].astype(str).str.lower() == "train"].reset_index(drop=True)
test_df = gt[gt["split"].astype(str).str.lower() == "test"].reset_index(drop=True)

vocab = torch.load(VOCAB_PATH, map_location="cpu", weights_only=False)
char2idx = vocab["char2idx"]
PAD_IDX = int(vocab["PAD_IDX"])
BOS_IDX = int(vocab["BOS_IDX"])
EOS_IDX = int(vocab["EOS_IDX"])
UNK_IDX = int(vocab["UNK_IDX"])
VOCAB_SIZE = int(vocab["vocab_size"])

# CTC blank must be distinct from actual target characters.
# PAD=0 is used as CTC blank; BOS/EOS are excluded from targets.
CTC_BLANK = PAD_IDX

# Resize the line to a fixed landscape size, keeping the line layout.
# Image files on Drive are never modified.
IMG_H, IMG_W = 64, 896
transform = transforms.Compose([
    transforms.Resize((IMG_H, IMG_W)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])

class IDPLLineDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        path = str(row["image_path"])
        text = str(row["text"]).replace("\n", " ").strip()
        with Image.open(path) as im:
            image = transform(im.convert("RGB"))

        # CTC targets: character IDs only, no BOS/EOS.
        target = [char2idx.get(ch, UNK_IDX) for ch in text]
        target = [t for t in target if t not in (PAD_IDX, BOS_IDX, EOS_IDX)]
        return image, torch.tensor(target, dtype=torch.long), text, path

def ctc_collate(batch):
    images, targets, texts, paths = zip(*batch)
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    flat_targets = torch.cat(targets) if target_lengths.sum().item() else torch.empty(0, dtype=torch.long)
    return torch.stack(images), flat_targets, target_lengths, list(texts), list(paths)

# Keep test data separate; no test examples used for optimization.
train_ds = IDPLLineDataset(train_df)
test_ds = IDPLLineDataset(test_df)

train_loader = DataLoader(
    train_ds, batch_size=4, shuffle=True, num_workers=2,
    collate_fn=ctc_collate, pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=4, shuffle=False, num_workers=2,
    collate_fn=ctc_collate, pin_memory=True
)

# Real MambaVision backbone
backbone = create_model("mamba_vision_T", pretrained=False).to(device).eval()
stage4 = backbone.levels[-1]
captured = {}

def save_stage_output(module, inputs, output):
    captured["feat"] = output

hook = stage4.register_forward_hook(save_stage_output)

# Use one real batch to verify CTC dimensions before training.
images, targets, target_lengths, texts, paths = next(iter(train_loader))
images = images.to(device, non_blocking=True)

with torch.no_grad():
    _ = backbone(images)
hook.remove()

feat = captured["feat"]
if not torch.is_tensor(feat) or feat.ndim != 4:
    raise RuntimeError(f"Expected spatial feature map [B,C,H,W], got {type(feat)}")

# Convert spatial map [B,C,H,W] -> sequence [T,B,C].
# Pool only vertical dimension, preserving the horizontal sequence.
feat = feat.mean(dim=2).permute(2, 0, 1).contiguous()
sequence_length = feat.shape[0]
batch_size = feat.shape[1]

print("Vocabulary size:", VOCAB_SIZE)
print("Image batch:", tuple(images.shape))
print("MambaVision sequence [T,B,C]:", tuple(feat.shape))
print("CTC blank index:", CTC_BLANK)
print("Target lengths:", target_lengths.tolist())
print("Text samples:", texts[:2])

if sequence_length < int(target_lengths.max().item()):
    print(
        "\nWARNING: CTC sequence is shorter than at least one target. "
        "Do not start training with this output stride. We must increase "
        "the horizontal sequence resolution before training."
    )
else:
    logits = torch.randn(sequence_length, batch_size, VOCAB_SIZE, device=device)
    log_probs = logits.log_softmax(dim=-1)
    input_lengths = torch.full((batch_size,), sequence_length, dtype=torch.long)
    loss_fn = nn.CTCLoss(blank=CTC_BLANK, zero_infinity=True)
    loss = loss_fn(log_probs, targets.to(device), input_lengths, target_lengths)
    print("CTC sanity loss finite:", bool(torch.isfinite(loss)))
    print("CTC sanity loss:", float(loss))

del backbone, images, feat, captured
gc.collect()
torch.cuda.empty_cache()
print("\nData pipeline verified. No training was started in this cell.")

Vocabulary size: 169
Image batch: (4, 3, 64, 896)
MambaVision sequence [T,B,C]: (28, 4, 640)
CTC blank index: 0
Target lengths: [67, 77, 72, 72]
Text samples: ['دشمنان در كمین هستند و تا ظهور امام عصر این انقلاب را به پیش ببریم.', 'به شهروندان مهم و قابل تحسین دانست. به گزارش تحریرنو از روابط عمومی بانک شهر،']


Data pipeline verified. No training was started in this cell.


In [18]:
# ============================================================
# MambaVision OCR: find a suitable spatial stage for CTC
# No training and no Drive writes
# ============================================================
import torch
import gc
from mambavision import create_model

device = "cuda"
model = create_model("mamba_vision_T", pretrained=False).to(device).eval()

captured = {}
handles = []

def make_hook(name):
    def hook(module, inputs, output):
        captured[name] = output
    return hook

for i, stage in enumerate(model.levels):
    handles.append(stage.register_forward_hook(make_hook(f"stage_{i+1}")))

# 64x896 line image, matching the tested OCR input shape
x = torch.randn(1, 3, 64, 896, device=device)

with torch.no_grad():
    _ = model(x)

for h in handles:
    h.remove()

print("Stage | Feature map [B,C,H,W] | CTC steps after vertical pooling")
for name, value in captured.items():
    if torch.is_tensor(value) and value.ndim == 4:
        print(f"{name}: {tuple(value.shape)} | T={value.shape[-1]}")

# Stage 2 is the first candidate: higher horizontal resolution
feat = captured["stage_2"]
assert feat.ndim == 4, f"Unexpected stage_2 output: {type(feat)}"
print("\nCandidate stage_2 channels:", feat.shape[1])
print("Candidate CTC steps:", feat.shape[-1])
print("Longest sample target length in previous batch: 77")
print("CTC minimum feasibility requires input steps >= target length.")
print("For repeated adjacent characters, additional steps may be required.")

del model, x, feat, captured
gc.collect()
torch.cuda.empty_cache()

Stage | Feature map [B,C,H,W] | CTC steps after vertical pooling
stage_1: (1, 160, 8, 112) | T=112
stage_2: (1, 320, 4, 56) | T=56
stage_3: (1, 640, 2, 28) | T=28
stage_4: (1, 640, 2, 28) | T=28

Candidate stage_2 channels: 320
Candidate CTC steps: 56
Longest sample target length in previous batch: 77
CTC minimum feasibility requires input steps >= target length.
For repeated adjacent characters, additional steps may be required.


In [19]:
# ============================================================
# PFMS-Net | MambaVision spatial-resolution check
# ============================================================
import torch
import pandas as pd
from mambavision import create_model

GT_PATH = "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv"
df = pd.read_csv(GT_PATH)
train_df = df[df["split"].astype(str).str.lower() == "train"].copy()

vocab = torch.load(
    "/content/drive/MyDrive/idpl_vocab.pt",
    map_location="cpu",
    weights_only=False
)
char2idx = vocab["char2idx"]
excluded = {
    int(vocab["PAD_IDX"]),
    int(vocab["BOS_IDX"]),
    int(vocab["EOS_IDX"])
}
unk = int(vocab["UNK_IDX"])

def target_len(text):
    return sum(
        1 for ch in str(text).replace("\n", " ").strip()
        if char2idx.get(ch, unk) not in excluded
    )

lengths = train_df["text"].map(target_len)
print("Train samples:", len(train_df))
print("Target length max:", int(lengths.max()))
print("Target length p99:", int(lengths.quantile(0.99)))
print("Targets longer than 128:", int((lengths > 128).sum()))

model = create_model("mamba_vision_T", pretrained=False).cuda().eval()
captured = {}

def hook_stage1(module, inputs, output):
    captured["stage1"] = output

h = model.levels[0].register_forward_hook(hook_stage1)
x = torch.randn(1, 3, 64, 1024, device="cuda")

with torch.no_grad():
    _ = model(x)

h.remove()
feat = captured["stage1"]
print("Stage-1 spatial feature map:", tuple(feat.shape))
print("CTC sequence length:", int(feat.shape[-1]))
print("CTC blank:", vocab["PAD_IDX"])

assert feat.ndim == 4
assert feat.shape[-1] >= int(lengths.max()), (
    "CTC sequence is shorter than the longest target. "
    "Increase width or use a different spatial feature strategy."
)

print("Resolution check passed for target lengths.")
del model, x, feat, captured
torch.cuda.empty_cache()

Train samples: 23054
Target length max: 114
Target length p99: 97
Targets longer than 128: 0
Stage-1 spatial feature map: (1, 160, 8, 128)
CTC sequence length: 128
CTC blank: 0
Resolution check passed for target lengths.


In [23]:

# CELL 1/2 — Complete PFMS-Net ablation setup and preflight
import os, sys, gc, json, time, random, subprocess
from pathlib import Path

def install_if_missing(module, package):
    try:
        __import__(module)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

install_if_missing("timm", "timm")
install_if_missing("mambavision", "mambavision")

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torch.utils.data import Dataset, DataLoader
from mambavision import create_model

# ---------------- CONFIG ----------------
SEED = 42
BATCH_SIZE = 4
EPOCHS = 10
LR = 1e-4
WEIGHT_DECAY = 1e-2
IMG_H, IMG_W = 64, 1024
SEQ_LEN = 128
D_MODEL = 256
NUM_WORKERS = 2
USE_AMP = True

CSV_PATH = "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv"
VOCAB_PATH = "/content/drive/MyDrive/idpl_vocab.pt"
IMAGE_ROOT = "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl"
OUT_DIR = Path("/content/PFMS_ABLATION_FINAL")
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "GPU فعال نیست؛ در Colab GPU را فعال کنید."
DEVICE = torch.device("cuda")
torch.backends.cuda.matmul.allow_tf32 = True

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

# ---------------- VOCABULARY ----------------
vocab_obj = torch.load(VOCAB_PATH, map_location="cpu", weights_only=False)
assert isinstance(vocab_obj, dict) and "char2idx" in vocab_obj, (
    "ساختار فایل واژگان با کلید char2idx مطابقت ندارد."
)

char2idx = {str(k): int(v) for k, v in vocab_obj["char2idx"].items()}
raw_idx2char = vocab_obj.get("idx2char")
idx2char = (
    {int(k): str(v) for k, v in raw_idx2char.items()}
    if raw_idx2char is not None
    else {v: k for k, v in char2idx.items()}
)
PAD_IDX = int(vocab_obj.get("PAD_IDX", 0))
BOS_IDX = int(vocab_obj.get("BOS_IDX", 1))
EOS_IDX = int(vocab_obj.get("EOS_IDX", 2))
UNK_IDX = int(vocab_obj.get("UNK_IDX", 3))
VOCAB_SIZE = max(max(char2idx.values()), max(idx2char.keys())) + 1
assert (PAD_IDX, BOS_IDX, EOS_IDX, UNK_IDX) == (0, 1, 2, 3)
assert VOCAB_SIZE == 169, f"Expected 169 tokens, found {VOCAB_SIZE}"
BLANK_IDX = PAD_IDX
SPECIAL_IDS = {PAD_IDX, BOS_IDX, EOS_IDX}

def encode_text(text):
    ids = [char2idx.get(ch, UNK_IDX) for ch in str(text)]
    ids = [i for i in ids if i not in SPECIAL_IDS]
    if not ids:
        raise ValueError(f"Empty target after special-token removal: {text!r}")
    if any(i < 0 or i >= VOCAB_SIZE for i in ids):
        raise ValueError("Target token outside vocabulary.")
    return ids

def ctc_min_steps(ids):
    return len(ids) + sum(ids[i] == ids[i-1] for i in range(1, len(ids)))

def decode_ctc(ids):
    # Greedy CTC decoding: collapse repeated tokens, then remove blanks.
    result, prev = [], None
    for token in ids:
        token = int(token)
        if token != prev and token != BLANK_IDX:
            result.append(idx2char.get(token, "�"))
        prev = token
    return "".join(result)

# ---------------- DATA SPLIT ----------------
df = pd.read_csv(CSV_PATH)
required = {"split", "image_path", "image_name", "text"}
assert required.issubset(df.columns), f"Missing columns: {required - set(df.columns)}"
df["split"] = df["split"].astype(str).str.lower().str.strip()
df["text"] = df["text"].fillna("").astype(str)

def resolve_path(row):
    candidates = [
        Path(str(row["image_path"])),
        Path(IMAGE_ROOT) / str(row["image_name"]),
        Path(IMAGE_ROOT) / Path(str(row["image_path"])).name,
    ]
    for p in candidates:
        if p.is_file():
            return str(p)
    return str(candidates[0])

df["resolved_path"] = df.apply(resolve_path, axis=1)
df = df[df["text"].str.len() > 0].copy()
missing = df[~df["resolved_path"].map(lambda p: Path(p).is_file())]
if len(missing):
    raise FileNotFoundError(
        f"{len(missing)} image files missing. Examples:\n" +
        "\n".join(missing["resolved_path"].head(5))
    )

train_all = df[df["split"].str.contains("train")].copy()
test_df = df[df["split"].str.contains("test")].copy()
assert len(train_all) > 0 and len(test_df) > 0

# Deterministic 90/10 train/validation split. Test remains untouched.
perm = np.random.RandomState(SEED).permutation(len(train_all))
n_val = max(1, int(round(0.10 * len(train_all))))
val_df = train_all.iloc[perm[:n_val]].reset_index(drop=True)
train_df = train_all.iloc[perm[n_val:]].reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

for split_name, part in [
    ("train", train_df), ("validation", val_df), ("test", test_df)
]:
    targets = [encode_text(t) for t in part["text"]]
    max_target = max(map(len, targets))
    max_required = max(map(ctc_min_steps, targets))
    print(f"{split_name}: n={len(part)}, max_target={max_target}, "
          f"max_CTC_min_steps={max_required}")
    if max_required > SEQ_LEN:
        raise ValueError(
            f"{split_name}: CTC needs {max_required} steps; "
            f"configured sequence length is {SEQ_LEN}."
        )

print("Vocabulary:", VOCAB_SIZE, "| CTC blank:", BLANK_IDX)
print("Train/validation/test:", len(train_df), len(val_df), len(test_df))

# ---------------- IMAGE DATASET ----------------
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

class PersianLineDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, i):
        row = self.frame.iloc[i]
        with Image.open(row["resolved_path"]) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            im = im.resize((IMG_W, IMG_H), Image.Resampling.BILINEAR)
            arr = np.asarray(im, dtype=np.float32) / 255.0
        image = torch.from_numpy(arr).permute(2, 0, 1)
        image = (image - MEAN) / STD
        target = torch.tensor(encode_text(row["text"]), dtype=torch.long)
        return image, target, str(row["text"]), str(row["image_name"])

def collate_lines(batch):
    images, targets, texts, names = zip(*batch)
    lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    return torch.stack(images), torch.cat(targets), lengths, list(texts), list(names)

def make_loader(frame, shuffle):
    return DataLoader(
        PersianLineDataset(frame),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
        collate_fn=collate_lines,
    )

train_loader = make_loader(train_df, True)
val_loader = make_loader(val_df, False)
test_loader = make_loader(test_df, False)

# ---------------- REAL SWIN ENCODER ----------------
class SwinStage1(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=False,
            features_only=True,
            out_indices=(0,),
            img_size=(IMG_H, IMG_W),
        )
        self.channels = int(self.net.feature_info.channels()[0])

    def forward(self, x):
        feat = self.net(x)[0]
        if feat.ndim != 4:
            raise RuntimeError(f"Swin expected 4D features; got {tuple(feat.shape)}")

        # Correctly handle NHWC and NCHW layouts.
        if feat.shape[-1] == self.channels:
            feat = feat.permute(0, 3, 1, 2).contiguous()
        elif feat.shape[1] == self.channels:
            pass
        else:
            raise RuntimeError(
                f"Swin channel axis unclear: shape={tuple(feat.shape)}, "
                f"channels={self.channels}"
            )
        return feat

# ---------------- REAL MAMBAVISION ENCODER ----------------
class RealMambaVisionStage1(nn.Module):
    """
    Actual MambaVision-T stem + stage 1, not its 1000-class classifier.
    This is a stage-1 feature encoder, not the complete four-stage backbone.
    """
    def __init__(self):
        super().__init__()
        backbone = create_model("mamba_vision_T", pretrained=False)
        if not hasattr(backbone, "patch_embed") or not hasattr(backbone, "levels"):
            raise RuntimeError(
                "Installed MambaVision API changed: patch_embed/levels unavailable."
            )
        self.patch_embed = backbone.patch_embed
        self.stage1 = backbone.levels[0]
        self.channels = 160
        del backbone

    def forward(self, x):
        x = self.patch_embed(x)
        x = self.stage1(x)
        if x.ndim != 4:
            raise RuntimeError(f"MambaVision stage-1 output is not 4D: {tuple(x.shape)}")
        if x.shape[1] != self.channels:
            raise RuntimeError(
                f"Expected MambaVision NCHW with {self.channels} channels; "
                f"got {tuple(x.shape)}"
            )
        return x

# ---------------- FUSION AND CTC CLASSIFIER ----------------
class CrossAttentionBlock(nn.Module):
    def __init__(self, dim=D_MODEL, heads=4):
        super().__init__()
        self.qnorm = nn.LayerNorm(dim)
        self.kvnorm = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True, dropout=0.1)
        self.ffn = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, 2 * dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(2 * dim, dim),
        )
        self.drop = nn.Dropout(0.1)

    def forward(self, q, kv):
        z, _ = self.attn(
            self.qnorm(q), self.kvnorm(kv), self.kvnorm(kv),
            need_weights=False
        )
        q = q + self.drop(z)
        return q + self.drop(self.ffn(q))

class OCRAblationModel(nn.Module):
    VALID = {"swin_only", "mamba_only", "simple_fusion",
             "cross_attention", "full_pfms"}

    def __init__(self, variant):
        super().__init__()
        if variant not in self.VALID:
            raise ValueError(f"Unknown variant: {variant}")
        self.variant = variant

        self.swin = SwinStage1() if variant != "mamba_only" else None
        self.mamba = RealMambaVisionStage1() if variant != "swin_only" else None

        if self.swin is not None:
            self.swin_proj = nn.Conv2d(self.swin.channels, D_MODEL, kernel_size=1)
        if self.mamba is not None:
            self.mamba_proj = nn.Conv2d(self.mamba.channels, D_MODEL, kernel_size=1)

        if variant == "simple_fusion":
            self.fuse = nn.Sequential(
                nn.Linear(2 * D_MODEL, D_MODEL),
                nn.GELU(),
                nn.LayerNorm(D_MODEL),
            )
        if variant in {"cross_attention", "full_pfms"}:
            self.swin_to_mamba = CrossAttentionBlock()
            self.mamba_to_swin = CrossAttentionBlock()

        if variant == "full_pfms":
            self.refine = nn.Sequential(
                nn.Conv1d(D_MODEL, D_MODEL, 3, padding=1),
                nn.GELU(),
                nn.Conv1d(D_MODEL, D_MODEL, 3, padding=1),
            )
        else:
            self.refine = None

        self.classifier = nn.Linear(D_MODEL, VOCAB_SIZE)

    @staticmethod
    def spatial_to_sequence(feat, projection):
        feat = projection(feat)       # [B,D,H,W]
        feat = feat.mean(dim=2)       # [B,D,W]
        feat = F.adaptive_avg_pool1d(feat, SEQ_LEN)
        return feat.transpose(1, 2).contiguous()  # [B,T,D]

    def forward(self, images):
        s = None
        m = None
        if self.swin is not None:
            s = self.spatial_to_sequence(self.swin(images), self.swin_proj)
        if self.mamba is not None:
            m = self.spatial_to_sequence(self.mamba(images), self.mamba_proj)

        if self.variant == "swin_only":
            z = s
        elif self.variant == "mamba_only":
            z = m
        elif self.variant == "simple_fusion":
            z = self.fuse(torch.cat((s, m), dim=-1))
        else:
            s2 = self.swin_to_mamba(s, m)
            m2 = self.mamba_to_swin(m, s)
            z = 0.5 * (s2 + m2)
            if self.refine is not None:
                z = z + self.refine(z.transpose(1, 2)).transpose(1, 2)

        logits = self.classifier(z)
        if logits.shape[1:] != (SEQ_LEN, VOCAB_SIZE):
            raise RuntimeError(f"Unexpected logits shape: {tuple(logits.shape)}")
        return logits

# ---------------- PREFLIGHT: ALL FIVE MODELS ----------------
def run_preflight():
    results = []
    variants = ["swin_only", "mamba_only", "simple_fusion",
                "cross_attention", "full_pfms"]
    x = torch.zeros(1, 3, IMG_H, IMG_W, device=DEVICE)

    for variant in variants:
        gc.collect()
        torch.cuda.empty_cache()
        model = OCRAblationModel(variant).to(DEVICE).eval()
        try:
            with torch.inference_mode():
                y = model(x)
            assert tuple(y.shape) == (1, SEQ_LEN, VOCAB_SIZE), (
                variant, tuple(y.shape)
            )
            n_total = sum(p.numel() for p in model.parameters())
            n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
            row = {
                "variant": variant,
                "logits_shape": str(tuple(y.shape)),
                "parameters_total": n_total,
                "parameters_trainable": n_train,
            }
            results.append(row)
            print(
                f"PREFLIGHT OK: {variant} | shape={tuple(y.shape)} | "
                f"trainable={n_train:,}"
            )
        finally:
            del model
            gc.collect()
            torch.cuda.empty_cache()

    pd.DataFrame(results).to_csv(OUT_DIR / "preflight.csv", index=False)
    return results

PREFLIGHT_RESULTS = run_preflight()
print("Preflight passed. No training has been run in Cell 1.")


PyTorch: 2.11.0+cu130 | GPU: Tesla T4
train: n=20749, max_target=115, max_CTC_min_steps=116
validation: n=2305, max_target=109, max_CTC_min_steps=111
test: n=4066, max_target=105, max_CTC_min_steps=107
Vocabulary: 169 | CTC blank: 0
Train/validation/test: 20749 2305 4066
PREFLIGHT OK: swin_only | shape=(1, 128, 169) | trainable=297,855
PREFLIGHT OK: mamba_only | shape=(1, 128, 169) | trainable=339,657
PREFLIGHT OK: simple_fusion | shape=(1, 128, 169) | trainable=725,919
PREFLIGHT OK: cross_attention | shape=(1, 128, 169) | trainable=1,649,311
PREFLIGHT OK: full_pfms | shape=(1, 128, 169) | trainable=2,043,039
Preflight passed. No training has been run in Cell 1.


In [24]:

# CELL 2/2 — Training, held-out evaluation, checkpoints and CSV tables
import gc, json, time
from pathlib import Path

# Stop immediately if Cell 1 did not finish successfully.
assert "PREFLIGHT_RESULTS" in globals(), (
    "ابتدا سلول اول را کامل اجرا کنید؛ PREFLIGHT_RESULTS وجود ندارد."
)
assert len(PREFLIGHT_RESULTS) == 5, "پیش‌آزمون هر پنج مدل موفق نشده است."
assert all(r["logits_shape"] == str((1, SEQ_LEN, VOCAB_SIZE))
           for r in PREFLIGHT_RESULTS)

VARIANTS = [
    "swin_only",
    "mamba_only",
    "simple_fusion",
    "cross_attention",
    "full_pfms",
]

ctc_loss_fn = nn.CTCLoss(
    blank=BLANK_IDX,
    reduction="mean",
    zero_infinity=False,
)

def edit_distance(a, b):
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(
                cur[-1] + 1,
                prev[j] + 1,
                prev[j - 1] + (ca != cb),
            ))
        prev = cur
    return prev[-1]

def calculate_metrics(refs, preds):
    char_edits = sum(edit_distance(list(r), list(p)) for r, p in zip(refs, preds))
    ref_chars = sum(max(1, len(r)) for r in refs)
    word_edits = sum(
        edit_distance(r.split(), p.split()) for r, p in zip(refs, preds)
    )
    ref_words = sum(max(1, len(r.split())) for r in refs)
    exact = sum(r == p for r, p in zip(refs, preds))
    cer = char_edits / max(1, ref_chars)
    wer = word_edits / max(1, ref_words)
    return {
        "CER": cer,
        "WER": wer,
        "character_accuracy_1_minus_CER": 1.0 - cer,
        "word_accuracy_1_minus_WER": 1.0 - wer,
        "sequence_accuracy": exact / max(1, len(refs)),
        "char_edits": char_edits,
        "reference_characters": ref_chars,
        "word_edits": word_edits,
        "reference_words": ref_words,
        "exact_sequences": exact,
        "samples": len(refs),
    }

def amp_context():
    return torch.autocast(
        device_type="cuda", dtype=torch.float16, enabled=USE_AMP
    )

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def evaluate_model(model, loader, save_predictions=False):
    model.eval()
    loss_sum, n_seen, elapsed = 0.0, 0, 0.0
    refs, preds, rows = [], [], []

    with torch.inference_mode():
        for images, targets, target_lengths, texts, names in loader:
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            target_lengths = target_lengths.to(DEVICE, non_blocking=True)
            bsz = images.size(0)

            torch.cuda.synchronize()
            t0 = time.perf_counter()
            with amp_context():
                logits = model(images)
            torch.cuda.synchronize()
            elapsed += time.perf_counter() - t0

            log_probs = logits.float().log_softmax(dim=-1)
            input_lengths = torch.full(
                (bsz,), log_probs.size(1), dtype=torch.long
            )
            loss = ctc_loss_fn(
                log_probs.transpose(0, 1),
                targets,
                input_lengths,
                target_lengths.cpu(),
            )
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite evaluation CTC loss.")

            loss_sum += loss.item() * bsz
            n_seen += bsz
            batch_preds = [
                decode_ctc(seq) for seq in log_probs.argmax(-1).cpu().tolist()
            ]
            refs.extend(texts)
            preds.extend(batch_preds)

            if save_predictions:
                rows.extend({
                    "image_name": name,
                    "reference": ref,
                    "prediction": pred,
                    "exact_match": (ref == pred),
                } for name, ref, pred in zip(names, texts, batch_preds))

    metrics = calculate_metrics(refs, preds)
    metrics["loss"] = loss_sum / max(1, n_seen)
    metrics["inference_ms_per_image"] = 1000 * elapsed / max(1, n_seen)
    metrics["images_per_second"] = n_seen / max(elapsed, 1e-12)

    if save_predictions:
        return metrics, pd.DataFrame(rows)
    return metrics

def train_variant(variant):
    print("\n" + "=" * 70)
    print("START:", variant)
    print("=" * 70)

    seed_everything(SEED)
    model = OCRAblationModel(variant).to(DEVICE)
    params_total = count_params(model)
    params_trainable = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    model_dir = OUT_DIR / variant
    model_dir.mkdir(parents=True, exist_ok=True)
    best_path = model_dir / "best_model.pt"
    last_path = model_dir / "last_model.pt"
    history = []
    best_val_loss = float("inf")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum, n_seen = 0.0, 0
        start_epoch = time.perf_counter()

        for step, (images, targets, target_lengths, texts, names) in enumerate(
            train_loader, start=1
        ):
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            target_lengths = target_lengths.to(DEVICE, non_blocking=True)
            bsz = images.size(0)

            # Check CTC alignment requirement for every sample in this batch.
            needed = max(ctc_min_steps(encode_text(t)) for t in texts)
            if needed > SEQ_LEN:
                raise RuntimeError(
                    f"CTC alignment invalid: batch needs {needed} steps, "
                    f"but output has {SEQ_LEN}."
                )

            optimizer.zero_grad(set_to_none=True)
            with amp_context():
                logits = model(images)

            log_probs = logits.float().log_softmax(dim=-1)
            input_lengths = torch.full(
                (bsz,), log_probs.size(1), dtype=torch.long
            )
            loss = ctc_loss_fn(
                log_probs.transpose(0, 1),
                targets,
                input_lengths,
                target_lengths.cpu(),
            )

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f"Non-finite training loss at {variant}, "
                    f"epoch {epoch}, batch {step}."
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()

            loss_sum += loss.item() * bsz
            n_seen += bsz

            if step % 100 == 0 or step == len(train_loader):
                print(
                    f"{variant} | epoch {epoch}/{EPOCHS} | "
                    f"batch {step}/{len(train_loader)} | loss={loss.item():.5f}"
                )

        train_loss = loss_sum / max(1, n_seen)
        val_metrics = evaluate_model(model, val_loader)
        epoch_seconds = time.perf_counter() - start_epoch

        record = {
            "variant": variant,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_CER": val_metrics["CER"],
            "val_WER": val_metrics["WER"],
            "val_sequence_accuracy": val_metrics["sequence_accuracy"],
            "epoch_seconds": epoch_seconds,
            "parameters_total": params_total,
            "parameters_trainable": params_trainable,
            "seed": SEED,
            "batch_size": BATCH_SIZE,
            "learning_rate": LR,
            "weight_decay": WEIGHT_DECAY,
        }
        history.append(record)
        pd.DataFrame(history).to_csv(
            model_dir / "training_history.csv", index=False
        )

        state = {
            "variant": variant,
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_metrics["loss"],
            "seed": SEED,
            "vocab_size": VOCAB_SIZE,
            "blank_idx": BLANK_IDX,
            "sequence_length": SEQ_LEN,
            "image_height": IMG_H,
            "image_width": IMG_W,
        }
        torch.save(state, last_path)

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            torch.save(state, best_path)

        print(
            f"Epoch {epoch}: train_loss={train_loss:.5f} | "
            f"val_loss={val_metrics['loss']:.5f} | "
            f"val_CER={val_metrics['CER']:.5f} | "
            f"val_WER={val_metrics['WER']:.5f} | "
            f"{epoch_seconds:.1f}s"
        )

    del optimizer, scaler
    gc.collect()
    torch.cuda.empty_cache()

    # Final held-out evaluation uses the best validation checkpoint.
    best = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(best["model_state_dict"])
    test_metrics, pred_df = evaluate_model(
        model, test_loader, save_predictions=True
    )

    pred_file = model_dir / "test_predictions.csv"
    pred_df.to_csv(pred_file, index=False)

    summary = {
        "variant": variant,
        "best_epoch": int(best["epoch"]),
        "best_val_loss": float(best["val_loss"]),
        "parameters_total": params_total,
        "parameters_trainable": params_trainable,
        **{f"test_{k}": v for k, v in test_metrics.items()},
        "seed": SEED,
        "epochs_requested": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "weight_decay": WEIGHT_DECAY,
        "image_height": IMG_H,
        "image_width": IMG_W,
        "ctc_sequence_length": SEQ_LEN,
        "vocab_size": VOCAB_SIZE,
        "checkpoint": str(best_path),
        "prediction_file": str(pred_file),
    }

    with open(model_dir / "test_metrics.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("TEST RESULTS:", json.dumps(summary, ensure_ascii=False, indent=2))
    del model, best
    gc.collect()
    torch.cuda.empty_cache()
    return summary

# Record protocol before the long-running experiment.
config = {
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "optimizer": "AdamW",
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "loss": "CTCLoss",
    "ctc_blank": BLANK_IDX,
    "image_size": [IMG_H, IMG_W],
    "sequence_length": SEQ_LEN,
    "vocab_size": VOCAB_SIZE,
    "variants": VARIANTS,
    "mamba_encoder": "MambaVision-T stem + actual stage 1",
    "swin_encoder": "timm Swin-T stage-1 spatial features",
    "validation": "deterministic 10% split from official training data",
    "test": "held out from training and model selection",
}
with open(OUT_DIR / "experiment_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

summaries = []
for variant in VARIANTS:
    result = train_variant(variant)
    summaries.append(result)
    pd.DataFrame(summaries).to_csv(
        OUT_DIR / "ablation_results_partial.csv", index=False
    )

results = pd.DataFrame(summaries)
results.to_csv(OUT_DIR / "ablation_results.csv", index=False)

paper_table = results[[
    "variant", "parameters_total",
    "test_CER", "test_WER",
    "test_character_accuracy_1_minus_CER",
    "test_word_accuracy_1_minus_WER",
    "test_sequence_accuracy",
    "test_inference_ms_per_image",
    "test_images_per_second",
]].copy()

percentage_cols = [
    "test_CER", "test_WER",
    "test_character_accuracy_1_minus_CER",
    "test_word_accuracy_1_minus_WER",
    "test_sequence_accuracy",
]
for col in percentage_cols:
    paper_table[col] *= 100.0

paper_table = paper_table.rename(columns={
    "variant": "Model",
    "parameters_total": "Parameters",
    "test_CER": "CER (%)",
    "test_WER": "WER (%)",
    "test_character_accuracy_1_minus_CER": "1-CER (%)",
    "test_word_accuracy_1_minus_WER": "1-WER (%)",
    "test_sequence_accuracy": "Sequence accuracy (%)",
    "test_inference_ms_per_image": "Inference ms/image",
    "test_images_per_second": "Images/s",
})
paper_table.to_csv(OUT_DIR / "ablation_paper_table.csv", index=False)

print("\nFINAL MEASURED RESULTS")
display(paper_table.round(4))
print("\nOutput directory:", OUT_DIR)



START: swin_only
swin_only | epoch 1/10 | batch 100/5188 | loss=3.19267
swin_only | epoch 1/10 | batch 200/5188 | loss=3.07598
swin_only | epoch 1/10 | batch 300/5188 | loss=3.14058
swin_only | epoch 1/10 | batch 400/5188 | loss=3.05177
swin_only | epoch 1/10 | batch 500/5188 | loss=3.00459
swin_only | epoch 1/10 | batch 600/5188 | loss=3.07159
swin_only | epoch 1/10 | batch 700/5188 | loss=3.17005
swin_only | epoch 1/10 | batch 800/5188 | loss=3.01859
swin_only | epoch 1/10 | batch 900/5188 | loss=3.20653
swin_only | epoch 1/10 | batch 1000/5188 | loss=2.97746
swin_only | epoch 1/10 | batch 1100/5188 | loss=3.23462
swin_only | epoch 1/10 | batch 1200/5188 | loss=3.02298
swin_only | epoch 1/10 | batch 1300/5188 | loss=3.06057
swin_only | epoch 1/10 | batch 1400/5188 | loss=3.04255
swin_only | epoch 1/10 | batch 1500/5188 | loss=2.96517
swin_only | epoch 1/10 | batch 1600/5188 | loss=2.99148
swin_only | epoch 1/10 | batch 1700/5188 | loss=3.07707
swin_only | epoch 1/10 | batch 1800/518

Exception ignored in atexit callback <function _exit_function at 0x7fbc7ae58180>:
Traceback (most recent call last):
  File "/usr/lib/python3.13/multiprocessing/util.py", line 431, in _exit_function
    

KeyboardInterrupt: 

_run_finalizers()
  File "/usr/lib/python3.13/multiprocessing/util.py", line 371, in _run_finalizers
    finalizer()
  File "/usr/lib/python3.13/multiprocessing/util.py", line 295, in __call__


In [3]:
# Recover vocabulary only — no training, no file deletion
from pathlib import Path
import torch

VOCAB_PATH = Path("/content/drive/MyDrive/idpl_vocab.pt")

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

if not VOCAB_PATH.is_file():
    raise FileNotFoundError(
        f"فایل واژگان پیدا نشد: {VOCAB_PATH}\n"
        "فایل را حذف یا بازسازی نکنید؛ ابتدا مسیر درست را بررسی می‌کنیم."
    )

vocab_obj = torch.load(VOCAB_PATH, map_location="cpu", weights_only=False)

if not isinstance(vocab_obj, dict) or "char2idx" not in vocab_obj:
    raise ValueError("ساختار فایل واژگان با انتظار کد مطابقت ندارد.")

char2idx = {str(k): int(v) for k, v in vocab_obj["char2idx"].items()}
raw_idx2char = vocab_obj.get("idx2char")
idx2char = (
    {int(k): str(v) for k, v in raw_idx2char.items()}
    if raw_idx2char is not None
    else {v: k for k, v in char2idx.items()}
)

PAD_IDX = int(vocab_obj.get("PAD_IDX", 0))
BOS_IDX = int(vocab_obj.get("BOS_IDX", 1))
EOS_IDX = int(vocab_obj.get("EOS_IDX", 2))
UNK_IDX = int(vocab_obj.get("UNK_IDX", 3))
VOCAB_SIZE = max(max(char2idx.values()), max(idx2char.keys())) + 1
BLANK_IDX = PAD_IDX
SPECIAL_IDS = {PAD_IDX, BOS_IDX, EOS_IDX}

print("Vocabulary loaded successfully.")
print("Vocabulary size:", VOCAB_SIZE)
print("Special IDs:", PAD_IDX, BOS_IDX, EOS_IDX, UNK_IDX)
print("Characters:", len(char2idx))

Vocabulary loaded successfully.
Vocabulary size: 169
Special IDs: 0 1 2 3
Characters: 169


In [4]:
# Read-only runtime status check
required_vars = [
    "char2idx", "idx2char", "train_df", "val_df", "test_df",
    "train_loader", "val_loader", "test_loader",
    "encode_text", "decode_ctc", "OCRAblationModel"
]

for name in required_vars:
    print(f"{name}: {'AVAILABLE' if name in globals() else 'MISSING'}")

char2idx: AVAILABLE
idx2char: AVAILABLE
train_df: MISSING
val_df: MISSING
test_df: MISSING
train_loader: MISSING
val_loader: MISSING
test_loader: MISSING
encode_text: MISSING
decode_ctc: MISSING
OCRAblationModel: MISSING


In [5]:
# STAGE 1 — Read-only data and vocabulary audit
# No model creation, no training, no checkpoint changes.

from pathlib import Path
import pandas as pd
from collections import Counter
import unicodedata

CSV_PATH = "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv"
IMAGE_ROOT = Path("/content/drive/Othercomputers/My Laptop/Desktop/idplimgl")

assert "char2idx" in globals() and "idx2char" in globals(), (
    "واژگان در حافظه نیستند؛ ابتدا سلول بازیابی واژگان را اجرا کنید."
)
assert Path(CSV_PATH).is_file(), f"CSV پیدا نشد: {CSV_PATH}"

df_audit = pd.read_csv(CSV_PATH)
required = {"split", "image_path", "image_name", "text"}
assert required.issubset(df_audit.columns), (
    f"ستون‌های ضروری موجود نیستند: {required - set(df_audit.columns)}"
)

df_audit["split"] = df_audit["split"].astype(str).str.lower().str.strip()
df_audit["text"] = df_audit["text"].fillna("").astype(str)

print("=== Dataset ===")
print("Rows:", len(df_audit))
print("Splits:", df_audit["split"].value_counts().to_dict())
print("Empty texts:", int((df_audit["text"].str.len() == 0).sum()))
print("Duplicate image names:", int(df_audit["image_name"].duplicated().sum()))

def resolve_image(row):
    candidates = [
        Path(str(row["image_path"])),
        IMAGE_ROOT / str(row["image_name"]),
        IMAGE_ROOT / Path(str(row["image_path"])).name,
    ]
    for p in candidates:
        if p.is_file():
            return str(p)
    return None

# Sample path validation only; does not scan/open every image.
sample = df_audit[df_audit["text"].str.len() > 0].head(100)
sample_paths = sample.apply(resolve_image, axis=1)
print("Sample images checked:", len(sample))
print("Sample image paths found:", int(sample_paths.notna().sum()))

print("\n=== Vocabulary coverage ===")
train_audit = df_audit[
    df_audit["split"].str.contains("train", na=False)
    & (df_audit["text"].str.len() > 0)
]
char_counts = Counter(ch for text in train_audit["text"] for ch in text)
missing_chars = {
    ch: n for ch, n in char_counts.items() if ch not in char2idx
}
print("Training texts:", len(train_audit))
print("Vocabulary size:", len(char2idx))
print("Distinct characters in training text:", len(char_counts))
print("Distinct characters missing from vocabulary:", len(missing_chars))
print("Missing-character frequency total:", sum(missing_chars.values()))
print("Missing-character examples:",
      [(repr(ch), n, unicodedata.name(ch, "UNKNOWN"))
       for ch, n in sorted(missing_chars.items(), key=lambda x: -x[1])[:20]])

print("\n=== Target encoding round-trip ===")
def audit_encode(text):
    special = {
        int(globals().get("PAD_IDX", 0)),
        int(globals().get("BOS_IDX", 1)),
        int(globals().get("EOS_IDX", 2)),
    }
    unk = int(globals().get("UNK_IDX", 3))
    ids = [int(char2idx.get(ch, unk)) for ch in str(text)]
    return [i for i in ids if i not in special]

for _, row in train_audit.head(5).iterrows():
    text = row["text"]
    ids = audit_encode(text)
    decoded = "".join(idx2char.get(i, "�") for i in ids)
    print({
        "image_name": str(row["image_name"]),
        "text": repr(text),
        "decoded": repr(decoded),
        "exact_match": decoded == text,
        "target_length": len(ids),
    })

print("\nAudit finished. No training or files changed.")

=== Dataset ===
Rows: 27120
Splits: {'train': 23054, 'test': 4066}
Empty texts: 0
Duplicate image names: 3
Sample images checked: 100
Sample image paths found: 100

=== Vocabulary coverage ===
Training texts: 23054
Vocabulary size: 169
Distinct characters in training text: 164
Distinct characters missing from vocabulary: 0
Missing-character frequency total: 0
Missing-character examples: []

=== Target encoding round-trip ===
{'image_name': '00490.tif', 'text': "'یابد و كار جمهوری اسلامی را یكسره كند. كافی است به تبلیغات «اكس»گونه یك ماه\\n'", 'decoded': "'یابد و كار جمهوری اسلامی را یكسره كند. كافی است به تبلیغات «اكس»گونه یك ماه\\n'", 'exact_match': True, 'target_length': 76}
{'image_name': '19617.tif', 'text': "'تا این عدد ۱۱۰ گرم به ۲۰۰ تا ۲۵۰ گرم برسد. وی با اشاره به\\n'", 'decoded': "'تا این عدد ۱۱۰ گرم به ۲۰۰ تا ۲۵۰ گرم برسد. وی با اشاره به\\n'", 'exact_match': True, 'target_length': 58}
{'image_name': '23763.tif', 'text': "'احساس كرد كه در برابر شاه تحقیر شده است. چند روز بعد اب

In [6]:
# Read-only inspection of duplicate image names
dup = df_audit[
    df_audit["image_name"].duplicated(keep=False)
].sort_values("image_name")

print(dup[["split", "image_name", "image_path", "text"]].to_string(index=False))

split image_name                                                         image_path                                                                             text
train  03363.tif /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/03363.tif داشته باشد. علاوه بر این تنظیمات برنامه در دستان شماست ؛ می توانید انتخاب کنید\n
train  03363.tif /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/03363.tif داشته باشد. علاوه بر این تنظیمات برنامه در دستان شماست ؛ می توانید انتخاب کنید\n
train  03375.tif /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/03375.tif برای ضبط مکالمات با ساده ترین حالت ممکن از استودیوی برای اندروید است. توسط این\n
 test  03375.tif /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/03375.tif برای ضبط مکالمات با ساده ترین حالت ممکن از استودیوی برای اندروید است. توسط این\n
train  15229.tif /content/drive/Othercomputers/My Laptop/Desktop/idplimgl/15229.tif    با بروزرسانی که گوگل برای بخش تصاویر خود انجام داده، از این پس متوجه خواهید\n
train  152

In [7]:
# SAFE DATA SPLIT CLEANUP — in-memory only
# Does not delete or modify any source files.

audit = df_audit.copy()
audit["image_name"] = audit["image_name"].astype(str).str.strip()
audit["split"] = audit["split"].astype(str).str.lower().str.strip()

# Report any duplicate image names with conflicting labels
label_counts = audit.groupby("image_name")["text"].nunique()
conflicting_names = label_counts[label_counts > 1].index.tolist()

print("Images with conflicting text labels:", conflicting_names)

train_raw = audit[audit["split"].str.contains("train", na=False)].copy()
test_raw = audit[audit["split"].str.contains("test", na=False)].copy()

train_names = set(train_raw["image_name"])
test_names = set(test_raw["image_name"])
overlap_names = train_names & test_names

print("Train/test overlapping image names:", sorted(overlap_names))

# Keep test membership intact; remove overlapping names from training.
train_clean = train_raw[~train_raw["image_name"].isin(overlap_names)].copy()

# Deduplicate within each split by image_name.
train_clean = train_clean.drop_duplicates(subset=["image_name"], keep="first")
test_clean = test_raw.drop_duplicates(subset=["image_name"], keep="first")

print("\nRows before cleanup:")
print("Train:", len(train_raw), "| Test:", len(test_raw))

print("\nRows after in-memory cleanup:")
print("Train:", len(train_clean), "| Test:", len(test_clean))

print("Remaining train/test overlap:",
      len(set(train_clean["image_name"]) & set(test_clean["image_name"])))
print("Remaining train duplicates:", int(train_clean["image_name"].duplicated().sum()))
print("Remaining test duplicates:", int(test_clean["image_name"].duplicated().sum()))

Images with conflicting text labels: []
Train/test overlapping image names: ['03375.tif']

Rows before cleanup:
Train: 23054 | Test: 4066

Rows after in-memory cleanup:
Train: 23051 | Test: 4066
Remaining train/test overlap: 0
Remaining train duplicates: 0
Remaining test duplicates: 0


In [8]:
# FINAL SPLIT PREPARATION — no model training
import numpy as np

SEED = 42

# train_clean and test_clean are the cleaned dataframes from the previous cell.
perm = np.random.RandomState(SEED).permutation(len(train_clean))
n_val = max(1, int(round(0.10 * len(train_clean))))

val_df = train_clean.iloc[perm[:n_val]].reset_index(drop=True)
train_df = train_clean.iloc[perm[n_val:]].reset_index(drop=True)
test_df = test_clean.reset_index(drop=True)

train_names = set(train_df["image_name"])
val_names = set(val_df["image_name"])
test_names = set(test_df["image_name"])

assert not (train_names & val_names), "Train/validation overlap detected."
assert not (train_names & test_names), "Train/test overlap detected."
assert not (val_names & test_names), "Validation/test overlap detected."

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Train/validation overlap:", len(train_names & val_names))
print("Train/test overlap:", len(train_names & test_names))
print("Validation/test overlap:", len(val_names & test_names))
print("Split preparation complete. No training performed.")

Train: 20746
Validation: 2305
Test: 4066
Train/validation overlap: 0
Train/test overlap: 0
Validation/test overlap: 0
Split preparation complete. No training performed.


In [9]:
# READ-ONLY IMAGE / TARGET AUDIT
from pathlib import Path
from PIL import Image
import numpy as np
import torch

IMG_H, IMG_W = 64, 1024
SEQ_LEN = 128

def encode_audit(text):
    special = {PAD_IDX, BOS_IDX, EOS_IDX}
    ids = [char2idx.get(ch, UNK_IDX) for ch in str(text)]
    return [int(i) for i in ids if i not in special]

def ctc_min_steps_audit(ids):
    return len(ids) + sum(ids[i] == ids[i-1] for i in range(1, len(ids)))

print("=== Target lengths ===")
for label, frame in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    encoded = [encode_audit(t) for t in frame["text"]]
    required = [ctc_min_steps_audit(ids) for ids in encoded]
    lengths = np.array([len(ids) for ids in encoded])
    print(
        label,
        "| n =", len(frame),
        "| max target =", int(lengths.max()),
        "| p99 target =", int(np.percentile(lengths, 99)),
        "| max CTC steps =", int(max(required)),
        "| samples needing >128 steps =", int(sum(x > SEQ_LEN for x in required))
    )

print("\n=== Image size and mode ===")
for _, row in train_df.head(8).iterrows():
    path = Path(str(row["image_path"]))
    candidates = [
        path,
        Path(IMAGE_ROOT) / str(row["image_name"]),
        Path(IMAGE_ROOT) / path.name,
    ]
    found = next((p for p in candidates if p.is_file()), None)
    if found is None:
        print(row["image_name"], "MISSING")
        continue
    with Image.open(found) as im:
        print(row["image_name"], "| original size =", im.size, "| mode =", im.mode)

print("\nRead-only audit complete.")

=== Target lengths ===
train | n = 20746 | max target = 115 | p99 target = 98 | max CTC steps = 116 | samples needing >128 steps = 0
validation | n = 2305 | max target = 110 | p99 target = 99 | max CTC steps = 111 | samples needing >128 steps = 0
test | n = 4066 | max target = 105 | p99 target = 99 | max CTC steps = 107 | samples needing >128 steps = 0

=== Image size and mode ===
26102.tif | original size = (700, 50) | mode = RGB
12486.tif | original size = (700, 50) | mode = RGB
17561.tif | original size = (700, 50) | mode = RGB
20077.tif | original size = (700, 50) | mode = RGB
27968.tif | original size = (700, 50) | mode = RGB
04974.tif | original size = (700, 50) | mode = RGB
15329.tif | original size = (700, 50) | mode = RGB
07558.tif | original size = (700, 50) | mode = L

Read-only audit complete.


In [10]:
# READ-ONLY: inspect current runtime and installed model APIs
import inspect
import torch
import timm

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\ntimm:", timm.__version__)

# Check the real MambaVision model registration and constructor
try:
    print("\nMambaVision model registered:",
          "mamba_vision_T" in timm.list_models())

    from timm import create_model
    mamba = create_model("mamba_vision_T", pretrained=False)
    print("MambaVision class:", type(mamba).__name__)
    print("Top-level children:", [name for name, _ in mamba.named_children()])

    if hasattr(mamba, "patch_embed"):
        print("patch_embed:", type(mamba.patch_embed).__name__)
    if hasattr(mamba, "levels"):
        print("Number of levels:", len(mamba.levels))
        print("Level classes:", [type(x).__name__ for x in mamba.levels])

    del mamba
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:
    print("MambaVision inspection failed:", repr(e))

# Check Swin feature API without loading any training checkpoint
try:
    model = timm.create_model(
        "swin_tiny_patch4_window7_224",
        pretrained=False,
        features_only=True,
        out_indices=(0,),
        img_size=(64, 1024),
    )
    model.eval()

    with torch.no_grad():
        x = torch.zeros(1, 3, 64, 1024)
        features = model(x)
        for i, feat in enumerate(features):
            print(f"Swin feature {i}: shape={tuple(feat.shape)}")

    del model, features, x
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:
    print("Swin inspection failed:", repr(e))

print("\nInspection complete. No training performed.")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4

timm: 1.0.30

MambaVision model registered: False
MambaVision inspection failed: RuntimeError('Unknown model (mamba_vision_T)')
Swin feature 0: shape=(1, 16, 256, 96)

Inspection complete. No training performed.


In [11]:
# READ-ONLY ARCHITECTURE CHECK — NO TRAINING
import torch
import timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("PyTorch:", torch.__version__)
print("timm:", timm.__version__)

# --------------------------------------------------
# 1. Official MambaVision package
# --------------------------------------------------
try:
    from mambavision import create_model as create_mambavision

    mamba = create_mambavision(
        "mamba_vision_T",
        pretrained=False
    ).to(device).eval()

    print("\nMambaVision class:", type(mamba).__name__)
    print("Patch embedding:", type(mamba.patch_embed).__name__)
    print("Number of backbone levels:", len(mamba.levels))

    x = torch.zeros(1, 3, 64, 1024, device=device)

    with torch.inference_mode():
        feat = mamba.patch_embed(x)
        print("After patch_embed:", tuple(feat.shape))

        for i, level in enumerate(mamba.levels):
            feat = level(feat)
            print(f"After level {i + 1}:", tuple(feat.shape))

    del mamba, x, feat
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:
    print("\nMAMBAVISION ERROR:", type(e).__name__, str(e))

# --------------------------------------------------
# 2. Swin feature layout
# --------------------------------------------------
try:
    swin = timm.create_model(
        "swin_tiny_patch4_window7_224",
        pretrained=False,
        features_only=True,
        out_indices=(0,),
        img_size=(64, 1024),
    ).to(device).eval()

    x = torch.zeros(1, 3, 64, 1024, device=device)

    with torch.inference_mode():
        feat = swin(x)[0]
        print("\nSwin raw feature:", tuple(feat.shape))

        channels = swin.feature_info.channels()[0]
        if feat.shape[-1] == channels:
            feat_nchw = feat.permute(0, 3, 1, 2).contiguous()
        elif feat.shape[1] == channels:
            feat_nchw = feat
        else:
            raise RuntimeError(
                f"Cannot identify channel dimension: "
                f"shape={tuple(feat.shape)}, expected channels={channels}"
            )

        print("Swin NCHW feature:", tuple(feat_nchw.shape))

    del swin, x, feat, feat_nchw
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:
    print("\nSWIN ERROR:", type(e).__name__, str(e))

print("\nArchitecture check finished. No training performed.")

Device: cuda
PyTorch: 2.11.0+cu130
timm: 1.0.30


/usr/local/lib/python3.13/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)



MambaVision class: MambaVision
Patch embedding: PatchEmbed
Number of backbone levels: 4
After patch_embed: (1, 80, 16, 256)
After level 1: (1, 160, 8, 128)
After level 2: (1, 320, 4, 64)
After level 3: (1, 640, 2, 32)
After level 4: (1, 640, 2, 32)

Swin raw feature: (1, 16, 256, 96)
Swin NCHW feature: (1, 96, 16, 256)

Architecture check finished. No training performed.


In [12]:
# FAST READ-ONLY PREFLIGHT — NO TRAINING
import torch
import pandas as pd
from pathlib import Path
from PIL import Image

assert "train_df" in globals(), "ابتدا سلول آماده‌سازی train_df را اجرا کنید."
assert "char2idx" in globals(), "واژگان بارگذاری نشده است."

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Train rows:", len(train_df))
print("Vocab:", len(char2idx))

# Check one actual image and its target
row = train_df.iloc[0]
p = Path(str(row["image_path"]))
if not p.is_file():
    p = Path("/content/drive/Othercomputers/My Laptop/Desktop/idplimgl") / str(row["image_name"])

assert p.is_file(), f"Image not found: {p}"

with Image.open(p) as im:
    print("Image:", row["image_name"], "| size:", im.size, "| mode:", im.mode)

text = str(row["text"])
ids = [char2idx[ch] for ch in text if ch in char2idx and char2idx[ch] not in (0, 1, 2)]
min_steps = len(ids) + sum(ids[i] == ids[i-1] for i in range(1, len(ids)))

print("Text characters:", len(text))
print("Encoded target length:", len(ids))
print("Minimum CTC steps:", min_steps)
print("CTC capacity 128:", min_steps <= 128)

# Verify encoder libraries and MambaVision availability
import timm
from mambavision import create_model as create_mambavision

print("timm:", timm.__version__)
print("MambaVision factory: available")
print("Preflight complete. No training performed.")

GPU: Tesla T4
Train rows: 20746
Vocab: 169
Image: 26102.tif | size: (700, 50) | mode: RGB
Text characters: 75
Encoded target length: 75
Minimum CTC steps: 75
CTC capacity 128: True
timm: 1.0.30
MambaVision factory: available
Preflight complete. No training performed.


In [13]:

# ============================================================
# PFMS-Net | SAFE RUN SETUP
# No training | No deletion | No overwriting existing files
# ============================================================
import os
import json
import shutil
import hashlib
from pathlib import Path
from datetime import datetime

assert "train_df" in globals(), "train_df is missing"
assert "val_df" in globals(), "val_df is missing"
assert "test_df" in globals(), "test_df is missing"

LOCAL_ROOT = Path("/content/PFMS_ABLATION_FINAL")
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

# Use a unique run directory so previous results are preserved.
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = LOCAL_ROOT / f"run_{run_id}"
RUN_DIR.mkdir(parents=True, exist_ok=False)

def disk_report(path):
    usage = shutil.disk_usage(path)
    return {
        "path": str(path),
        "free_GB": round(usage.free / 1024**3, 2),
        "used_GB": round(usage.used / 1024**3, 2),
        "total_GB": round(usage.total / 1024**3, 2),
    }

print("LOCAL DISK:", disk_report("/content"))

drive_root = Path("/content/drive/MyDrive")
drive_ok = drive_root.exists()

if drive_ok:
    print("DRIVE:", disk_report(drive_root))
else:
    print("Drive not mounted; local-only setup.")

# Validate split independence before saving.
def image_names(df):
    if "image_name" in df.columns:
        return set(df["image_name"].astype(str))
    return set(Path(str(x)).name for x in df["image_path"])

tr_names = image_names(train_df)
va_names = image_names(val_df)
te_names = image_names(test_df)

assert not (tr_names & va_names), "Train/validation overlap!"
assert not (tr_names & te_names), "Train/test overlap!"
assert not (va_names & te_names), "Validation/test overlap!"

# Save split manifests locally. These are small CSV files.
for split_name, frame in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df),
]:
    frame.to_csv(RUN_DIR / f"{split_name}_manifest.csv", index=False)

manifest = {
    "run_id": run_id,
    "created_at": datetime.now().isoformat(),
    "train_rows": len(train_df),
    "validation_rows": len(val_df),
    "test_rows": len(test_df),
    "train_validation_overlap": len(tr_names & va_names),
    "train_test_overlap": len(tr_names & te_names),
    "validation_test_overlap": len(va_names & te_names),
    "seed": 42,
    "image_size": [64, 1024],
    "vocabulary_size": len(char2idx),
    "note": "Test split reserved for final evaluation."
}

manifest_path = RUN_DIR / "run_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

# Hash split contents for later reproducibility checks.
for split_name in ["train", "validation", "test"]:
    f = RUN_DIR / f"{split_name}_manifest.csv"
    digest = hashlib.sha256(f.read_bytes()).hexdigest()
    print(f"{split_name} manifest SHA256:", digest[:20])

print("\nRUN DIRECTORY:", RUN_DIR)
print("Rows:", len(train_df), len(val_df), len(test_df))
print("Overlap:", len(tr_names & va_names),
      len(tr_names & te_names), len(va_names & te_names))
print("Local manifests saved.")
print("No model training performed.")

# Only attempt a small manifest backup if Drive has sufficient free space.
if drive_ok:
    try:
        free_gb = shutil.disk_usage(drive_root).free / 1024**3
        if free_gb >= 0.10:
            backup_dir = drive_root / "PFMS_ABLATION_MANIFEST_BACKUPS"
            backup_dir.mkdir(parents=True, exist_ok=True)
            target = backup_dir / f"run_{run_id}"
            shutil.copytree(RUN_DIR, target)
            print("Manifest backup:", target)
        else:
            print("Drive backup skipped: less than 0.10 GB free.")
    except Exception as e:
        print("Drive backup failed safely:", repr(e))
        print("Local manifests remain available for this session.")


LOCAL DISK: {'path': '/content', 'free_GB': 68.88, 'used_GB': 43.74, 'total_GB': 112.64}
DRIVE: {'path': '/content/drive/MyDrive', 'free_GB': 0.0, 'used_GB': 15.0, 'total_GB': 15.0}
train manifest SHA256: 0723f5b71078272e5735
validation manifest SHA256: a609dc94175cd9d97623
test manifest SHA256: 308b0563a909a911397b

RUN DIRECTORY: /content/PFMS_ABLATION_FINAL/run_20261027_093203
Rows: 20746 2305 4066
Overlap: 0 0 0
Local manifests saved.
No model training performed.
Drive backup skipped: less than 0.10 GB free.


In [14]:
# PFMS-Net | SAFE DIAGNOSTIC PRECHECK
# No training, no file deletion, no checkpoint overwrite

import torch
from pathlib import Path

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("CUDA available:", torch.cuda.is_available())

required = [
    "train_df", "val_df", "test_df",
    "char2idx", "idx2char",
]
for name in required:
    print(f"{name}: {'OK' if name in globals() else 'MISSING'}")

print("\nModel and loader objects:")
for name in [
    "OCRAblationModel", "train_loader", "val_loader",
    "encode_text", "decode_ctc",
]:
    print(f"{name}: {'OK' if name in globals() else 'MISSING'}")

print("\nLocal run directory:")
p = Path("/content/PFMS_ABLATION_FINAL/run_20261027_093203")
print("Exists:", p.exists())
if p.exists():
    for f in sorted(p.iterdir()):
        print(f.name, "|", f.stat().st_size, "bytes")

if torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print("\nGPU free memory (GB):", round(free_bytes / 1024**3, 2))
    print("GPU total memory (GB):", round(total_bytes / 1024**3, 2))

print("\nPrecheck finished. No training performed.")

GPU: Tesla T4
CUDA available: True
train_df: OK
val_df: OK
test_df: OK
char2idx: OK
idx2char: OK

Model and loader objects:
OCRAblationModel: MISSING
train_loader: MISSING
val_loader: MISSING
encode_text: MISSING
decode_ctc: MISSING

Local run directory:
Exists: True
run_manifest.json | 377 bytes
test_manifest.csv | 927567 bytes
train_manifest.csv | 4775656 bytes
validation_manifest.csv | 529958 bytes

GPU free memory (GB): 14.33
GPU total memory (GB): 14.56

Precheck finished. No training performed.


In [15]:
# PFMS-Net | ENCODER GEOMETRY CHECK
# No training and no file writes

import torch
import timm
from mambavision import create_model as create_mambavision

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.zeros(1, 3, 64, 1024, device=device)

# MambaVision: run all four backbone stages
mamba = create_mambavision(
    "mamba_vision_T", pretrained=False
).to(device).eval()

with torch.inference_mode():
    z = mamba.patch_embed(x)
    print("Mamba after patch_embed:", tuple(z.shape))
    for i, level in enumerate(mamba.levels):
        z = level(z)
        print(f"Mamba stage {i+1}:", tuple(z.shape))

# Swin: convert NHWC to NCHW when needed
swin = timm.create_model(
    "swin_tiny_patch4_window7_224",
    pretrained=False,
    features_only=True,
    out_indices=(0,),
    img_size=(64, 1024),
).to(device).eval()

with torch.inference_mode():
    s = swin(x)[0]
    channels = swin.feature_info.channels()[0]
    if s.shape[-1] == channels:
        s = s.permute(0, 3, 1, 2).contiguous()
    elif s.shape[1] != channels:
        raise RuntimeError(f"Unknown Swin layout: {tuple(s.shape)}")

    print("Swin NCHW:", tuple(s.shape))

print("\nMamba stage-1 width:", z.shape[-1] if z.ndim == 4 else "check output")
print("Swin width:", s.shape[-1])
print("Geometry check complete. No training performed.")

del mamba, swin, x, z, s
torch.cuda.empty_cache()

Mamba after patch_embed: (1, 80, 16, 256)
Mamba stage 1: (1, 160, 8, 128)
Mamba stage 2: (1, 320, 4, 64)
Mamba stage 3: (1, 640, 2, 32)
Mamba stage 4: (1, 640, 2, 32)
Swin NCHW: (1, 96, 16, 256)

Mamba stage-1 width: 32
Swin width: 256
Geometry check complete. No training performed.


In [16]:
# PFMS-Net | ONE-SAMPLE ENCODER TEST
# No training, no file writes

import torch
import timm
from PIL import Image
from pathlib import Path
from torchvision import transforms
from mambavision import create_model as create_mambavision

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

row = train_df.iloc[0]
img_path = Path(str(row["image_path"]))
if not img_path.is_file():
    img_path = Path(
        "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl"
    ) / str(row["image_name"])

assert img_path.is_file(), f"Image not found: {img_path}"

transform = transforms.Compose([
    transforms.Resize((64, 1024)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    ),
])

with Image.open(img_path) as im:
    x = transform(im.convert("RGB")).unsqueeze(0).to(device)

mamba = create_mambavision(
    "mamba_vision_T", pretrained=False
).to(device).eval()

swin = timm.create_model(
    "swin_tiny_patch4_window7_224",
    pretrained=False,
    features_only=True,
    out_indices=(0,),
    img_size=(64, 1024),
).to(device).eval()

try:
    with torch.inference_mode():
        m = mamba.patch_embed(x)
        for level in mamba.levels:
            m = level(m)
            # Save stage 1 before later stages reduce width.
            if level is mamba.levels[0]:
                m_stage1 = m

        s = swin(x)[0]
        c = swin.feature_info.channels()[0]
        if s.shape[-1] == c:
            s = s.permute(0, 3, 1, 2).contiguous()

    print("Image tensor:", tuple(x.shape))
    print("Mamba stage 1:", tuple(m_stage1.shape))
    print("Mamba stage 4:", tuple(m.shape))
    print("Swin feature:", tuple(s.shape))
    print("Mamba stage-1 width:", m_stage1.shape[-1])
    print("Swin width:", s.shape[-1])
    print("Sample text length:", len(str(row["text"])))
    print("Single-sample test passed.")

finally:
    del mamba, swin, x
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Image tensor: (1, 3, 64, 1024)
Mamba stage 1: (1, 160, 8, 128)
Mamba stage 4: (1, 640, 2, 32)
Swin feature: (1, 96, 16, 256)
Mamba stage-1 width: 128
Swin width: 256
Sample text length: 75
Single-sample test passed.


In [17]:

# ============================================================
# PFMS-Net | ONE-BATCH CTC GRADIENT TEST
# Real MambaVision + Swin | No full training | No file writes
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from PIL import Image
from pathlib import Path
from torchvision import transforms
from mambavision import create_model as create_mambavision

assert "train_df" in globals() and "char2idx" in globals(), \
    "ابتدا سلول آماده‌سازی داده و واژگان را اجرا کنید."

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PAD_IDX, BOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3
BLANK_IDX = PAD_IDX
IMG_H, IMG_W = 64, 1024
D_MODEL = 256
SEQ_LEN = 128
VOCAB_SIZE = len(char2idx)

def encode_ctc(text):
    return [
        int(char2idx[ch]) for ch in str(text)
        if ch in char2idx and int(char2idx[ch]) not in
        (PAD_IDX, BOS_IDX, EOS_IDX)
    ]

class CrossAttention(nn.Module):
    def __init__(self, d=256, heads=4):
        super().__init__()
        self.norm_q = nn.LayerNorm(d)
        self.norm_kv = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(
            d, heads, batch_first=True
        )
        self.ffn = nn.Sequential(
            nn.LayerNorm(d),
            nn.Linear(d, 4*d),
            nn.GELU(),
            nn.Linear(4*d, d),
        )

    def forward(self, q, kv):
        qn = self.norm_q(q)
        kvn = self.norm_kv(kv)
        y = self.attn(qn, kvn, kvn, need_weights=False)[0]
        q = q + y
        return q + self.ffn(q)

class SafePFMSCTC(nn.Module):
    """
    Diagnostic CTC model, NOT the complete manuscript decoder.
    MambaVision stage 1 and Swin stage 1.
    """
    def __init__(self, variant="full_pfms"):
        super().__init__()
        self.variant = variant

        self.mamba = create_mambavision(
            "mamba_vision_T", pretrained=False
        )
        self.swin = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=False,
            features_only=True,
            out_indices=(0,),
            img_size=(IMG_H, IMG_W),
        )

        self.m_proj = nn.Conv2d(160, D_MODEL, 1)
        self.s_proj = nn.Conv2d(96, D_MODEL, 1)

        self.cross_ms = CrossAttention(D_MODEL)
        self.cross_sm = CrossAttention(D_MODEL)
        self.refine = nn.Sequential(
            nn.Conv1d(D_MODEL, D_MODEL, 3, padding=1),
            nn.GELU(),
            nn.Conv1d(D_MODEL, D_MODEL, 3, padding=1),
        )
        self.classifier = nn.Linear(D_MODEL, VOCAB_SIZE)

    def forward(self, x):
        # MambaVision stage 1
        m = self.mamba.patch_embed(x)
        m = self.mamba.levels[0](m)   # [B,160,8,128]

        # Swin returns NHWC in this timm configuration
        s = self.swin(x)[0]            # [B,16,256,96]
        if s.shape[-1] == 96:
            s = s.permute(0, 3, 1, 2).contiguous()

        m = self.m_proj(m)             # [B,256,8,128]
        s = self.s_proj(s)             # [B,256,16,256]

        # Preserve horizontal ordering; average only over height.
        m = m.mean(dim=2).transpose(1, 2)  # [B,128,256]
        s = s.mean(dim=2).transpose(1, 2)  # [B,256,256]

        if self.variant == "mamba_only":
            seq = m
        elif self.variant == "swin_only":
            seq = s
        elif self.variant == "simple_fusion":
            # Interpolate to common width for a controlled fusion baseline.
            s2 = F.interpolate(
                s.transpose(1, 2),
                size=m.shape[1],
                mode="linear",
                align_corners=False,
            ).transpose(1, 2)
            seq = (m + s2) / 2
        else:
            # Bidirectional cross-attention, aligned at Mamba width.
            s2 = F.interpolate(
                s.transpose(1, 2),
                size=m.shape[1],
                mode="linear",
                align_corners=False,
            ).transpose(1, 2)
            ms = self.cross_ms(m, s2)
            sm = self.cross_sm(s2, m)
            seq = (ms + sm) / 2
            if self.variant == "full_pfms":
                seq = seq + self.refine(seq.transpose(1, 2)).transpose(1, 2)

        # Keep the true encoder sequence length; do NOT force 128
        # for Swin-only. This makes CTC lengths explicit.
        return self.classifier(seq)    # [B,T,V]

# Choose full cross-attention path for the diagnostic only.
model = SafePFMSCTC("full_pfms").to(device)
model.train()

row = train_df.iloc[0]
img_path = Path(str(row["image_path"]))
if not img_path.is_file():
    img_path = Path(
        "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl"
    ) / str(row["image_name"])
assert img_path.is_file(), f"Image not found: {img_path}"

transform = transforms.Compose([
    transforms.Resize((IMG_H, IMG_W)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225),
    ),
])

with Image.open(img_path) as im:
    image = transform(im.convert("RGB")).unsqueeze(0).to(device)

target_ids = encode_ctc(row["text"])
target = torch.tensor(target_ids, dtype=torch.long, device=device)
target_lengths = torch.tensor([len(target_ids)], dtype=torch.long)
input_lengths = None
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
ctc = nn.CTCLoss(blank=BLANK_IDX, zero_infinity=False)

try:
    optimizer.zero_grad(set_to_none=True)
    logits = model(image)
    input_lengths = torch.tensor([logits.shape[1]], dtype=torch.long)

    min_steps = len(target_ids) + sum(
        target_ids[i] == target_ids[i-1]
        for i in range(1, len(target_ids))
    )
    print("Logits:", tuple(logits.shape))
    print("Target length:", len(target_ids))
    print("Minimum CTC steps:", min_steps)
    print("Available CTC steps:", int(input_lengths.item()))

    assert logits.shape[-1] == VOCAB_SIZE
    assert input_lengths.item() >= min_steps, \
        "CTC sequence too short for this target."

    log_probs = logits.float().log_softmax(dim=-1).transpose(0, 1)
    loss = ctc(
        log_probs,
        target,
        input_lengths,
        target_lengths,
    )
    print("CTC loss:", float(loss.detach().cpu()))
    assert torch.isfinite(loss), "CTC loss is not finite."

    loss.backward()
    grads = [
        p.grad.detach().norm().item()
        for p in model.parameters()
        if p.grad is not None
    ]
    nonzero_grads = sum(g > 0 for g in grads)
    print("Parameters with gradients:", len(grads))
    print("Non-zero gradient norms:", nonzero_grads)
    assert nonzero_grads > 0, "No non-zero gradients detected."

    print("\nPASS: one-sample forward/backward succeeded.")
    print("No optimizer step taken; no model training performed.")

finally:
    del model, image, logits, loss, optimizer, ctc
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Logits: (1, 128, 169)
Target length: 75
Minimum CTC steps: 75
Available CTC steps: 128
CTC loss: 7.456085681915283
Parameters with gradients: 83
Non-zero gradient norms: 83

PASS: one-sample forward/backward succeeded.
No optimizer step taken; no model training performed.


In [18]:

# ============================================================
# PFMS-Net Research | MambaVision-only CTC Pilot
# Real MambaVision-T | 100 training batches maximum
# Local atomic checkpoints | No Google Drive writes/deletions
# ============================================================

import os, csv, time, math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from mambavision import create_model as create_mambavision

# ---------- 1. Preconditions ----------
assert "train_df" in globals(), "ابتدا سلول آماده‌سازی train_df را اجرا کنید."
assert "val_df" in globals(), "ابتدا سلول آماده‌سازی val_df را اجرا کنید."
assert "char2idx" in globals(), "واژگان char2idx موجود نیست."

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PAD_IDX, BOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3
BLANK_IDX = PAD_IDX
IMG_H, IMG_W = 64, 1024
D_MODEL = 256
BATCH_SIZE = 4
MAX_TRAIN_BATCHES = 100
MAX_VAL_BATCHES = 20
LR = 1e-4
WEIGHT_DECAY = 0.01
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Checkpoint is on the temporary local Colab disk.
OUT_DIR = Path("/content/PFMS_ABLATION_FINAL/mambavision_pilot")
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUT_DIR / "pilot_log.csv"
CKPT_PATH = OUT_DIR / "mambavision_pilot_latest.pt"

# ---------- 2. Vocabulary validation ----------
def encode_text(text):
    text = str(text)
    missing = sorted(set(text) - set(char2idx))
    if missing:
        raise ValueError(
            f"کاراکتر خارج از واژگان یافت شد: {missing[:20]}"
        )
    ids = [
        int(char2idx[ch]) for ch in text
        if int(char2idx[ch]) not in (PAD_IDX, BOS_IDX, EOS_IDX)
    ]
    min_steps = len(ids) + sum(
        ids[i] == ids[i-1] for i in range(1, len(ids))
    )
    if min_steps > 128:
        raise ValueError(
            f"هدف با CTC سازگار نیست: min_steps={min_steps}"
        )
    return ids

# ---------- 3. Dataset ----------
IMAGE_ROOT = Path(
    "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl"
)

transform = transforms.Compose([
    transforms.Resize((IMG_H, IMG_W)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    ),
])

class PrintedOCRDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True).copy()
        required = {"text", "image_name"}
        if not required.issubset(self.frame.columns):
            raise ValueError(
                f"ستون‌های لازم موجود نیستند: {required - set(self.frame.columns)}"
            )
        self.targets = [
            encode_text(t) for t in self.frame["text"].tolist()
        ]

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        p = Path(str(row["image_path"])) if "image_path" in row.index else Path("__missing__")
        if not p.is_file():
            p = IMAGE_ROOT / str(row["image_name"])
        if not p.is_file():
            raise FileNotFoundError(f"تصویر یافت نشد: {p}")

        with Image.open(p) as im:
            image = transform(im.convert("RGB"))

        target = torch.tensor(self.targets[idx], dtype=torch.long)
        return image, target

def collate_ctc(batch):
    images, targets = zip(*batch)
    images = torch.stack(images)
    lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    flat_targets = torch.cat(targets) if sum(lengths).item() else torch.empty(0, dtype=torch.long)
    return images, flat_targets, lengths

train_ds = PrintedOCRDataset(train_df)
val_ds = PrintedOCRDataset(val_df)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=(device.type == "cuda"),
    collate_fn=collate_ctc
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=(device.type == "cuda"),
    collate_fn=collate_ctc
)

print("Train samples:", len(train_ds))
print("Validation samples:", len(val_ds))
print("Batch size:", BATCH_SIZE)
print("Vocabulary size:", len(char2idx))

# ---------- 4. Real MambaVision-T encoder + CTC head ----------
class MambaVisionCTC(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.encoder = create_mambavision(
            "mamba_vision_T", pretrained=False
        )
        self.projection = nn.Conv1d(160, D_MODEL, kernel_size=1)
        self.refine = nn.Sequential(
            nn.Conv1d(D_MODEL, D_MODEL, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Conv1d(D_MODEL, D_MODEL, kernel_size=3, padding=1),
            nn.GELU(),
        )
        self.classifier = nn.Linear(D_MODEL, vocab_size)

    def forward(self, x):
        x = self.encoder.patch_embed(x)
        x = self.encoder.levels[0](x)  # [B,160,8,128]
        x = x.mean(dim=2)              # [B,160,128]
        x = self.projection(x)         # [B,256,128]
        x = x + self.refine(x)
        x = x.transpose(1, 2)          # [B,128,256]
        return self.classifier(x)      # [B,128,V]

model = MambaVisionCTC(len(char2idx)).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)
criterion = nn.CTCLoss(blank=BLANK_IDX, zero_infinity=False)

print("Trainable parameters:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))
print("Checkpoint directory:", OUT_DIR)

# ---------- 5. Greedy decoding and metrics ----------
def greedy_decode(logits):
    pred = logits.argmax(dim=-1)[0].detach().cpu().tolist()
    result, previous = [], None
    for token in pred:
        if token != previous and token != BLANK_IDX:
            result.append(token)
        previous = token
    return result

def ids_to_text(ids):
    chars = []
    for token in ids:
        ch = idx2char.get(int(token), "")
        if ch and int(token) not in (PAD_IDX, BOS_IDX, EOS_IDX):
            chars.append(ch)
    return "".join(chars)

def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(
                cur[-1] + 1,
                prev[j] + 1,
                prev[j-1] + (ca != cb)
            ))
        prev = cur
    return prev[-1]

def save_checkpoint(step, train_loss):
    temp_path = CKPT_PATH.with_suffix(".tmp")
    torch.save({
        "step": step,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "model_name": "MambaVision-T-stage1-CTC-pilot",
        "seed": SEED,
        "vocab_size": len(char2idx),
        "image_size": [IMG_H, IMG_W],
        "note": "Pilot only; not the complete PFMS-Net decoder."
    }, temp_path)
    os.replace(temp_path, CKPT_PATH)

def append_log(record):
    exists = LOG_PATH.exists()
    with open(LOG_PATH, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(record.keys()))
        if not exists:
            writer.writeheader()
        writer.writerow(record)

# ---------- 6. Short training pilot ----------
model.train()
train_iter = iter(train_loader)
running_loss, successful_steps = 0.0, 0
start_time = time.time()
last_loss = None

for step in range(1, MAX_TRAIN_BATCHES + 1):
    try:
        images, targets, target_lengths = next(train_iter)
    except StopIteration:
        print("Training loader exhausted.")
        break

    images = images.to(device, non_blocking=True)
    targets = targets.to(device, non_blocking=True)
    target_lengths = target_lengths.to(device)

    optimizer.zero_grad(set_to_none=True)
    logits = model(images)

    batch_size = images.shape[0]
    input_lengths = torch.full(
        (batch_size,), logits.shape[1], dtype=torch.long
    )

    log_probs = logits.float().log_softmax(-1).transpose(0, 1)
    loss = criterion(
        log_probs, targets, input_lengths, target_lengths
    )

    if not torch.isfinite(loss):
        raise RuntimeError(
            f"Loss نامتناهی در batch شماره {step}: {loss.item()}"
        )

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
    optimizer.step()

    last_loss = float(loss.detach().cpu())
    running_loss += last_loss
    successful_steps += 1

    if step == 1 or step % 10 == 0:
        print(
            f"Step {step:03d}/{MAX_TRAIN_BATCHES} | "
            f"loss={last_loss:.4f} | "
            f"elapsed={time.time()-start_time:.1f}s"
        )

    # Atomic checkpoint every 25 steps.
    if step % 25 == 0:
        save_checkpoint(step, last_loss)
        append_log({
            "step": step,
            "train_loss": last_loss,
            "elapsed_seconds": round(time.time()-start_time, 2)
        })
        print("Checkpoint saved:", CKPT_PATH)

    if step % 25 == 0 and torch.cuda.is_available():
        torch.cuda.empty_cache()

if successful_steps == 0:
    raise RuntimeError("هیچ batch آموزشی با موفقیت اجرا نشد.")

save_checkpoint(successful_steps, last_loss)

# ---------- 7. Limited validation ----------
model.eval()
val_loss_sum, val_batches = 0.0, 0
char_edits, char_total = 0, 0
word_edits, word_total = 0, 0
exact_matches = 0
examples = []

with torch.no_grad():
    for batch_idx, (images, targets, target_lengths) in enumerate(val_loader):
        if batch_idx >= MAX_VAL_BATCHES:
            break

        images = images.to(device, non_blocking=True)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)

        logits = model(images)
        input_lengths = torch.full(
            (images.shape[0],), logits.shape[1], dtype=torch.long
        )
        log_probs = logits.float().log_softmax(-1).transpose(0, 1)
        loss = criterion(
            log_probs, targets, input_lengths, target_lengths
        )
        val_loss_sum += float(loss.cpu())
        val_batches += 1

        offset = 0
        for i in range(images.shape[0]):
            n = int(target_lengths[i].item())
            true_ids = targets[offset:offset+n].detach().cpu().tolist()
            offset += n

            pred_ids = greedy_decode(logits[i:i+1])
            true_text = ids_to_text(true_ids)
            pred_text = ids_to_text(pred_ids)

            char_edits += edit_distance(pred_text, true_text)
            char_total += max(1, len(true_text))

            true_words = true_text.split()
            pred_words = pred_text.split()
            word_edits += edit_distance(pred_words, true_words)
            word_total += max(1, len(true_words))

            exact_matches += int(pred_text == true_text)
            if len(examples) < 5:
                examples.append((true_text, pred_text))

val_cer = char_edits / max(1, char_total)
val_wer = word_edits / max(1, word_total)
val_exact = exact_matches / max(1, min(
    len(val_ds), val_batches * BATCH_SIZE
))

summary = {
    "train_steps": successful_steps,
    "mean_train_loss": running_loss / successful_steps,
    "validation_batches": val_batches,
    "validation_loss": val_loss_sum / max(1, val_batches),
    "validation_CER": val_cer,
    "validation_WER": val_wer,
    "validation_exact_match": val_exact,
    "elapsed_seconds": time.time() - start_time,
    "checkpoint": str(CKPT_PATH),
    "log": str(LOG_PATH),
}

print("\n========== PILOT SUMMARY ==========")
for k, v in summary.items():
    print(f"{k}: {v}")

print("\n========== OCR EXAMPLES ==========")
for i, (truth, pred) in enumerate(examples, 1):
    print(f"\nExample {i}\nTRUE: {truth}\nPRED: {pred}")

print("\nPilot complete. These are preliminary results, not final paper metrics.")
print("Checkpoint:", CKPT_PATH)
print("CSV log:", LOG_PATH)


Device: cuda
GPU: Tesla T4
Train samples: 20746
Validation samples: 2305
Batch size: 4
Vocabulary size: 169
Trainable parameters: 32272625
Checkpoint directory: /content/PFMS_ABLATION_FINAL/mambavision_pilot
Step 001/100 | loss=7.0155 | elapsed=0.3s
Step 010/100 | loss=6.5381 | elapsed=0.6s
Step 020/100 | loss=5.0084 | elapsed=0.8s
Checkpoint saved: /content/PFMS_ABLATION_FINAL/mambavision_pilot/mambavision_pilot_latest.pt
Step 030/100 | loss=3.9351 | elapsed=1.3s
Step 040/100 | loss=3.3620 | elapsed=1.5s
Step 050/100 | loss=3.1438 | elapsed=1.8s
Checkpoint saved: /content/PFMS_ABLATION_FINAL/mambavision_pilot/mambavision_pilot_latest.pt
Step 060/100 | loss=3.1701 | elapsed=2.4s
Step 070/100 | loss=3.3374 | elapsed=2.7s
Checkpoint saved: /content/PFMS_ABLATION_FINAL/mambavision_pilot/mambavision_pilot_latest.pt
Step 080/100 | loss=3.2700 | elapsed=3.3s
Step 090/100 | loss=3.0883 | elapsed=3.6s
Step 100/100 | loss=3.0341 | elapsed=3.9s
Checkpoint saved: /content/PFMS_ABLATION_FINAL/mamb

In [19]:

import torch
from collections import Counter

assert "model" in globals(), "مدل پایلوت در حافظه نیست؛ سلول آموزش را دوباره اجرا نکنید."
assert "val_ds" in globals(), "مجموعه اعتبارسنجی موجود نیست."
assert "val_loader" in globals(), "بارگذار اعتبارسنجی موجود نیست."
assert "idx2char" in globals(), "واژگان معکوس idx2char موجود نیست."

device = next(model.parameters()).device
model.eval()

raw_counts = Counter()
decoded_counts = Counter()
blank_probabilities = []
examples = []

with torch.no_grad():
    for batch_i, (images, targets, target_lengths) in enumerate(val_loader):
        if batch_i >= 5:  # فقط 20 تصویر
            break

        images = images.to(device)
        logits = model(images)
        probs = logits.float().softmax(dim=-1)
        raw_ids = logits.argmax(dim=-1).cpu()

        blank_probabilities.extend(
            probs[..., 0].mean(dim=1).cpu().tolist()
        )

        offset = 0
        for i in range(images.shape[0]):
            n = int(target_lengths[i])
            true_ids = targets[offset:offset+n].tolist()
            offset += n

            pred_ids = raw_ids[i].tolist()
            raw_counts.update(pred_ids)

            # CTC collapse: remove repeated adjacent IDs and blanks.
            collapsed = []
            previous = None
            for token in pred_ids:
                if token != previous and token != 0:
                    collapsed.append(token)
                previous = token

            decoded_counts.update(collapsed)

            if len(examples) < 5:
                truth = ids_to_text(true_ids)
                pred = ids_to_text(collapsed)
                examples.append((truth, pred, pred_ids[:30]))

print("Blank probability mean:", round(
    sum(blank_probabilities) / max(1, len(blank_probabilities)), 4
))
print("Raw argmax token counts:", raw_counts.most_common(15))
print("Decoded token counts (excluding blank):", decoded_counts.most_common(15))
print("Vocabulary sample:", list(char2idx.items())[:20])
print("idx2char sample:", list(idx2char.items())[:20])

for i, (truth, pred, raw_first) in enumerate(examples, 1):
    print(f"\nExample {i}")
    print("TRUE:", truth)
    print("PRED:", repr(pred))
    print("First 30 raw token IDs:", raw_first)

print("\nDiagnostic finished; no weights or files changed.")


Blank probability mean: 0.3861
Raw argmax token counts: [(0, 2560)]
Decoded token counts (excluding blank): []
Vocabulary sample: [('<PAD>', 0), ('<BOS>', 1), ('<EOS>', 2), ('<UNK>', 3), ('\n', 4), (' ', 5), ('!', 6), ('%', 7), ('(', 8), (')', 9), ('*', 10), (',', 11), ('-', 12), ('.', 13), ('/', 14), (':', 15), ('?', 16), ('[', 17), (']', 18), ('`', 19)]
idx2char sample: [(0, '<PAD>'), (1, '<BOS>'), (2, '<EOS>'), (3, '<UNK>'), (4, '\n'), (5, ' '), (6, '!'), (7, '%'), (8, '('), (9, ')'), (10, '*'), (11, ','), (12, '-'), (13, '.'), (14, '/'), (15, ':'), (16, '?'), (17, '['), (18, ']'), (19, '`')]

Example 1
TRUE: راه نجات انقلاب است. یک شریعه و فراتی است از عاشورای اباعبدالله الحسین (ع) و

PRED: ''
First 30 raw token IDs: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Example 2
TRUE: دلیل رفتار غیرورزشی نسبت به تیم داوری به مدت یک جلسه از همراهی تیم خود

PRED: ''
First 30 raw token IDs: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [20]:

import torch
import torch.nn.functional as F

model.eval()
device = next(model.parameters()).device

images, targets, target_lengths = next(iter(train_loader))
images = images[:2].to(device)
target_lengths = target_lengths[:2]
targets = targets.to(device)

with torch.no_grad():
    logits = model(images)
    log_probs = logits.float().log_softmax(dim=-1)
    probs = log_probs.exp()

print("Logits shape:", tuple(logits.shape))
print("Logits finite:", bool(torch.isfinite(logits).all()))
print("Logits min/max:", float(logits.min()), float(logits.max()))
print("Mean probability blank:", float(probs[..., 0].mean()))
print("Mean probability nonblank:", float(probs[..., 1:].sum(-1).mean()))

for i in range(images.shape[0]):
    print(f"\nSample {i+1}:")
    for t in [0, 1, 5, 10, 20, 40, 80, 120]:
        if t >= logits.shape[1]:
            continue
        top = torch.topk(probs[i, t], k=5)
        items = []
        for idx, p in zip(top.indices.tolist(), top.values.tolist()):
            items.append((idx, repr(idx2char.get(idx, "?")), round(p, 4)))
        print("time", t, "top-5:", items)

# Check the target IDs are valid and do not contain special tokens.
offset = 0
for i in range(len(images)):
    n = int(target_lengths[i])
    ids = targets[offset:offset+n].detach().cpu().tolist()
    offset += n
    print(
        f"Target {i+1}: length={n}, "
        f"min_id={min(ids) if ids else None}, "
        f"max_id={max(ids) if ids else None}, "
        f"special_ids_present={any(x in (0,1,2) for x in ids)}"
    )

print("\nDiagnostic complete. No optimizer step or checkpoint write.")


Logits shape: (2, 128, 169)
Logits finite: True
Logits min/max: -5.1992597579956055 6.986242771148682
Mean probability blank: 0.3914414644241333
Mean probability nonblank: 0.6085585355758667

Sample 1:
time 0 top-5: [(0, "'<PAD>'", 0.2661), (5, "' '", 0.0702), (35, "'ا'", 0.0485), (60, "'م'", 0.0402), (36, "'ب'", 0.0376)]
time 1 top-5: [(0, "'<PAD>'", 0.3903), (5, "' '", 0.0998), (35, "'ا'", 0.0645), (90, "'ی'", 0.0428), (45, "'ر'", 0.0334)]
time 5 top-5: [(0, "'<PAD>'", 0.3367), (5, "' '", 0.1449), (35, "'ا'", 0.0608), (90, "'ی'", 0.0434), (45, "'ر'", 0.0365)]
time 10 top-5: [(0, "'<PAD>'", 0.3721), (5, "' '", 0.1395), (35, "'ا'", 0.0588), (90, "'ی'", 0.0422), (61, "'ن'", 0.0344)]
time 20 top-5: [(0, "'<PAD>'", 0.3475), (5, "' '", 0.1353), (35, "'ا'", 0.0601), (90, "'ی'", 0.0442), (45, "'ر'", 0.0371)]
time 40 top-5: [(0, "'<PAD>'", 0.3649), (5, "' '", 0.1377), (35, "'ا'", 0.0613), (90, "'ی'", 0.0417), (45, "'ر'", 0.0344)]
time 80 top-5: [(0, "'<PAD>'", 0.3943), (5, "' '", 0.1379), (35

In [21]:

import torch

model.eval()
device = next(model.parameters()).device

images, targets, target_lengths = next(iter(val_loader))
images = images[:2].to(device)

with torch.no_grad():
    logits = model(images).float()
    probs = logits.softmax(dim=-1)

    # تفاوت توزیع احتمالات دو تصویر
    mean_abs_diff = (probs[0] - probs[1]).abs().mean().item()

    # تفاوت خروجی در هر موقعیت
    pred0 = logits[0].argmax(dim=-1)
    pred1 = logits[1].argmax(dim=-1)
    same_ratio = (pred0 == pred1).float().mean().item()

    # تعداد توکن‌های غیر blank در خروجی خام
    nonblank0 = int((pred0 != 0).sum().item())
    nonblank1 = int((pred1 != 0).sum().item())

    # تنوع احتمالات در هر موقعیت
    entropy = -(probs * probs.clamp_min(1e-12).log()).sum(dim=-1).mean().item()

print("Mean absolute probability difference:", mean_abs_diff)
print("Same argmax ratio across two images:", same_ratio)
print("Nonblank positions, image 1:", nonblank0)
print("Nonblank positions, image 2:", nonblank1)
print("Mean predictive entropy:", entropy)
print("No weights changed.")


Mean absolute probability difference: 0.0001948433491634205
Same argmax ratio across two images: 1.0
Nonblank positions, image 1: 0
Nonblank positions, image 2: 0
Mean predictive entropy: 2.576709508895874
No weights changed.


In [22]:

import torch

model.eval()
device = next(model.parameters()).device

images, _, _ = next(iter(val_loader))
images = images[:2].to(device)

with torch.no_grad():
    features = []
    for i in range(2):
        z = model.encoder.patch_embed(images[i:i+1])
        z = model.encoder.levels[0](z)
        features.append(z.float())

    f0, f1 = features
    feature_diff = (f0 - f1).abs().mean().item()
    feature_std0 = f0.std().item()
    feature_std1 = f1.std().item()

print("Feature shape:", tuple(f0.shape))
print("Mean absolute feature difference:", feature_diff)
print("Image 1 feature std:", feature_std0)
print("Image 2 feature std:", feature_std1)
print("Images are pixel-identical:", torch.equal(images[0], images[1]))
print("No weights changed.")


Feature shape: (1, 160, 8, 128)
Mean absolute feature difference: 0.14264270663261414
Image 1 feature std: 1.139338493347168
Image 2 feature std: 1.1489826440811157
Images are pixel-identical: False
No weights changed.


In [23]:

# Small-set overfit test: 8 fixed samples, separate checkpoint
# Existing pilot checkpoint remains untouched.

import os, torch
import torch.nn as nn
from pathlib import Path

assert "MambaVisionCTC" in globals()
device = next(model.parameters()).device

torch.manual_seed(42)
torch.cuda.manual_seed_all(42) if torch.cuda.is_available() else None

small_ds = torch.utils.data.Subset(train_ds, list(range(8)))
small_loader = torch.utils.data.DataLoader(
    small_ds, batch_size=4, shuffle=False, num_workers=0,
    collate_fn=collate_ctc
)

# Fresh model: do not continue from the collapsed pilot.
small_model = MambaVisionCTC(len(char2idx)).to(device)
small_optimizer = torch.optim.AdamW(
    small_model.parameters(), lr=3e-4, weight_decay=0.0
)
small_ctc = nn.CTCLoss(blank=0, zero_infinity=False)

small_ckpt = Path(
    "/content/PFMS_ABLATION_FINAL/mambavision_pilot/overfit8_latest.pt"
)

for step in range(1, 101):
    small_model.train()
    total_loss = 0.0

    for images, targets, target_lengths in small_loader:
        images = images.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)
        small_optimizer.zero_grad(set_to_none=True)

        logits = small_model(images)
        input_lengths = torch.full(
            (images.shape[0],), logits.shape[1], dtype=torch.long
        )
        log_probs = logits.float().log_softmax(-1).transpose(0, 1)
        loss = small_ctc(
            log_probs, targets, input_lengths, target_lengths
        )

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss at step {step}")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(small_model.parameters(), 5.0)
        small_optimizer.step()
        total_loss += float(loss.detach().cpu())

    if step % 10 == 0 or step == 1:
        print(f"Epoch {step:03d}/100 | mean loss={total_loss/len(small_loader):.4f}")

    if step % 10 == 0:
        tmp = small_ckpt.with_suffix(".tmp")
        torch.save({
            "step": step,
            "model_state_dict": small_model.state_dict(),
            "optimizer_state_dict": small_optimizer.state_dict(),
            "loss": total_loss / len(small_loader),
            "note": "8-sample overfit diagnostic only"
        }, tmp)
        os.replace(tmp, small_ckpt)

print("Overfit test finished.")
print("Separate checkpoint:", small_ckpt)
print("Original pilot checkpoint was not modified.")


Epoch 001/100 | mean loss=7.1504
Epoch 010/100 | mean loss=3.2687
Epoch 020/100 | mean loss=3.0151
Epoch 030/100 | mean loss=2.9350
Epoch 040/100 | mean loss=2.8371
Epoch 050/100 | mean loss=2.7092
Epoch 060/100 | mean loss=2.5175
Epoch 070/100 | mean loss=2.1433
Epoch 080/100 | mean loss=1.7698
Epoch 090/100 | mean loss=1.4294
Epoch 100/100 | mean loss=1.2303
Overfit test finished.
Separate checkpoint: /content/PFMS_ABLATION_FINAL/mambavision_pilot/overfit8_latest.pt
Original pilot checkpoint was not modified.


In [24]:

import torch
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt_path = Path(
    "/content/PFMS_ABLATION_FINAL/mambavision_pilot/overfit8_latest.pt"
)
assert ckpt_path.is_file(), f"Checkpoint not found: {ckpt_path}"

# Load only the separate 8-sample diagnostic checkpoint.
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
small_model.load_state_dict(ckpt["model_state_dict"])
small_model.eval()

def ctc_greedy(logits):
    ids = logits.argmax(dim=-1)[0].detach().cpu().tolist()
    out, prev = [], None
    for token in ids:
        if token != prev and token != 0:
            out.append(token)
        prev = token
    return out

def decode_ids(ids):
    return "".join(
        idx2char.get(int(i), "")
        for i in ids
        if int(i) not in (0, 1, 2)
    )

with torch.no_grad():
    for i in range(8):
        image, target = train_ds[i]
        logits = small_model(image.unsqueeze(0).to(device))
        pred_ids = ctc_greedy(logits)
        true_text = decode_ids(target.tolist())
        pred_text = decode_ids(pred_ids)

        print(f"\nSample {i+1}")
        print("TRUE:", true_text)
        print("PRED:", repr(pred_text))
        print("Exact match:", pred_text == true_text)

print("\nDiagnostic only; no training or checkpoint modification.")



Sample 1
TRUE: تاریخ مخابره : ۱۳۹۶/۵/۲۳ - ۱۵:۵۸ ۹۶ میلیون پوند برای خرید فرش مساجد در مصر

PRED: 'رارخ مخابر : ۱۳۹۶/۲۳ - ۱۵:۵۸ ۹۶ مینی ون برا خری ر\n'
Exact match: False

Sample 2
TRUE: که برای اولین‌بار فیلم خبری ساخت. معتضدی اولین لابراتوار ظهور فیلم را نیز در زیرزمین

PRED: 'م برا وی\u200cبار فیم خاری ا معفد ر\n'
Exact match: False

Sample 3
TRUE: عدم رعایت موارد بهداشتی و ذبح در خارج از کشتارگاه توسط دامپزشکی قائم شهر ضبط

PRED: 'عزم رعات موار ب ارتی و ذبح ار خارا از بشتارگا وس ر\n'
Exact match: False

Sample 4
TRUE: بزرگتری از انرژی خورشید را به فضا بازتاب می كنند در نتیجه مقدار تابش خورشیدی

PRED: 'مقرگتر ا انرژی خری را ب فا بازار\n'
Exact match: False

Sample 5
TRUE: را ارائه ندادند و با این حساب ما به مشکل خوردیم و نتوانستیم آن طور

PRED: 'را اراه ندادند و با این حساب مار\n'
Exact match: False

Sample 6
TRUE: مقررات جاری ‌باید به حساب خزانه واریز شود و در مدل مالی مصوب جز درآمدهای

PRED: 'مقرر جاری \u200cباید به حساب خان واریر\n'
Exact match: False

Sample 7
TRUE: ممنوع است و 

In [25]:

# Evaluate the 8-sample overfit checkpoint using CER and WER.
import torch

def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(
                cur[-1] + 1,
                prev[j] + 1,
                prev[j-1] + (ca != cb)
            ))
        prev = cur
    return prev[-1]

char_edits = char_total = 0
word_edits = word_total = 0
exact = 0

small_model.eval()
with torch.no_grad():
    for i in range(8):
        image, target = train_ds[i]
        logits = small_model(image.unsqueeze(0).to(device))
        pred_text = decode_ids(ctc_greedy(logits))
        true_text = decode_ids(target.tolist())

        char_edits += edit_distance(pred_text, true_text)
        char_total += max(1, len(true_text))

        true_words = true_text.split()
        pred_words = pred_text.split()
        word_edits += edit_distance(pred_words, true_words)
        word_total += max(1, len(true_words))

        exact += int(pred_text == true_text)

print("Train-subset CER:", round(char_edits / max(1, char_total), 4))
print("Train-subset WER:", round(word_edits / max(1, word_total), 4))
print("Exact matches:", exact, "/ 8")
print("These are training-subset diagnostics, not validation metrics.")


Train-subset CER: 0.4579
Train-subset WER: 0.6833
Exact matches: 1 / 8
These are training-subset diagnostics, not validation metrics.


In [26]:

# ============================================================
# Continue 8-sample overfit diagnostic from saved checkpoint
# Separate checkpoint; original pilot remains untouched.
# ============================================================

import os
import torch
import torch.nn as nn
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

base_dir = Path("/content/PFMS_ABLATION_FINAL/mambavision_pilot")
source_ckpt = base_dir / "overfit8_latest.pt"
continued_ckpt = base_dir / "overfit8_continued_latest.pt"

assert source_ckpt.is_file(), f"Checkpoint not found: {source_ckpt}"
assert "small_model" in globals(), "مدل small_model در حافظه نیست."
assert "small_loader" in globals(), "small_loader موجود نیست."
assert "decode_ids" in globals() and "ctc_greedy" in globals()
assert "train_ds" in globals() and "char2idx" in globals()

# Restore model and optimizer from the existing diagnostic checkpoint.
checkpoint = torch.load(
    source_ckpt, map_location=device, weights_only=False
)
small_model.load_state_dict(checkpoint["model_state_dict"])

small_optimizer = torch.optim.AdamW(
    small_model.parameters(), lr=5e-5, weight_decay=0.0
)
if "optimizer_state_dict" in checkpoint:
    small_optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

criterion_continue = nn.CTCLoss(blank=0, zero_infinity=False)

def evaluate_eight():
    small_model.eval()
    char_edits = char_total = 0
    word_edits = word_total = 0
    exact = 0

    with torch.no_grad():
        for i in range(8):
            image, target = train_ds[i]
            logits = small_model(image.unsqueeze(0).to(device))
            pred_text = decode_ids(ctc_greedy(logits))
            true_text = decode_ids(target.tolist())

            char_edits += edit_distance(pred_text, true_text)
            char_total += max(1, len(true_text))
            word_edits += edit_distance(
                pred_text.split(), true_text.split()
            )
            word_total += max(1, len(true_text.split()))
            exact += int(pred_text == true_text)

    return (
        char_edits / max(1, char_total),
        word_edits / max(1, word_total),
        exact
    )

before = evaluate_eight()
print(
    f"Before continuation: CER={before[0]:.4f}, "
    f"WER={before[1]:.4f}, exact={before[2]}/8"
)

for epoch in range(1, 101):
    small_model.train()
    epoch_loss = 0.0

    for images, targets, target_lengths in small_loader:
        images = images.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)

        small_optimizer.zero_grad(set_to_none=True)
        logits = small_model(images)

        input_lengths = torch.full(
            (images.shape[0],), logits.shape[1], dtype=torch.long
        )
        log_probs = logits.float().log_softmax(-1).transpose(0, 1)
        loss = criterion_continue(
            log_probs, targets, input_lengths, target_lengths
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Loss is not finite at continuation epoch {epoch}"
            )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(small_model.parameters(), 5.0)
        small_optimizer.step()
        epoch_loss += float(loss.detach().cpu())

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Continuation epoch {epoch:03d}/100 | "
            f"loss={epoch_loss / len(small_loader):.4f}"
        )

    # Atomic save to a separate file.
    if epoch % 10 == 0:
        tmp = continued_ckpt.with_suffix(".tmp")
        torch.save({
            "continuation_epoch": epoch,
            "source_checkpoint": str(source_ckpt),
            "model_state_dict": small_model.state_dict(),
            "optimizer_state_dict": small_optimizer.state_dict(),
            "note": "8-sample overfit diagnostic; not final OCR metrics"
        }, tmp)
        os.replace(tmp, continued_ckpt)

after = evaluate_eight()
print("\n========== BEFORE / AFTER ==========")
print(f"CER:  {before[0]:.4f} -> {after[0]:.4f}")
print(f"WER:  {before[1]:.4f} -> {after[1]:.4f}")
print(f"Exact matches: {before[2]}/8 -> {after[2]}/8")
print("Separate checkpoint:", continued_ckpt)
print("Original checkpoints were not overwritten.")
print("Do not report this 8-image test as validation or test performance.")


Before continuation: CER=0.4579, WER=0.6833, exact=1/8
Continuation epoch 001/100 | loss=1.2024
Continuation epoch 010/100 | loss=1.1495
Continuation epoch 020/100 | loss=1.1119
Continuation epoch 030/100 | loss=1.0977
Continuation epoch 040/100 | loss=1.0549
Continuation epoch 050/100 | loss=1.0440
Continuation epoch 060/100 | loss=1.0415
Continuation epoch 070/100 | loss=1.0216
Continuation epoch 080/100 | loss=1.0086
Continuation epoch 090/100 | loss=1.0026
Continuation epoch 100/100 | loss=0.9940

========== BEFORE / AFTER ==========
CER:  0.4579 -> 0.3488
WER:  0.6833 -> 0.4083
Exact matches: 1/8 -> 1/8
Separate checkpoint: /content/PFMS_ABLATION_FINAL/mambavision_pilot/overfit8_continued_latest.pt
Original checkpoints were not overwritten.
Do not report this 8-image test as validation or test performance.


In [27]:

# مرحله تشخیصی امن؛ بدون آموزش و بدون حذف فایل
import os
import torch
from pathlib import Path

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Train samples:", len(train_ds))
print("Validation samples:", len(val_ds))
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Vocabulary size:", len(char2idx))
print("Input image shape:", tuple(train_ds[0][0].shape))
print("Current model:", type(small_model).__name__)
print("Current checkpoint exists:", Path(
    "/content/PFMS_ABLATION_FINAL/mambavision_pilot/"
    "overfit8_continued_latest.pt"
).is_file())

print("\nModel output shape:")
small_model.eval()
with torch.no_grad():
    sample_image = train_ds[0][0].unsqueeze(0).to(device)
    sample_logits = small_model(sample_image)
    print(tuple(sample_logits.shape))
    print("Output finite:", bool(torch.isfinite(sample_logits).all()))
    print("Greedy prediction:", repr(
        decode_ids(ctc_greedy(sample_logits))
    ))
    print("Reference text:", repr(decode_ids(train_ds[0][1].tolist())))


GPU: Tesla T4
Train samples: 20746
Validation samples: 2305
Train batches: 5187
Validation batches: 577
Vocabulary size: 169
Input image shape: (3, 64, 1024)
Current model: MambaVisionCTC
Current checkpoint exists: True

Model output shape:
(1, 128, 169)
Output finite: True
Greedy prediction: 'تاریخ مخابره : ۱۳۹۶/۵/۲۳ - ۱۵:۵۸ ۹۶ میلیون پوند برای خرید فدط\n'
Reference text: 'تاریخ مخابره : ۱۳۹۶/۵/۲۳ - ۱۵:۵۸ ۹۶ میلیون پوند برای خرید فرش مساجد در مصر\n'


In [28]:

from pathlib import Path
import torch

ckpt_path = Path(
    "/content/PFMS_ABLATION_FINAL/mambavision_pilot/"
    "overfit8_continued_latest.pt"
)

ckpt = torch.load(
    ckpt_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint keys:", list(ckpt.keys()))
print("Saved continuation epoch:", ckpt.get("continuation_epoch"))
print("Model state available:", "model_state_dict" in ckpt)
print("Optimizer state available:", "optimizer_state_dict" in ckpt)
print("Model parameters:", sum(p.numel() for p in small_model.parameters()))
print("Trainable parameters:", sum(
    p.numel() for p in small_model.parameters() if p.requires_grad
))


Checkpoint keys: ['continuation_epoch', 'source_checkpoint', 'model_state_dict', 'optimizer_state_dict', 'note']
Saved continuation epoch: 100
Model state available: True
Optimizer state available: True
Model parameters: 32272625
Trainable parameters: 32272625


In [29]:

import torch
from pathlib import Path

base = Path("/content/PFMS_ABLATION_FINAL/mambavision_pilot")
candidates = [
    base / "mambavision_pilot_latest.pt",
    base / "overfit8_latest.pt",
    base / "overfit8_continued_latest.pt",
]

for path in candidates:
    print("\n" + "=" * 65)
    print("File:", path.name)
    print("Exists:", path.is_file())

    if not path.is_file():
        continue

    obj = torch.load(path, map_location="cpu", weights_only=False)
    print("Keys:", list(obj.keys()) if isinstance(obj, dict) else type(obj))

    if isinstance(obj, dict):
        for key in (
            "epoch", "continuation_epoch", "step",
            "train_loss", "val_loss", "val_cer",
            "best_val_cer", "note", "source_checkpoint"
        ):
            if key in obj:
                print(f"{key}: {obj[key]}")

        state = obj.get("model_state_dict")
        if isinstance(state, dict):
            print("Parameter tensors:", len(state))
            print("Total parameters:", sum(
                value.numel() for value in state.values()
                if torch.is_tensor(value)
            ))



File: mambavision_pilot_latest.pt
Exists: True
Keys: ['step', 'model_state_dict', 'optimizer_state_dict', 'train_loss', 'model_name', 'seed', 'vocab_size', 'image_size', 'note']
step: 100
train_loss: 3.0340607166290283
note: Pilot only; not the complete PFMS-Net decoder.
Parameter tensors: 260
Total parameters: 32276380

File: overfit8_latest.pt
Exists: True
Keys: ['step', 'model_state_dict', 'optimizer_state_dict', 'loss', 'note']
step: 100
note: 8-sample overfit diagnostic only
Parameter tensors: 260
Total parameters: 32276380

File: overfit8_continued_latest.pt
Exists: True
Keys: ['continuation_epoch', 'source_checkpoint', 'model_state_dict', 'optimizer_state_dict', 'note']
continuation_epoch: 100
note: 8-sample overfit diagnostic; not final OCR metrics
source_checkpoint: /content/PFMS_ABLATION_FINAL/mambavision_pilot/overfit8_latest.pt
Parameter tensors: 260
Total parameters: 32276380


In [30]:

# ============================================================
# MambaVision-CTC: controlled full-data training
# Starts from pilot weights; fresh optimizer.
# Saves locally; does not overwrite existing checkpoints.
# ============================================================

import os, time, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn

# ---------- Configuration ----------
SEED = 42
LR = 1e-4
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 10
PATIENCE = 3
PRINT_EVERY = 200

OUT = Path("/content/PFMS_ABLATION_FINAL/mambavision_fullrun")
OUT.mkdir(parents=True, exist_ok=True)

INIT_CKPT = Path(
    "/content/PFMS_ABLATION_FINAL/mambavision_pilot/"
    "mambavision_pilot_latest.pt"
)
LATEST_CKPT = OUT / "mambavision_ctc_latest.pt"
BEST_CKPT = OUT / "mambavision_ctc_best_val_cer.pt"
LOG_PATH = OUT / "epoch_metrics.csv"

assert INIT_CKPT.is_file(), f"Initial checkpoint missing: {INIT_CKPT}"
assert "model" in globals(), "مدل pilot با نام model در حافظه موجود نیست."
assert "train_loader" in globals() and "val_loader" in globals()
assert "decode_ids" in globals() and "ctc_greedy" in globals()

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Restore weights ONLY. Do not reuse the pilot optimizer.
initial = torch.load(
    INIT_CKPT, map_location="cpu", weights_only=False
)
model.load_state_dict(initial["model_state_dict"], strict=True)
del initial

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)
criterion = nn.CTCLoss(blank=0, zero_infinity=True)

def levenshtein_distance(a, b):
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        current = [i]
        for j, cb in enumerate(b, 1):
            current.append(min(
                current[-1] + 1,
                previous[j] + 1,
                previous[j - 1] + (ca != cb)
            ))
        previous = current
    return previous[-1]

@torch.no_grad()
def evaluate_model():
    model.eval()
    char_errors = char_count = 0
    word_errors = word_count = 0
    exact = samples = 0
    val_loss_sum = 0.0
    batches = 0

    for images, targets, target_lengths in val_loader:
        images = images.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)

        logits = model(images)
        input_lengths = torch.full(
            (images.size(0),), logits.size(1),
            dtype=torch.long, device="cpu"
        )
        log_probs = logits.float().log_softmax(-1).transpose(0, 1)
        loss = criterion(
            log_probs,
            targets,
            input_lengths,
            target_lengths.cpu()
        )
        val_loss_sum += float(loss.detach().cpu())
        batches += 1

        offset = 0
        for i, length in enumerate(target_lengths.tolist()):
            target_ids = targets[offset:offset + length].detach().cpu().tolist()
            offset += length

            pred_text = decode_ids(ctc_greedy(logits[i:i+1]))
            true_text = decode_ids(target_ids)

            char_errors += levenshtein_distance(pred_text, true_text)
            char_count += max(1, len(true_text))

            pred_words = pred_text.split()
            true_words = true_text.split()
            word_errors += levenshtein_distance(pred_words, true_words)
            word_count += max(1, len(true_words))

            exact += int(pred_text == true_text)
            samples += 1

    return {
        "val_loss": val_loss_sum / max(1, batches),
        "CER": char_errors / max(1, char_count),
        "WER": word_errors / max(1, word_count),
        "exact_match": exact / max(1, samples),
        "samples": samples,
    }

# Fresh-run log: do not append accidentally to an older experiment.
with open(LOG_PATH, "w", encoding="utf-8") as f:
    f.write("epoch,train_loss,val_loss,CER,WER,exact_match,seconds,lr\n")

best_cer = float("inf")
bad_epochs = 0

print("Device:", device)
print("Initial checkpoint:", INIT_CKPT)
print("Train samples:", len(train_ds), "| batches:", len(train_loader))
print("Validation samples:", len(val_ds), "| batches:", len(val_loader))
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Output directory:", OUT)
print("Starting full-data training...")

for epoch in range(1, MAX_EPOCHS + 1):
    start = time.time()
    model.train()
    train_loss_sum = 0.0
    train_batches = 0

    for batch_idx, (images, targets, target_lengths) in enumerate(train_loader, 1):
        images = images.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)

        input_lengths = torch.full(
            (images.size(0),), logits.size(1),
            dtype=torch.long, device="cpu"
        )
        log_probs = logits.float().log_softmax(-1).transpose(0, 1)

        loss = criterion(
            log_probs, targets, input_lengths, target_lengths.cpu()
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at epoch {epoch}, batch {batch_idx}. "
                "Stop and inspect the batch; do not report this run."
            )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        train_loss_sum += float(loss.detach().cpu())
        train_batches += 1

        if batch_idx == 1 or batch_idx % PRINT_EVERY == 0:
            print(
                f"Epoch {epoch}/{MAX_EPOCHS} | "
                f"batch {batch_idx}/{len(train_loader)} | "
                f"loss {float(loss.detach().cpu()):.4f}"
            )

    train_loss = train_loss_sum / max(1, train_batches)
    metrics = evaluate_model()
    elapsed = time.time() - start
    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"\nEpoch {epoch}: train_loss={train_loss:.4f}, "
        f"val_loss={metrics['val_loss']:.4f}, "
        f"CER={metrics['CER']:.4f}, WER={metrics['WER']:.4f}, "
        f"exact_match={metrics['exact_match']:.4f}, "
        f"time={elapsed/60:.1f} min"
    )

    record = (
        f"{epoch},{train_loss:.6f},{metrics['val_loss']:.6f},"
        f"{metrics['CER']:.6f},{metrics['WER']:.6f},"
        f"{metrics['exact_match']:.6f},{elapsed:.1f},{current_lr:.8g}\n"
    )
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(record)

    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "validation_metrics": metrics,
        "seed": SEED,
        "learning_rate": current_lr,
        "initial_checkpoint": str(INIT_CKPT),
        "note": "MambaVision-T stage-1 CTC pilot; not full PFMS-Net"
    }

    # Atomic latest checkpoint, separate from all earlier files.
    tmp_latest = LATEST_CKPT.with_suffix(".tmp")
    torch.save(payload, tmp_latest)
    os.replace(tmp_latest, LATEST_CKPT)

    if metrics["CER"] < best_cer:
        best_cer = metrics["CER"]
        bad_epochs = 0
        tmp_best = BEST_CKPT.with_suffix(".tmp")
        torch.save(payload, tmp_best)
        os.replace(tmp_best, BEST_CKPT)
        print("New best validation CER. Best checkpoint saved.")
    else:
        bad_epochs += 1

    print("Latest checkpoint:", LATEST_CKPT)
    print("Best checkpoint:", BEST_CKPT)
    print("Metrics log:", LOG_PATH, "\n")

    if bad_epochs >= PATIENCE:
        print("Early stopping: validation CER did not improve.")
        break

print("\nTraining run finished.")
print("Best validation CER:", best_cer)
print("Best checkpoint:", BEST_CKPT)
print("Metrics CSV:", LOG_PATH)


Device: cuda
Initial checkpoint: /content/PFMS_ABLATION_FINAL/mambavision_pilot/mambavision_pilot_latest.pt
Train samples: 20746 | batches: 5187
Validation samples: 2305 | batches: 577
Parameters: 32272625
Output directory: /content/PFMS_ABLATION_FINAL/mambavision_fullrun
Starting full-data training...
Epoch 1/10 | batch 1/5187 | loss 3.0567
Epoch 1/10 | batch 200/5187 | loss 3.1565
Epoch 1/10 | batch 400/5187 | loss 3.2319
Epoch 1/10 | batch 600/5187 | loss 3.1805
Epoch 1/10 | batch 800/5187 | loss 3.0781
Epoch 1/10 | batch 1000/5187 | loss 3.1150
Epoch 1/10 | batch 1200/5187 | loss 3.1203
Epoch 1/10 | batch 1400/5187 | loss 2.9417
Epoch 1/10 | batch 1600/5187 | loss 3.0766
Epoch 1/10 | batch 1800/5187 | loss 2.8543
Epoch 1/10 | batch 2000/5187 | loss 2.9575
Epoch 1/10 | batch 2200/5187 | loss 2.8566
Epoch 1/10 | batch 2400/5187 | loss 2.9618
Epoch 1/10 | batch 2600/5187 | loss 3.1081
Epoch 1/10 | batch 2800/5187 | loss 3.0084
Epoch 1/10 | batch 3000/5187 | loss 2.8612
Epoch 1/10 | ba

In [31]:

# Diagnose CTC collapse and inspect real validation predictions.
import torch
from pathlib import Path

best_path = Path(
    "/content/PFMS_ABLATION_FINAL/mambavision_fullrun/"
    "mambavision_ctc_best_val_cer.pt"
)
assert best_path.is_file(), f"Missing checkpoint: {best_path}"

saved = torch.load(best_path, map_location="cpu", weights_only=False)
model.load_state_dict(saved["model_state_dict"], strict=True)
model.to(device).eval()

print("Checkpoint epoch:", saved["epoch"])
print("Validation CER:", saved["validation_metrics"]["CER"])
print("Validation WER:", saved["validation_metrics"]["WER"])

with torch.no_grad():
    for i in range(min(10, len(val_ds))):
        image, target = val_ds[i]
        logits = model(image.unsqueeze(0).to(device))
        raw_ids = logits[0].argmax(dim=-1).cpu().tolist()

        # Blank is 0 in the current CTC vocabulary.
        blank_ratio = sum(t == 0 for t in raw_ids) / len(raw_ids)
        nonblank_ids = [t for t in raw_ids if t != 0]
        unique_nonblank = len(set(nonblank_ids))

        pred = decode_ids(ctc_greedy(logits))
        truth = decode_ids(target.tolist())

        print("\n" + "-" * 70)
        print(f"Sample {i+1}")
        print(f"Blank ratio: {blank_ratio:.3f}")
        print(f"Unique nonblank classes: {unique_nonblank}")
        print(f"Raw nonblank positions: {len(nonblank_ids)}")
        print(f"Decoded lengths: pred={len(pred)}, truth={len(truth)}")
        print("TRUE:", repr(truth))
        print("PRED:", repr(pred))


Checkpoint epoch: 1
Validation CER: 0.7700932160959502
Validation WER: 0.9999710932531652

----------------------------------------------------------------------
Sample 1
Blank ratio: 0.859
Unique nonblank classes: 3
Raw nonblank positions: 18
Decoded lengths: pred=14, truth=77
TRUE: 'راه نجات انقلاب است. یک شریعه و فراتی است از عاشورای اباعبدالله الحسین (ع) و\n'
PRED: '   ا ا ا ا ا \n'

----------------------------------------------------------------------
Sample 2
Blank ratio: 0.789
Unique nonblank classes: 3
Raw nonblank positions: 27
Decoded lengths: pred=22, truth=71
TRUE: 'دلیل رفتار غیرورزشی نسبت به تیم داوری به مدت یک جلسه از همراهی تیم خود\n'
PRED: '  اا  ا  ا اا  ا ا ا \n'

----------------------------------------------------------------------
Sample 3
Blank ratio: 0.828
Unique nonblank classes: 3
Raw nonblank positions: 22
Decoded lengths: pred=21, truth=70
TRUE: 'گفت: ملت ایران ملت باتجربه ای هستند که با وجود تمام تحریم ها و مشکلات\n'
PRED: '  ا    اا ا ا   ا  ا\n'

-------

In [32]:

import torch
from collections import Counter

print("========== 1. VOCABULARY ==========")
print("Vocabulary size:", len(char2idx))
print("Model output classes:", model.classifier.out_features)
print("Blank ID:", 0)
print("Characters mapped to IDs 0-10:")
for ch, idx in sorted(char2idx.items(), key=lambda x: x[1]):
    if idx <= 10:
        print(repr(ch), "->", idx)

print("\nSpecial IDs:")
for name, idx in [("PAD", 0), ("BOS", 1), ("EOS", 2), ("UNK", 3)]:
    print(name, "=", idx, "| used by a real character:",
          any(ch_id == idx for ch, ch_id in char2idx.items()))

print("\n========== 2. TARGET IDS ==========")
for i in range(3):
    image, target = train_ds[i]
    ids = target.tolist()
    print(
        f"Train sample {i+1}: target_len={len(ids)}, "
        f"min_id={min(ids) if ids else None}, "
        f"max_id={max(ids) if ids else None}, "
        f"contains_blank_id_0={0 in ids}, "
        f"contains_BOS_1={1 in ids}, "
        f"contains_EOS_2={2 in ids}"
    )
    print("Text:", repr(decode_ids(ids)))

print("\n========== 3. TARGET DISTRIBUTION ==========")
target_counter = Counter()
for i in range(min(500, len(train_ds))):
    _, target = train_ds[i]
    target_counter.update(target.tolist())

print("Most frequent target IDs:", target_counter.most_common(15))
print("Unique target IDs:", len(target_counter))
print("Target tokens total:", sum(target_counter.values()))

print("\n========== 4. MODEL PREDICTION CLASSES ==========")
model.eval()
with torch.no_grad():
    for i in range(3):
        image, target = val_ds[i]
        logits = model(image.unsqueeze(0).to(device))[0]
        pred_ids = logits.argmax(dim=-1).cpu().tolist()
        counts = Counter(pred_ids)
        print(f"\nValidation sample {i+1}")
        print("Most frequent raw predicted IDs:", counts.most_common(10))
        print("Raw unique classes:", len(counts))
        print("Blank ratio:", round(counts.get(0, 0) / len(pred_ids), 4))
        print("Predicted decoded text:",
              repr(decode_ids(ctc_greedy(logits.unsqueeze(0)))))

print("\n========== 5. CTC INPUT/TARGET LENGTH CHECK ==========")
bad = []
for i in range(min(1000, len(train_ds))):
    _, target = train_ds[i]
    ids = target.tolist()
    min_steps = len(ids) + sum(
        ids[j] == ids[j-1] for j in range(1, len(ids))
    )
    if min_steps > 128 or 0 in ids:
        bad.append((i, len(ids), min_steps, 0 in ids))

print("Checked samples:", min(1000, len(train_ds)))
print("Invalid samples (index, target_len, min_ctc_steps, has_blank):",
      bad[:20])
print("Invalid count:", len(bad))


========== 1. VOCABULARY ==========
Vocabulary size: 169
Model output classes: 169
Blank ID: 0
Characters mapped to IDs 0-10:
'<PAD>' -> 0
'<BOS>' -> 1
'<EOS>' -> 2
'<UNK>' -> 3
'\n' -> 4
' ' -> 5
'!' -> 6
'%' -> 7
'(' -> 8
')' -> 9
'*' -> 10

Special IDs:
PAD = 0 | used by a real character: True
BOS = 1 | used by a real character: True
EOS = 2 | used by a real character: True
UNK = 3 | used by a real character: True

========== 2. TARGET IDS ==========
Train sample 1: target_len=75, min_id=4, max_id=100, contains_blank_id_0=False, contains_BOS_1=False, contains_EOS_2=False
Text: 'تاریخ مخابره : ۱۳۹۶/۵/۲۳ - ۱۵:۵۸ ۹۶ میلیون پوند برای خرید فرش مساجد در مصر\n'
Train sample 2: target_len=85, min_id=4, max_id=102, contains_blank_id_0=False, contains_BOS_1=False, contains_EOS_2=False
Text: 'که برای اولین\u200cبار فیلم خبری ساخت. معتضدی اولین لابراتوار ظهور فیلم را نیز در زیرزمین\n'
Train sample 3: target_len=77, min_id=4, max_id=90, contains_blank_id_0=False, contains_BOS_1=False, contains_E

In [33]:

# Read-only vocabulary integrity audit
assert len(char2idx) == 169
assert len(idx2char) == 169

print("char2idx type:", type(char2idx).__name__)
print("idx2char type:", type(idx2char).__name__)

# Check both directions of the mapping.
forward_errors = []
for ch, idx in char2idx.items():
    if idx2char.get(idx) != ch:
        forward_errors.append((repr(ch), idx, repr(idx2char.get(idx))))

reverse_errors = []
for idx, ch in idx2char.items():
    if char2idx.get(ch) != idx:
        reverse_errors.append((idx, repr(ch), char2idx.get(ch)))

print("Forward mapping errors:", len(forward_errors))
print("Reverse mapping errors:", len(reverse_errors))
print("First forward errors:", forward_errors[:10])
print("First reverse errors:", reverse_errors[:10])

print("\nSpecial token mapping:")
for token in ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]:
    print(repr(token), "->", char2idx.get(token))

print("\nSpecial tokens in idx2char:")
for idx in range(4):
    print(idx, "->", repr(idx2char.get(idx)))

print("\nCheck whether any actual dataset character is a special token:")
specials = {"<PAD>", "<BOS>", "<EOS>", "<UNK>"}
actual_chars = set()
for i in range(min(1000, len(train_ds))):
    _, target = train_ds[i]
    actual_chars.update(decode_ids(target.tolist()))

print("This last check uses decoded text; special tokens present as literal strings:",
      sorted(actual_chars.intersection(specials)))


char2idx type: dict
idx2char type: dict
Forward mapping errors: 0
Reverse mapping errors: 0
First forward errors: []
First reverse errors: []

Special token mapping:
'<PAD>' -> 0
'<BOS>' -> 1
'<EOS>' -> 2
'<UNK>' -> 3

Special tokens in idx2char:
0 -> '<PAD>'
1 -> '<BOS>'
2 -> '<EOS>'
3 -> '<UNK>'

Check whether any actual dataset character is a special token:
This last check uses decoded text; special tokens present as literal strings: []


In [34]:

import torch
from collections import Counter
from pathlib import Path

pilot_path = Path(
    "/content/PFMS_ABLATION_FINAL/mambavision_pilot/"
    "mambavision_pilot_latest.pt"
)
best_path = Path(
    "/content/PFMS_ABLATION_FINAL/mambavision_fullrun/"
    "mambavision_ctc_best_val_cer.pt"
)

pilot = torch.load(pilot_path, map_location="cpu", weights_only=False)
best = torch.load(best_path, map_location="cpu", weights_only=False)

# Take one fixed validation batch.
images, targets, target_lengths = next(iter(val_loader))
images = images[:8].to(device)

def prediction_stats(state_dict, label):
    model.load_state_dict(state_dict, strict=True)
    model.to(device).eval()

    with torch.no_grad():
        logits = model(images)
        probs = logits.float().softmax(dim=-1)
        pred_ids = logits.argmax(dim=-1).cpu()

    all_ids = pred_ids.reshape(-1).tolist()
    counts = Counter(all_ids)

    print(f"\n--- {label} ---")
    print("Output shape:", tuple(logits.shape))
    print("Mean max probability:", round(float(probs.max(-1).values.mean()), 4))
    print("Mean blank probability:", round(float(probs[..., 0].mean()), 4))
    print("Blank argmax ratio:", round(counts.get(0, 0) / len(all_ids), 4))
    print("Most frequent argmax IDs:", counts.most_common(8))
    print("Unique predicted classes:", len(counts))

prediction_stats(pilot["model_state_dict"], "Initial pilot checkpoint")
prediction_stats(best["model_state_dict"], "Best validation checkpoint")

print("\nNo checkpoint was modified or saved.")



--- Initial pilot checkpoint ---
Output shape: (4, 128, 169)
Mean max probability: 0.384
Mean blank probability: 0.384
Blank argmax ratio: 1.0
Most frequent argmax IDs: [(0, 512)]
Unique predicted classes: 1

--- Best validation checkpoint ---
Output shape: (4, 128, 169)
Mean max probability: 0.3825
Mean blank probability: 0.3371
Blank argmax ratio: 0.8203
Most frequent argmax IDs: [(0, 420), (5, 56), (35, 32), (4, 4)]
Unique predicted classes: 4

No checkpoint was modified or saved.


In [35]:

# Check whether pretrained MambaVision-T weights are available.
# This does not modify the current OCR model or its checkpoints.

import torch

print("Testing pretrained MambaVision-T availability...")

try:
    pretrained_encoder = create_mambavision(
        "mamba_vision_T",
        pretrained=True
    )
    pretrained_encoder.eval()

    print("SUCCESS: pretrained MambaVision-T was created.")
    print("Model class:", type(pretrained_encoder).__name__)

    pretrained_params = sum(
        p.numel() for p in pretrained_encoder.parameters()
    )
    print("Encoder parameters:", pretrained_params)

    # Compare a representative parameter with the current randomly
    # initialized encoder to confirm that weights can be inspected.
    current_state = model.encoder.state_dict()
    pretrained_state = pretrained_encoder.state_dict()

    common = [
        k for k in current_state
        if k in pretrained_state
        and current_state[k].shape == pretrained_state[k].shape
        and current_state[k].is_floating_point()
    ]

    if common:
        key = common[0]
        a = current_state[key].detach().cpu().float()
        b = pretrained_state[key].detach().cpu().float()
        print("Compared parameter:", key)
        print("Mean absolute difference:", float((a - b).abs().mean()))
    else:
        print("No matching parameter found for comparison.")

    del pretrained_encoder
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:
    print("Pretrained weights unavailable or loading failed.")
    print("Error type:", type(e).__name__)
    print("Error:", str(e)[:1200])
    print("Current model and checkpoints remain unchanged.")


Testing pretrained MambaVision-T availability...


100%|██████████| 364M/364M [00:03<00:00, 125MB/s]


Pretrained weights unavailable or loading failed.
Error type: UnpicklingError
Error: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL argparse.Namespace was not an allowed global by default. Please use `torch.serialization.add_safe_globals([argparse.Namespace])` or the `torch.serialization.safe_globals([argparse.Namespace])` context manager to allowlist this global if you trust this class/function.

Check the documentat

In [36]:

import argparse
import torch

# Allowlist only the metadata class named in the loading error.
torch.serialization.add_safe_globals([argparse.Namespace])

print("Retrying pretrained MambaVision-T with restricted safe loading...")

pretrained_encoder = create_mambavision(
    "mamba_vision_T",
    pretrained=True
)

pretrained_encoder.eval()

print("SUCCESS: pretrained MambaVision-T loaded.")
print("Model class:", type(pretrained_encoder).__name__)
print(
    "Parameter count:",
    sum(p.numel() for p in pretrained_encoder.parameters())
)

# Confirm that the model has finite parameters.
finite_ok = all(
    torch.isfinite(p.detach()).all().item()
    for p in pretrained_encoder.parameters()
)
print("All parameters finite:", finite_ok)

del pretrained_encoder
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Retrying pretrained MambaVision-T with restricted safe loading...
SUCCESS: pretrained MambaVision-T loaded.
Model class: MambaVision
Parameter count: 31794248
All parameters finite: True


In [37]:

import torch

pretrained_encoder = create_mambavision(
    "mamba_vision_T",
    pretrained=True
).to(device).eval()

images, targets, target_lengths = next(iter(val_loader))
x = images[:2].to(device)

with torch.no_grad():
    print("Input:", tuple(x.shape))

    z = pretrained_encoder.patch_embed(x)
    print("Patch embedding:", tuple(z.shape))

    for i, stage in enumerate(pretrained_encoder.levels):
        z = stage(z)
        print(f"Stage {i+1}:", tuple(z.shape))

        if i == 0:
            # Expected OCR sequence: preserve horizontal width.
            # Feature map is expected in [B, C, H, W] format.
            sequence = z.mean(dim=2).transpose(1, 2)
            print("Stage-1 horizontal sequence:", tuple(sequence.shape))

print("\nEncoder shape inspection completed.")
del pretrained_encoder
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Input: (2, 3, 64, 1024)
Patch embedding: (2, 80, 16, 256)
Stage 1: (2, 160, 8, 128)
Stage-1 horizontal sequence: (2, 128, 160)
Stage 2: (2, 320, 4, 64)
Stage 3: (2, 640, 2, 32)
Stage 4: (2, 640, 2, 32)

Encoder shape inspection completed.


In [38]:

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from itertools import cycle

# 1. New experimental model; the previous model remains untouched.
class PretrainedMambaVisionCTC(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.encoder = create_mambavision(
            "mamba_vision_T", pretrained=True
        )
        self.projection = nn.Conv1d(160, 256, kernel_size=1)
        self.refine = nn.Sequential(
            nn.Conv1d(256, 256, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Conv1d(256, 256, kernel_size=3, padding=1),
            nn.GELU(),
        )
        self.classifier = nn.Linear(256, vocab_size)

    def forward(self, x):
        x = self.encoder.patch_embed(x)
        x = self.encoder.levels[0](x)  # [B,160,8,128]
        x = x.mean(dim=2)              # [B,160,128]
        x = self.projection(x)
        x = x + self.refine(x)
        x = x.transpose(1, 2)          # [B,128,256]
        return self.classifier(x)      # [B,128,vocab_size]


torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

vocab_size = len(char2idx)
trial_model = PretrainedMambaVisionCTC(vocab_size).to(device)

# Freeze the encoder for this short pipeline diagnostic.
for p in trial_model.encoder.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(
    list(trial_model.projection.parameters())
    + list(trial_model.refine.parameters())
    + list(trial_model.classifier.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)
ctc_loss_fn = nn.CTCLoss(blank=0, zero_infinity=True)

fixed_ds = Subset(train_ds, range(8))
fixed_loader = DataLoader(
    fixed_ds,
    batch_size=4,
    shuffle=False,
    collate_fn=train_loader.collate_fn
)
batches = cycle(fixed_loader)

def one_batch_loss(batch):
    images, targets, target_lengths = batch
    images = images.to(device)
    targets = targets.to(device)
    target_lengths = torch.as_tensor(
        target_lengths, dtype=torch.long
    ).cpu()

    logits = trial_model(images)
    log_probs = logits.float().log_softmax(-1).transpose(0, 1)
    input_lengths = torch.full(
        (images.size(0),), logits.size(1), dtype=torch.long
    )

    loss = ctc_loss_fn(
        log_probs, targets, input_lengths, target_lengths
    )
    return loss, logits

trial_model.train()
trial_model.encoder.eval()

print("Starting 100-step fixed-8-sample diagnostic...")
loss_history = []

for step in range(101):
    batch = next(batches)
    optimizer.zero_grad(set_to_none=True)
    loss, logits = one_batch_loss(batch)

    if not torch.isfinite(loss):
        raise RuntimeError(
            f"Non-finite CTC loss at step {step}: {loss.item()}"
        )

    if step % 20 == 0:
        with torch.no_grad():
            blank_ratio = (
                logits.argmax(-1) == 0
            ).float().mean().item()
        loss_history.append((step, float(loss.item()), blank_ratio))
        print(
            f"Step {step:3d} | loss={loss.item():.4f} "
            f"| blank_argmax_ratio={blank_ratio:.3f}"
        )

    if step < 100:
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in trial_model.parameters() if p.requires_grad],
            max_norm=1.0
        )
        optimizer.step()

print("\nDiagnostic summary:")
print("First recorded loss:", round(loss_history[0][1], 4))
print("Last recorded loss:", round(loss_history[-1][1], 4))
print("This was a diagnostic only; no checkpoint was saved.")


Starting 100-step fixed-8-sample diagnostic...
Step   0 | loss=6.9748 | blank_argmax_ratio=0.000
Step  20 | loss=3.8536 | blank_argmax_ratio=1.000
Step  40 | loss=3.1260 | blank_argmax_ratio=1.000
Step  60 | loss=3.0879 | blank_argmax_ratio=0.992
Step  80 | loss=3.0050 | blank_argmax_ratio=0.992
Step 100 | loss=2.9555 | blank_argmax_ratio=0.984

Diagnostic summary:
First recorded loss: 6.9748
Last recorded loss: 2.9555
This was a diagnostic only; no checkpoint was saved.


In [39]:

# Read-only inspection of the current 8-sample diagnostic model.
trial_model.eval()

batch = next(iter(fixed_loader))
images, targets, target_lengths = batch
images = images.to(device)

with torch.no_grad():
    logits = trial_model(images)
    pred_ids = logits.argmax(dim=-1).cpu()

def greedy_ctc_decode(ids, blank_id=0):
    result = []
    previous = None
    for token_id in ids:
        token_id = int(token_id)
        if token_id != previous and token_id != blank_id:
            result.append(token_id)
        previous = token_id
    return result

def decode_target(ids):
    chars = []
    for token_id in ids:
        token_id = int(token_id)
        if token_id in (0, 1, 2):
            continue
        chars.append(idx2char.get(token_id, "�"))
    return "".join(chars)

for i in range(min(4, len(pred_ids))):
    target_len = int(target_lengths[i])
    if isinstance(targets, torch.Tensor) and targets.ndim == 2:
        target_ids = targets[i][:target_len].tolist()
    else:
        # Handle concatenated CTC targets if the collate function uses them.
        start = sum(int(x) for x in target_lengths[:i])
        target_ids = targets[start:start + target_len].tolist()

    pred_tokens = greedy_ctc_decode(pred_ids[i].tolist())
    print(f"\n--- Sample {i+1} ---")
    print("REFERENCE:", repr(decode_target(target_ids)))
    print("PREDICTED:", repr(decode_target(pred_tokens)))
    print("Raw argmax IDs:", pred_ids[i][:40].tolist())



--- Sample 1 ---
REFERENCE: 'تاریخ مخابره : ۱۳۹۶/۵/۲۳ - ۱۵:۵۸ ۹۶ میلیون پوند برای خرید فرش مساجد در مصر\n'
PREDICTED: 'م\n'
Raw argmax IDs: [60, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

--- Sample 2 ---
REFERENCE: 'که برای اولین\u200cبار فیلم خبری ساخت. معتضدی اولین لابراتوار ظهور فیلم را نیز در زیرزمین\n'
PREDICTED: 'م\n'
Raw argmax IDs: [60, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

--- Sample 3 ---
REFERENCE: 'عدم رعایت موارد بهداشتی و ذبح در خارج از کشتارگاه توسط دامپزشکی قائم شهر ضبط\n'
PREDICTED: 'م\n'
Raw argmax IDs: [60, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

--- Sample 4 ---
REFERENCE: 'بزرگتری از انرژی خورشید را به فضا بازتاب می كنند در نتیجه مقدار تابش خورشیدی\n'
PREDICTED: 'م\n'
Raw argmax IDs: [60, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [40]:

import inspect
import torch

batch = next(iter(fixed_loader))
images, targets, target_lengths = batch

print("========== BATCH CONTRACT ==========")
print("Images:", tuple(images.shape), images.dtype)
print("Targets shape:", tuple(targets.shape))
print("Targets dtype:", targets.dtype)
print("Target lengths:", target_lengths)
print("Target lengths type:", type(target_lengths).__name__)

lengths = torch.as_tensor(target_lengths, dtype=torch.long)
print("Number of images:", images.shape[0])
print("Number of target lengths:", len(lengths))
print("Sum of target lengths:", int(lengths.sum()))

if targets.ndim == 1:
    print("Target format: concatenated 1D")
    print("Target count:", targets.numel())
    print("Lengths sum equals target count:", int(lengths.sum()) == targets.numel())
elif targets.ndim == 2:
    print("Target format: padded 2D")
    print("Padded width:", targets.shape[1])
    print(
        "Every length fits padded width:",
        bool((lengths <= targets.shape[1]).all())
    )
    print("First target rows:")
    print(targets[:min(4, len(targets))])
else:
    print("WARNING: unexpected target dimensionality")

print("\n========== TARGET ID VALIDATION ==========")
print("Minimum target ID:", int(targets.min()))
print("Maximum target ID:", int(targets.max()))
print("Blank ID 0 occurs in target tensor:", bool((targets == 0).any()))

print("\n========== COLLATE FUNCTION ==========")
collate_fn = train_loader.collate_fn
print("Function:", collate_fn)

try:
    print(inspect.getsource(collate_fn))
except Exception as e:
    print("Source unavailable:", repr(e))

print("\n========== DATASET ITEM STRUCTURE ==========")
sample = train_ds[0]
print("Item type:", type(sample).__name__)

if isinstance(sample, dict):
    print("Keys:", list(sample.keys()))
    for key, value in sample.items():
        if isinstance(value, torch.Tensor):
            print(key, "tensor shape:", tuple(value.shape), "dtype:", value.dtype)
        else:
            print(key, "type:", type(value).__name__, "value:", repr(value)[:250])
elif isinstance(sample, (tuple, list)):
    print("Number of fields:", len(sample))
    for i, value in enumerate(sample):
        if isinstance(value, torch.Tensor):
            print(i, "tensor shape:", tuple(value.shape), "dtype:", value.dtype)
        else:
            print(i, "type:", type(value).__name__, "value:", repr(value)[:250])


========== BATCH CONTRACT ==========
Images: (4, 3, 64, 1024) torch.float32
Targets shape: (314,)
Targets dtype: torch.int64
Target lengths: tensor([75, 85, 77, 77])
Target lengths type: Tensor
Number of images: 4
Number of target lengths: 4
Sum of target lengths: 314
Target format: concatenated 1D
Target count: 314
Lengths sum equals target count: True

========== TARGET ID VALIDATION ==========
Minimum target ID: 4
Maximum target ID: 102
Blank ID 0 occurs in target tensor: False

========== COLLATE FUNCTION ==========
Function: <function collate_ctc at 0x7e3726b600e0>
def collate_ctc(batch):
    images, targets = zip(*batch)
    images = torch.stack(images)
    lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    flat_targets = torch.cat(targets) if sum(lengths).item() else torch.empty(0, dtype=torch.long)
    return images, flat_targets, lengths


========== DATASET ITEM STRUCTURE ==========
Item type: tuple
Number of fields: 2
0 tensor shape: (3, 64, 1024) dtype:

In [41]:

# Controlled overfit diagnostic: fine-tune Stage 1 + CTC head.
# No checkpoint is saved; original files remain untouched.

import torch

trial_model.train()
trial_model.encoder.eval()

# Unfreeze only the encoder stage used by this OCR model.
for p in trial_model.encoder.levels[0].parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW(
    [
        {"params": trial_model.encoder.levels[0].parameters(), "lr": 1e-5},
        {"params": trial_model.projection.parameters(), "lr": 3e-4},
        {"params": trial_model.refine.parameters(), "lr": 3e-4},
        {"params": trial_model.classifier.parameters(), "lr": 3e-4},
    ],
    weight_decay=1e-4
)

trial_model.encoder.patch_embed.eval()
trial_model.encoder.levels[0].train()

batches = cycle(fixed_loader)
print("Fine-tuning Stage 1 and CTC head on the same 8 samples...")

for step in range(301):
    images, targets, target_lengths = next(batches)
    images = images.to(device)
    targets = targets.to(device)
    target_lengths = torch.as_tensor(
        target_lengths, dtype=torch.long
    ).cpu()

    optimizer.zero_grad(set_to_none=True)
    logits = trial_model(images)
    log_probs = logits.float().log_softmax(-1).transpose(0, 1)
    input_lengths = torch.full(
        (images.size(0),), logits.size(1), dtype=torch.long
    )

    loss = ctc_loss_fn(
        log_probs, targets, input_lengths, target_lengths
    )

    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite loss at step {step}")

    if step % 50 == 0:
        with torch.no_grad():
            blank_ratio = (
                logits.argmax(-1) == 0
            ).float().mean().item()
        print(
            f"Step {step:3d} | loss={loss.item():.4f} "
            f"| blank ratio={blank_ratio:.3f}"
        )

    if step < 300:
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in trial_model.parameters() if p.requires_grad],
            1.0
        )
        optimizer.step()

# Inspect final decoded predictions.
trial_model.eval()
images, targets, target_lengths = next(iter(fixed_loader))
with torch.no_grad():
    logits = trial_model(images.to(device))
    predictions = logits.argmax(-1).cpu()

print("\nFinal predictions:")
offset = 0
for i, length in enumerate(target_lengths.tolist()):
    true_ids = targets[offset:offset + length].tolist()
    offset += length
    pred_ids = greedy_ctc_decode(predictions[i].tolist())

    print(f"\nSample {i+1}")
    print("TRUE:", repr(decode_target(true_ids)))
    print("PRED:", repr(decode_target(pred_ids)))

print("\nDiagnostic finished; no checkpoint saved.")


Fine-tuning Stage 1 and CTC head on the same 8 samples...
Step   0 | loss=2.9862 | blank ratio=0.986
Step  50 | loss=2.8946 | blank ratio=0.984
Step 100 | loss=2.7899 | blank ratio=0.963
Step 150 | loss=2.6747 | blank ratio=0.863
Step 200 | loss=2.4926 | blank ratio=0.730
Step 250 | loss=2.2465 | blank ratio=0.648
Step 300 | loss=1.9766 | blank ratio=0.598

Final predictions:

Sample 1
TRUE: 'تاریخ مخابره : ۱۳۹۶/۵/۲۳ - ۱۵:۵۸ ۹۶ میلیون پوند برای خرید فرش مساجد در مصر\n'
PRED: 'تاری وره : ۱۳۹/۵۲۳ - ۱۵:۵۸ ۹- ب ن ای رید ی\n'

Sample 2
TRUE: 'که برای اولین\u200cبار فیلم خبری ساخت. معتضدی اولین لابراتوار ظهور فیلم را نیز در زیرزمین\n'
PRED: 'که بایا ا  ا ی\n'

Sample 3
TRUE: 'عدم رعایت موارد بهداشتی و ذبح در خارج از کشتارگاه توسط دامپزشکی قائم شهر ضبط\n'
PRED: 'عد ا مارد بهداتی - ت ر ا ارا  ی\n'

Sample 4
TRUE: 'بزرگتری از انرژی خورشید را به فضا بازتاب می كنند در نتیجه مقدار تابش خورشیدی\n'
PRED: 'مقرا  را و را  ا باا ی\n'

Diagnostic finished; no checkpoint saved.


In [42]:

import os, csv, time, torch
import torch.nn as nn
from pathlib import Path

# Uses the existing train_loader, val_loader, device,
# create_mambavision, char2idx and idx2char.
OUT = Path("/content/PFMS_ABLATION_FINAL/mambavision_pretrained_full")
OUT.mkdir(parents=True, exist_ok=True)

class MambaVisionCTCFinal(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.encoder = create_mambavision(
            "mamba_vision_T", pretrained=True
        )
        self.projection = nn.Conv1d(160, 256, kernel_size=1)
        self.refine = nn.Sequential(
            nn.Conv1d(256, 256, 3, padding=1),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Conv1d(256, 256, 3, padding=1),
            nn.GELU(),
        )
        self.classifier = nn.Linear(256, vocab_size)

    def forward(self, x):
        x = self.encoder.patch_embed(x)
        x = self.encoder.levels[0](x)  # B,C,H,W
        x = x.mean(dim=2)              # B,C,W
        x = self.projection(x)
        x = x + self.refine(x)
        return self.classifier(x.transpose(1, 2))

def ctc_decode(ids):
    out, previous = [], None
    for idx in ids:
        idx = int(idx)
        if idx != previous and idx != 0:
            if idx not in (1, 2, 3):
                out.append(idx2char.get(idx, "�"))
        previous = idx
    return "".join(out)

def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(
                cur[-1] + 1,
                prev[j] + 1,
                prev[j-1] + (ca != cb)
            ))
        prev = cur
    return prev[-1]

def decode_reference(ids):
    return "".join(
        idx2char.get(int(i), "�")
        for i in ids
        if int(i) not in (0, 1, 2, 3)
    )

def evaluate(model, loader):
    model.eval()
    char_errors = char_total = 0
    word_errors = word_total = 0
    exact = samples = 0

    with torch.no_grad():
        for images, targets, target_lengths in loader:
            images = images.to(device)
            logits = model(images)
            pred = logits.argmax(-1).cpu()

            lengths = [int(x) for x in target_lengths]
            offset = 0

            for i, length in enumerate(lengths):
                true_ids = targets[offset:offset + length].tolist()
                offset += length

                truth = decode_reference(true_ids)
                guess = ctc_decode(pred[i].tolist())

                char_errors += edit_distance(guess, truth)
                char_total += max(len(truth), 1)

                tw, pw = truth.split(), guess.split()
                word_errors += edit_distance(pw, tw)
                word_total += max(len(tw), 1)

                exact += int(guess == truth)
                samples += 1

    return {
        "CER": char_errors / max(char_total, 1),
        "WER": word_errors / max(word_total, 1),
        "ExactMatch": exact / max(samples, 1),
        "Samples": samples,
    }

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

model = MambaVisionCTCFinal(len(char2idx)).to(device)

# Train the used stage and OCR head; unused stages remain frozen.
for p in model.encoder.parameters():
    p.requires_grad = False
for p in model.encoder.levels[0].parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW([
    {"params": model.encoder.levels[0].parameters(), "lr": 1e-5},
    {"params": model.projection.parameters(), "lr": 3e-4},
    {"params": model.refine.parameters(), "lr": 3e-4},
    {"params": model.classifier.parameters(), "lr": 3e-4},
], weight_decay=1e-4)

criterion = nn.CTCLoss(blank=0, zero_infinity=True)
epochs = 10
best_cer = float("inf")
history = []

for epoch in range(1, epochs + 1):
    start = time.time()
    model.train()
    model.encoder.eval()
    model.encoder.patch_embed.eval()
    model.encoder.levels[0].train()

    loss_sum = 0.0
    steps = 0

    for images, targets, target_lengths in train_loader:
        images = images.to(device)
        targets = targets.to(device)
        target_lengths = torch.as_tensor(
            target_lengths, dtype=torch.long
        ).cpu()

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        log_probs = logits.float().log_softmax(-1).transpose(0, 1)
        input_lengths = torch.full(
            (images.size(0),), logits.size(1), dtype=torch.long
        )

        loss = criterion(
            log_probs, targets, input_lengths, target_lengths
        )
        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at epoch {epoch}, step {steps}"
            )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], 1.0
        )
        optimizer.step()

        loss_sum += float(loss.detach())
        steps += 1

    train_loss = loss_sum / max(steps, 1)
    metrics = evaluate(model, val_loader)
    elapsed = time.time() - start

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        **metrics,
        "seconds": elapsed,
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d}/{epochs} | "
        f"loss={train_loss:.4f} | "
        f"val_CER={metrics['CER']:.4f} | "
        f"val_WER={metrics['WER']:.4f} | "
        f"val_exact={metrics['ExactMatch']:.4f} | "
        f"time={elapsed/60:.1f} min"
    )

    if metrics["CER"] < best_cer:
        best_cer = metrics["CER"]
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            **metrics,
            "vocab_size": len(char2idx),
            "blank_id": 0,
            "architecture": "MambaVision-T stage1 + CTC",
            "seed": 42,
        }, OUT / "best_checkpoint.pt")
        print("  Saved new best validation checkpoint.")

    with open(OUT / "epoch_metrics.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
        writer.writeheader()
        writer.writerows(history)

print("\nTraining finished.")
print("Best validation CER:", round(best_cer, 6))
print("Artifacts:", OUT)
print("This is MambaVision-only, not the complete PFMS-Net.")


Epoch 01/10 | loss=3.0096 | val_CER=0.7900 | val_WER=1.0000 | val_exact=0.0000 | time=2.5 min
  Saved new best validation checkpoint.
Epoch 02/10 | loss=2.9338 | val_CER=0.7663 | val_WER=0.9999 | val_exact=0.0000 | time=2.5 min
  Saved new best validation checkpoint.
Epoch 03/10 | loss=2.9251 | val_CER=0.7777 | val_WER=0.9998 | val_exact=0.0000 | time=2.5 min
Epoch 04/10 | loss=2.9199 | val_CER=0.7564 | val_WER=0.9993 | val_exact=0.0000 | time=2.5 min
  Saved new best validation checkpoint.
Epoch 05/10 | loss=2.9160 | val_CER=0.7835 | val_WER=0.9999 | val_exact=0.0000 | time=2.5 min
Epoch 06/10 | loss=2.9128 | val_CER=0.7751 | val_WER=0.9999 | val_exact=0.0000 | time=2.5 min
Epoch 07/10 | loss=2.9092 | val_CER=0.7811 | val_WER=0.9999 | val_exact=0.0000 | time=2.5 min
Epoch 08/10 | loss=2.9068 | val_CER=0.7677 | val_WER=0.9998 | val_exact=0.0000 | time=2.5 min
Epoch 09/10 | loss=2.9041 | val_CER=0.7752 | val_WER=0.9997 | val_exact=0.0000 | time=2.5 min
Epoch 10/10 | loss=2.9014 | val_CE

In [43]:

# بررسی بهترین checkpoint مدل MambaVision-only
# فقط خواندن checkpoint و ارزیابی نمونه‌ها؛ بدون آموزش یا حذف فایل

import os
import torch
import torch.nn as nn

CKPT_PATH = "/content/PFMS_ABLATION_FINAL/mambavision_pretrained_full/best_checkpoint.pt"
assert os.path.isfile(CKPT_PATH), f"Checkpoint پیدا نشد: {CKPT_PATH}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Checkpoint:", CKPT_PATH)

# اجازهٔ بارگذاری checkpoint ذخیره‌شده توسط همین اجرای پژوهشی
try:
    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
except TypeError:
    ckpt = torch.load(CKPT_PATH, map_location=device)

if isinstance(ckpt, dict):
    print("Checkpoint keys:", list(ckpt.keys())[:25])
else:
    print("Checkpoint type:", type(ckpt))

# پیدا کردن state_dict در قالب‌های رایج
state = None
if isinstance(ckpt, dict):
    for key in ("model_state_dict", "state_dict", "model", "model_state"):
        value = ckpt.get(key)
        if isinstance(value, dict) and value and all(
            torch.is_tensor(v) for v in value.values()
        ):
            state = value
            print("Weights found under:", key)
            break
    if state is None and ckpt and all(
        torch.is_tensor(v) for v in ckpt.values()
    ):
        state = ckpt
        print("Checkpoint itself is the state_dict")

assert state is not None, (
    "ساختار وزن‌ها به‌صورت خودکار شناسایی نشد؛ خروجی Checkpoint keys را بفرست."
)

# استفاده از مدل موجود در حافظه؛ در صورت نبود، ساخت نمونهٔ مدل
if "model" in globals() and isinstance(model, MambaVisionCTCFinal):
    net = model
else:
    assert "MambaVisionCTCFinal" in globals(), (
        "تعریف کلاس MambaVisionCTCFinal در حافظه نیست؛ "
        "بدون اجرای دوبارهٔ آموزش، کلاس را در همین runtime تعریف کن."
    )
    net = MambaVisionCTCFinal(vocab_size=len(char2idx))

# بارگذاری امن و بررسی سازگاری وزن‌ها
load_result = net.load_state_dict(state, strict=False)
print("Missing keys:", load_result.missing_keys[:10])
print("Unexpected keys:", load_result.unexpected_keys[:10])

net = net.to(device)
net.eval()

def _decode_pred(ids):
    out, prev = [], None
    for token in ids:
        token = int(token)
        if token != prev and token != 0 and token not in (1, 2, 3):
            out.append(idx2char.get(token, "�"))
        prev = token
    return "".join(out)

def _decode_ref(ids):
    return "".join(
        idx2char.get(int(t), "�")
        for t in ids
        if int(t) not in (0, 1, 2, 3)
    )

blank_count = 0
total_count = 0
nonblank_unique = set()
shown = 0

with torch.no_grad():
    for batch in val_loader:
        images, flat_targets, target_lengths = batch
        images = images.to(device)
        logits = net(images)                 # (B, T, V)
        pred_ids = logits.argmax(dim=-1).cpu()
        flat_targets = flat_targets.cpu()
        target_lengths = target_lengths.cpu()

        offset = 0
        for i in range(len(images)):
            n = int(target_lengths[i])
            truth_ids = flat_targets[offset:offset+n].tolist()
            offset += n

            ids = pred_ids[i].tolist()
            blank_count += sum(int(t) == 0 for t in ids)
            total_count += len(ids)
            nonblank_unique.update(int(t) for t in ids if int(t) != 0)

            if shown < 12:
                print(f"\n--- نمونه {shown+1} ---")
                print("واقعی:", _decode_ref(truth_ids)[:250])
                print("پیش‌بینی:", _decode_pred(ids)[:250])
                print("تعداد نویسه‌های پیش‌بینی‌شده:", len(_decode_pred(ids)))
                shown += 1

        if shown >= 12:
            break

print("\n========== تشخیص نهایی ==========")
print(f"Blank ratio: {blank_count / max(total_count, 1):.4%}")
print("تعداد شناسه‌های غیرblank پیش‌بینی‌شده:", len(nonblank_unique))
print("شناسه‌های غیرblank نمونه:", sorted(nonblank_unique)[:60])


Device: cuda
Checkpoint: /content/PFMS_ABLATION_FINAL/mambavision_pretrained_full/best_checkpoint.pt
Checkpoint keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'train_loss', 'CER', 'WER', 'ExactMatch', 'Samples', 'vocab_size', 'blank_id', 'architecture', 'seed']
Weights found under: model_state_dict
Missing keys: []
Unexpected keys: []

--- نمونه 1 ---
واقعی: راه نجات انقلاب است. یک شریعه و فراتی است از عاشورای اباعبدالله الحسین (ع) و

پیش‌بینی: ا ی ا اا  ا ا ه 

تعداد نویسه‌های پیش‌بینی‌شده: 17

--- نمونه 2 ---
واقعی: دلیل رفتار غیرورزشی نسبت به تیم داوری به مدت یک جلسه از همراهی تیم خود

پیش‌بینی: ا  ا ا ا   ا ا با

تعداد نویسه‌های پیش‌بینی‌شده: 18

--- نمونه 3 ---
واقعی: گفت: ملت ایران ملت باتجربه ای هستند که با وجود تمام تحریم ها و مشکلات

پیش‌بینی:    ا   ا  ا ا ا ا ا  ا

تعداد نویسه‌های پیش‌بینی‌شده: 23

--- نمونه 4 ---
واقعی: پذیرند منابعی مانند جنگل ها یا تنوع زیستی حیوانی... جنگل ها و شیلات می توان

پیش‌بینی:  ا ا ا  اا  ا ا ا ا ا  ا ا ا

تعداد نویسه‌های پیش‌بینی‌ش

In [44]:

# MambaVision-only: fine-tuning اصلاح‌شده
# checkpoint قبلی حفظ می‌شود؛ خروجی‌ها در مسیر جدید ذخیره می‌شوند.

import os, time, csv, copy
import torch
import torch.nn as nn

assert "net" in globals(), "مدل بارگذاری‌شده (net) در حافظه نیست."
assert "train_loader" in globals() and "val_loader" in globals()
assert "idx2char" in globals()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = net.to(device)

OUT_DIR = "/content/PFMS_ABLATION_FINAL/mambavision_unfrozen_patch_embed"
os.makedirs(OUT_DIR, exist_ok=True)
BEST_PATH = os.path.join(OUT_DIR, "best_checkpoint.pt")
CSV_PATH = os.path.join(OUT_DIR, "epoch_metrics.csv")

# آزادسازی patch embedding و Stage 1؛ سایر مراحل استفاده‌نشده ثابت می‌مانند.
for p in net.encoder.parameters():
    p.requires_grad = False
for p in net.encoder.patch_embed.parameters():
    p.requires_grad = True
for p in net.encoder.levels[0].parameters():
    p.requires_grad = True

for module in (net.projection, net.refine, net.classifier):
    for p in module.parameters():
        p.requires_grad = True

optimizer = torch.optim.AdamW([
    {"params": net.encoder.patch_embed.parameters(), "lr": 2e-6},
    {"params": net.encoder.levels[0].parameters(), "lr": 1e-5},
    {"params": list(net.projection.parameters()) +
                list(net.refine.parameters()) +
                list(net.classifier.parameters()), "lr": 1e-4},
], weight_decay=1e-4)

criterion = nn.CTCLoss(blank=0, zero_infinity=True)

def _decode_ids_ctc(ids):
    chars, prev = [], None
    for token in ids:
        token = int(token)
        if token != prev and token != 0 and token not in (1, 2, 3):
            chars.append(idx2char.get(token, "�"))
        prev = token
    return "".join(chars)

def _decode_target_ctc(ids):
    return "".join(
        idx2char.get(int(t), "�")
        for t in ids if int(t) not in (0, 1, 2, 3)
    )

def _edit_distance(a, b):
    # حافظه O(len(b))
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(
                cur[-1] + 1,
                prev[j] + 1,
                prev[j-1] + (ca != cb)
            ))
        prev = cur
    return prev[-1]

@torch.no_grad()
def evaluate_ctc(model, loader):
    model.eval()
    char_err = word_err = char_total = word_total = exact = count = 0
    blank_n = token_n = 0

    for images, flat_targets, target_lengths in loader:
        images = images.to(device)
        logits = model(images)
        pred_ids = logits.argmax(dim=-1).cpu()
        flat_targets = flat_targets.cpu()
        target_lengths = target_lengths.cpu()
        offset = 0

        for i in range(len(images)):
            n = int(target_lengths[i])
            ref_ids = flat_targets[offset:offset+n].tolist()
            offset += n

            ref = _decode_target_ctc(ref_ids)
            hyp = _decode_ids_ctc(pred_ids[i].tolist())

            char_err += _edit_distance(ref, hyp)
            char_total += max(len(ref), 1)

            ref_words, hyp_words = ref.split(), hyp.split()
            word_err += _edit_distance(ref_words, hyp_words)
            word_total += max(len(ref_words), 1)

            exact += int(ref == hyp)
            count += 1
            blank_n += int((pred_ids[i] == 0).sum())
            token_n += pred_ids.shape[1]

    return {
        "CER": char_err / max(char_total, 1),
        "WER": word_err / max(word_total, 1),
        "ExactMatch": exact / max(count, 1),
        "BlankRatio": blank_n / max(token_n, 1),
        "Samples": count,
    }

best_cer = float("inf")
rows = []

for epoch in range(1, 6):
    start = time.time()
    net.train()
    # رمزگذار را ثابت نگه دار، جز patch_embed و Stage 1.
    net.encoder.eval()
    net.encoder.patch_embed.train()
    net.encoder.levels[0].train()

    total_loss = 0.0
    batches = 0

    for images, targets, target_lengths in train_loader:
        images = images.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = net(images)  # (B, T, V)

        input_lengths = torch.full(
            (logits.size(0),), logits.size(1),
            dtype=torch.long, device="cpu"
        )
        loss = criterion(
            logits.log_softmax(-1).transpose(0, 1),
            targets,
            input_lengths,
            target_lengths.cpu()
        )

        if not torch.isfinite(loss):
            raise RuntimeError(f"CTC loss غیرمعتبر در epoch={epoch}: {loss.item()}")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in net.parameters() if p.requires_grad], 1.0
        )
        optimizer.step()

        total_loss += loss.item()
        batches += 1

    train_loss = total_loss / max(batches, 1)
    metrics = evaluate_ctc(net, val_loader)
    elapsed = (time.time() - start) / 60

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        **metrics,
        "minutes": elapsed,
    }
    rows.append(row)

    print(
        f"Epoch {epoch}/5 | loss={train_loss:.4f} | "
        f"CER={metrics['CER']:.4f} | WER={metrics['WER']:.4f} | "
        f"Exact={metrics['ExactMatch']:.4f} | "
        f"Blank={metrics['BlankRatio']:.2%} | {elapsed:.1f} min"
    )

    # ذخیره فقط در مسیر جدید؛ بهترین مدل بر اساس CER اعتبارسنجی
    if metrics["CER"] < best_cer:
        best_cer = metrics["CER"]
        torch.save({
            "epoch": epoch,
            "model_state_dict": net.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            **metrics,
            "vocab_size": len(idx2char),
            "blank_id": 0,
            "architecture": "MambaVision-T Stage1 CTC; patch_embed and Stage1 fine-tuned",
            "seed": 42,
        }, BEST_PATH)
        print("  بهترین checkpoint جدید ذخیره شد.")

with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print("\nپایان آموزش تکمیلی")
print("Best validation CER:", best_cer)
print("Checkpoint:", BEST_PATH)
print("Metrics:", CSV_PATH)


Epoch 1/5 | loss=2.9177 | CER=0.7814 | WER=0.9997 | Exact=0.0000 | Blank=84.05% | 2.5 min
  بهترین checkpoint جدید ذخیره شد.
Epoch 2/5 | loss=2.9134 | CER=0.7697 | WER=0.9995 | Exact=0.0000 | Blank=82.69% | 2.5 min
  بهترین checkpoint جدید ذخیره شد.
Epoch 3/5 | loss=2.9112 | CER=0.7755 | WER=0.9999 | Exact=0.0000 | Blank=83.12% | 2.5 min
Epoch 4/5 | loss=2.9091 | CER=0.7869 | WER=0.9993 | Exact=0.0000 | Blank=84.50% | 2.5 min
Epoch 5/5 | loss=2.9075 | CER=0.7686 | WER=0.9997 | Exact=0.0000 | Blank=82.19% | 2.5 min
  بهترین checkpoint جدید ذخیره شد.

پایان آموزش تکمیلی
Best validation CER: 0.7685686800668321
Checkpoint: /content/PFMS_ABLATION_FINAL/mambavision_unfrozen_patch_embed/best_checkpoint.pt
Metrics: /content/PFMS_ABLATION_FINAL/mambavision_unfrozen_patch_embed/epoch_metrics.csv


In [45]:

import os, copy, torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt_path = (
    "/content/PFMS_ABLATION_FINAL/"
    "mambavision_unfrozen_patch_embed/best_checkpoint.pt"
)
assert os.path.isfile(ckpt_path), f"Checkpoint not found: {ckpt_path}"

# بازیابی بهترین وزن‌های موجود
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
net.load_state_dict(ckpt["model_state_dict"])
net = net.to(device)

# جمع‌آوری ۸ نمونه ثابت از train
images_list, targets_list, lengths_list = [], [], []
for batch in train_loader:
    imgs, targets, lengths = batch
    images_list.append(imgs)
    targets_list.append(targets)
    lengths_list.append(lengths)
    if sum(x.size(0) for x in images_list) >= 8:
        break

images = torch.cat(images_list, dim=0)[:8].to(device)
lengths = torch.cat(lengths_list, dim=0)[:8].long().cpu()

# بازسازی targets تخت برای دقیقاً همان ۸ نمونه
target_parts = []
for batch_targets, batch_lengths in zip(targets_list, lengths_list):
    offset = 0
    for n in batch_lengths.tolist():
        target_parts.append(batch_targets[offset:offset+n])
        offset += n
targets = torch.cat(target_parts[:8]).to(device)

print("Images:", tuple(images.shape))
print("Target lengths:", lengths.tolist())
print("Target IDs min/max:", int(targets.min()), int(targets.max()))
print("Contains CTC blank in targets:", bool((targets == 0).any()))

# کپی مستقل؛ checkpoint و مدل اصلی حفظ می‌شوند
probe = copy.deepcopy(net).to(device)

for p in probe.encoder.parameters():
    p.requires_grad = False
for p in probe.encoder.patch_embed.parameters():
    p.requires_grad = True
for p in probe.encoder.levels[0].parameters():
    p.requires_grad = True
for module in (probe.projection, probe.refine, probe.classifier):
    for p in module.parameters():
        p.requires_grad = True

optimizer = torch.optim.AdamW([
    {"params": probe.encoder.patch_embed.parameters(), "lr": 2e-6},
    {"params": probe.encoder.levels[0].parameters(), "lr": 1e-5},
    {"params": list(probe.projection.parameters()) +
                list(probe.refine.parameters()) +
                list(probe.classifier.parameters()), "lr": 3e-4},
], weight_decay=1e-5)

criterion = nn.CTCLoss(blank=0, zero_infinity=True)
input_lengths = torch.full((8,), 128, dtype=torch.long)

for step in range(101):
    probe.train()
    probe.encoder.patch_embed.train()
    probe.encoder.levels[0].train()

    optimizer.zero_grad(set_to_none=True)
    logits = probe(images)
    assert logits.ndim == 3 and logits.shape[0] == 8
    assert int(lengths.max()) <= logits.shape[1], (
        f"Target length {int(lengths.max())} exceeds output steps {logits.shape[1]}"
    )

    loss = criterion(
        logits.log_softmax(-1).transpose(0, 1),
        targets,
        torch.full((8,), logits.shape[1], dtype=torch.long),
        lengths
    )
    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite CTC loss at step {step}: {loss.item()}")

    if step < 100:
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in probe.parameters() if p.requires_grad], 1.0
        )
        optimizer.step()

    if step in (0, 25, 50, 75, 100):
        probe.eval()
        with torch.no_grad():
            pred = probe(images).argmax(-1)
            blank_ratio = (pred == 0).float().mean().item()
        print(
            f"Step {step:03d} | CTC loss={loss.item():.4f} | "
            f"blank ratio={blank_ratio:.2%}"
        )

# نمایش پیش‌بینی‌های نهایی روی همان نمونه‌ها
probe.eval()
with torch.no_grad():
    pred = probe(images).argmax(-1).cpu()

def show_ctc(ids):
    out, prev = [], None
    for t in ids:
        t = int(t)
        if t != prev and t not in (0, 1, 2, 3):
            out.append(idx2char.get(t, "�"))
        prev = t
    return "".join(out)

offset = 0
for i, n in enumerate(lengths.tolist()):
    ref = "".join(
        idx2char.get(int(t), "�")
        for t in targets[offset:offset+n].cpu().tolist()
        if int(t) not in (0, 1, 2, 3)
    )
    offset += n
    print(f"\nنمونه {i+1}")
    print("واقعی:", ref[:180])
    print("پیش‌بینی:", show_ctc(pred[i].tolist())[:180])

print("\nآزمون تمام شد؛ مدل اصلی و checkpoint ذخیره‌شده بازنویسی نشدند.")


Images: (8, 3, 64, 1024)
Target lengths: [78, 86, 69, 71, 90, 72, 62, 75]
Target IDs min/max: 4 102
Contains CTC blank in targets: False
Step 000 | CTC loss=2.8385 | blank ratio=83.11%
Step 025 | CTC loss=2.4791 | blank ratio=72.46%
Step 050 | CTC loss=2.0799 | blank ratio=61.72%
Step 075 | CTC loss=1.6152 | blank ratio=55.57%
Step 100 | CTC loss=1.2513 | blank ratio=51.07%

نمونه 1
واقعی: به شدت شکست خورده است. فیشمن افزود: این تهران بود که پیروز شد. روزنامه هاآرتص

پیش‌بینی: اه شدت ادست اده اس. فیشور اسزاد: ایر تهران بو که ای اه روزاین رابرتص


نمونه 2
واقعی: اخوان المسلمین بودند با نیروهای امنیتی به شدت درگیر شدند. این دانشجویان روز دوشنبه را

پیش‌بینی: اخون اشسلمین بودند با ایروا امنیدی به شدت درگیر شدند. رین رانوان روز برب را


نمونه 3
واقعی: زنگ زده‌ام و نه مراجعه کرده‌ام و در واقع تلاش وی برای فرار از مجازات

پیش‌بینی: زنگ زده‌ام و ه مااه کرده‌ام و ار اع بااش وی برار ارار از دازات


نمونه 4
واقعی: شعبده و جادوی آنها به محیط بنگرند. ملت رشید ایران می داند كه دوست كیست

پیش‌بینی: 

In [46]:

import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
probe = probe.to(device).eval()

# همان ۸ نمونهٔ آزمون قبلی
with torch.no_grad():
    logits = probe(images.to(device))

B, T, V = logits.shape
target_lengths_cpu = lengths.detach().cpu().long()
input_lengths_cpu = torch.full((B,), T, dtype=torch.long)

# حداقل گام‌های لازم CTC = طول هدف + تعداد تکرارهای متوالی در هدف
offset = 0
required_steps = []
for n in target_lengths_cpu.tolist():
    seq = targets[offset:offset+n].detach().cpu().tolist()
    repeats = sum(seq[i] == seq[i-1] for i in range(1, len(seq)))
    required_steps.append(len(seq) + repeats)
    offset += n

print("Logits shape (B,T,V):", tuple(logits.shape))
print("Output steps:", T)
print("Target lengths:", target_lengths_cpu.tolist())
print("Minimum CTC steps required:", required_steps)
print("Any target exceeds available steps:",
      any(x > T for x in required_steps))

# مقایسهٔ loss با zero_infinity روشن/خاموش
log_probs = logits.log_softmax(-1).transpose(0, 1)
loss_safe = nn.CTCLoss(blank=0, zero_infinity=True)(
    log_probs, targets.to(device), input_lengths_cpu, target_lengths_cpu
)
loss_strict = nn.CTCLoss(blank=0, zero_infinity=False)(
    log_probs, targets.to(device), input_lengths_cpu, target_lengths_cpu
)

print("CTC loss (zero_infinity=True):", float(loss_safe))
print("CTC loss (zero_infinity=False):", float(loss_strict))
print("Finite strict loss:", bool(torch.isfinite(loss_strict)))


Logits shape (B,T,V): (8, 128, 169)
Output steps: 128
Target lengths: [78, 86, 69, 71, 90, 72, 62, 75]
Minimum CTC steps required: [78, 86, 69, 71, 92, 74, 62, 75]
Any target exceeds available steps: False
CTC loss (zero_infinity=True): 1.2387243509292603
CTC loss (zero_infinity=False): 1.2387243509292603
Finite strict loss: True


In [47]:

# گزارش وضعیت فعلی برای آماده‌سازی آموزش نهایی MambaVision-only
import os, torch

print("=== Runtime ===")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\n=== Variables in memory ===")
for name in [
    "net", "probe", "train_loader", "val_loader",
    "train_ds", "val_ds", "char2idx", "idx2char",
    "MambaVisionCTCFinal"
]:
    exists = name in globals()
    print(f"{name}: {'available' if exists else 'not found'}")
    if exists and name.endswith("_loader"):
        obj = globals()[name]
        print("  batches:", len(obj))
        print("  batch_size:", getattr(obj, "batch_size", None))

print("\n=== Existing checkpoints ===")
paths = [
    "/content/PFMS_ABLATION_FINAL/mambavision_pretrained_full/best_checkpoint.pt",
    "/content/PFMS_ABLATION_FINAL/mambavision_unfrozen_patch_embed/best_checkpoint.pt",
]
for p in paths:
    if os.path.isfile(p):
        print(f"FOUND | {os.path.getsize(p)/1024**2:.1f} MB | {p}")
    else:
        print("NOT FOUND |", p)

print("\n=== Fixed split manifest ===")
manifest_root = "/content/PFMS_ABLATION_FINAL/run_20261027_093203"
if os.path.isdir(manifest_root):
    for root, dirs, files in os.walk(manifest_root):
        for f in files:
            if f.endswith((".json", ".csv", ".txt")):
                print(os.path.join(root, f))
else:
    print("Manifest directory not found:", manifest_root)

print("\nInspection complete. No training performed; no files modified.")


=== Runtime ===
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4

=== Variables in memory ===
net: available
probe: available
train_loader: available
  batches: 5187
  batch_size: 4
val_loader: available
  batches: 577
  batch_size: 4
train_ds: available
val_ds: available
char2idx: available
idx2char: available
MambaVisionCTCFinal: available

=== Existing checkpoints ===
FOUND | 128.6 MB | /content/PFMS_ABLATION_FINAL/mambavision_pretrained_full/best_checkpoint.pt
FOUND | 128.8 MB | /content/PFMS_ABLATION_FINAL/mambavision_unfrozen_patch_embed/best_checkpoint.pt

=== Fixed split manifest ===
/content/PFMS_ABLATION_FINAL/run_20261027_093203/train_manifest.csv
/content/PFMS_ABLATION_FINAL/run_20261027_093203/test_manifest.csv
/content/PFMS_ABLATION_FINAL/run_20261027_093203/validation_manifest.csv
/content/PFMS_ABLATION_FINAL/run_20261027_093203/run_manifest.json

Inspection complete. No training performed; no files modified.


In [48]:

import torch
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_check = net.to(device).eval()

batch = next(iter(train_loader))
images_check, targets_check, lengths_check = batch
images_check = images_check.to(device)

with torch.no_grad():
    logits_check = model_check(images_check)
    probs_check = logits_check.softmax(-1)
    pred_check = logits_check.argmax(-1)

print("Logits:", tuple(logits_check.shape))
print("Logits finite:", bool(torch.isfinite(logits_check).all()))
print("Target lengths:", lengths_check.tolist())
print("Target ID range:", int(targets_check.min()), int(targets_check.max()))
print("Target contains blank 0:", bool((targets_check == 0).any()))

print("\nMean probability of blank class:",
      round(probs_check[..., 0].mean().item(), 6))
print("Blank argmax ratio:",
      round((pred_check == 0).float().mean().item(), 4))
print("Unique predicted IDs:",
      len(torch.unique(pred_check)))
print("Most common predicted IDs:",
      Counter(pred_check.cpu().flatten().tolist()).most_common(15))
print("Most common target IDs:",
      Counter(targets_check.cpu().tolist()).most_common(15))

print("\nInspection only; no training and no files modified.")


Logits: (4, 128, 169)
Logits finite: True
Target lengths: [77, 86, 69, 79]
Target ID range: 4 98
Target contains blank 0: False

Mean probability of blank class: 0.401278
Blank argmax ratio: 0.8457
Unique predicted IDs: 4
Most common predicted IDs: [(0, 433), (5, 50), (35, 25), (4, 4)]
Most common target IDs: [(5, 56), (35, 36), (90, 30), (45, 30), (63, 19), (43, 17), (61, 15), (62, 12), (88, 9), (60, 8), (36, 8), (47, 6), (59, 6), (46, 6), (13, 6)]

Inspection only; no training and no files modified.


In [49]:
# CELL 1 — Parallel Swin + REAL MambaVision + Bidirectional Cross-Attention
import os, argparse, random, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
import timm

from mambavision import create_model as create_mambavision

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR = "/content/PFMS_ABLATION_FINAL/parallel_cross_attention"
os.makedirs(OUT_DIR, exist_ok=True)

# Use the vocabulary already loaded in your current Colab runtime.
if "char2idx" not in globals():
    raise RuntimeError("متغیر char2idx در Runtime موجود نیست؛ سلول واژگان قبلی را اجرا کنید.")

VOCAB_SIZE = len(char2idx)
PAD_ID = 0
CTC_BLANK = 0
D_MODEL = 256
NUM_HEADS = 8

# Allow loading the trusted pretrained MambaVision checkpoint on this runtime.
torch.serialization.add_safe_globals([argparse.Namespace])

class BidirectionalCrossAttention(nn.Module):
    def __init__(self, dim=256, heads=8, dropout=0.1):
        super().__init__()
        self.swin_from_mamba = nn.MultiheadAttention(
            dim, heads, dropout=dropout, batch_first=True
        )
        self.mamba_from_swin = nn.MultiheadAttention(
            dim, heads, dropout=dropout, batch_first=True
        )
        self.norm_s = nn.LayerNorm(dim)
        self.norm_m = nn.LayerNorm(dim)
        self.norm_out = nn.LayerNorm(dim)
        self.fuse = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim)
        )

    def forward(self, s, m):
        s_update, _ = self.swin_from_mamba(s, m, m, need_weights=False)
        m_update, _ = self.mamba_from_swin(m, s, s, need_weights=False)
        s = self.norm_s(s + s_update)
        m = self.norm_m(m + m_update)
        # Both branches are retained; output has the shared sequence length.
        return self.norm_out(self.fuse(torch.cat([s, m], dim=-1)))

class PFMSParallelCTC(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        # Real pretrained Swin-B; intermediate hierarchical feature stage.
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            out_indices=(1,)
        )
        swin_dim = self.swin.feature_info.channels()[0]

        # Real pretrained MambaVision-T, not a placeholder.
        self.mamba = create_mambavision(
            "mamba_vision_T", pretrained=True
        )

        self.swin_proj = nn.Conv1d(swin_dim, D_MODEL, kernel_size=1)
        self.mamba_proj = nn.Conv1d(160, D_MODEL, kernel_size=1)

        self.cross_attention = BidirectionalCrossAttention(
            D_MODEL, NUM_HEADS
        )
        self.refine = nn.Sequential(
            nn.Conv1d(D_MODEL, D_MODEL, 3, padding=1),
            nn.GELU(),
            nn.Dropout(0.1)
        )
        self.classifier = nn.Linear(D_MODEL, vocab_size)

        # Freeze pretrained encoders for a faster, memory-efficient first run.
        for p in self.swin.parameters():
            p.requires_grad = False
        for p in self.mamba.parameters():
            p.requires_grad = False

    def forward(self, images):
        # Swin features: [B,C,H,W] -> [B,T,C]
        with torch.no_grad():
            sf = self.swin(images)[0]
            mf = self.mamba.patch_embed(images)
            mf = self.mamba.levels[0](mf)

        s = self.swin_proj(sf.mean(dim=2)).transpose(1, 2)
        m = self.mamba_proj(mf.mean(dim=2)).transpose(1, 2)

        # Match temporal lengths if the selected backbones differ.
        if s.size(1) != m.size(1):
            m = F.interpolate(
                m.transpose(1, 2), size=s.size(1),
                mode="linear", align_corners=False
            ).transpose(1, 2)

        z = self.cross_attention(s, m)
        z = z + self.refine(z.transpose(1, 2)).transpose(1, 2)
        return self.classifier(z)  # [B,T,V]

net = PFMSParallelCTC(VOCAB_SIZE).to(DEVICE)
trainable = sum(p.numel() for p in net.parameters() if p.requires_grad)
total = sum(p.numel() for p in net.parameters())

print("Device:", DEVICE)
print("Vocabulary:", VOCAB_SIZE)
print(f"Parameters: {total:,} total | {trainable:,} trainable")
print("Architecture: Swin-B || MambaVision-T -> Bidirectional Cross-Attention -> CTC")

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Device: cuda
Vocabulary: 169
Parameters: 34,985,289 total | 1,072,297 trainable
Architecture: Swin-B || MambaVision-T -> Bidirectional Cross-Attention -> CTC


In [52]:
class PFMSParallelCTC(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        # Keep OCR's wide aspect ratio; allow dynamic image sizes.
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            out_indices=(1,),
            strict_img_size=False,
            dynamic_img_size=True,
        )
        swin_dim = self.swin.feature_info.channels()[0]

        self.mamba = create_mambavision(
            "mamba_vision_T", pretrained=True
        )

        self.swin_proj = nn.Conv1d(swin_dim, 256, kernel_size=1)
        self.mamba_proj = nn.Conv1d(160, 256, kernel_size=1)

        self.cross_attention = BidirectionalCrossAttention(256, 8)

        self.refine = nn.Sequential(
            nn.Conv1d(256, 256, 3, padding=1),
            nn.GELU(),
            nn.Dropout(0.1),
        )
        self.classifier = nn.Linear(256, vocab_size)

        for p in self.swin.parameters():
            p.requires_grad = False
        for p in self.mamba.parameters():
            p.requires_grad = False

    def forward(self, images):
        with torch.no_grad():
            sf = self.swin(images)[0]

            mf = self.mamba.patch_embed(images)
            mf = self.mamba.levels[0](mf)

        s = self.swin_proj(sf.mean(dim=2)).transpose(1, 2)
        m = self.mamba_proj(mf.mean(dim=2)).transpose(1, 2)

        if s.size(1) != m.size(1):
            m = F.interpolate(
                m.transpose(1, 2),
                size=s.size(1),
                mode="linear",
                align_corners=False,
            ).transpose(1, 2)

        z = self.cross_attention(s, m)
        z = z + self.refine(z.transpose(1, 2)).transpose(1, 2)
        return self.classifier(z)

In [61]:

import torch
import torch.nn.functional as F

def corrected_forward(self, images):
    with torch.no_grad():
        swin_features = self.swin(images)[0]
        mamba_features = self.mamba.patch_embed(images)
        mamba_features = self.mamba.levels[0](mamba_features)

    # Swin output is channels-last: [B, H, W, C].
    if swin_features.ndim != 4:
        raise RuntimeError(
            f"Unexpected Swin shape: {tuple(swin_features.shape)}"
        )

    # Average height, then convert [B, W, C] -> [B, C, W].
    s = swin_features.mean(dim=1).transpose(1, 2)
    s = self.swin_proj(s).transpose(1, 2)

    # MambaVision stage-1 output is channels-first: [B, C, H, W].
    if mamba_features.ndim != 4:
        raise RuntimeError(
            f"Unexpected MambaVision shape: {tuple(mamba_features.shape)}"
        )

    m = mamba_features.mean(dim=2)
    m = self.mamba_proj(m).transpose(1, 2)

    # Align horizontal sequence lengths.
    if s.size(1) != m.size(1):
        m = F.interpolate(
            m.transpose(1, 2),
            size=s.size(1),
            mode="linear",
            align_corners=False,
        ).transpose(1, 2)

    fused = self.cross(s, m)
    fused = fused + self.refine(
        fused.transpose(1, 2)
    ).transpose(1, 2)

    return self.classifier(fused)


# Update the existing model class without rebuilding its weights.
PFMSParallelCTC.forward = corrected_forward

# Test forward and CTC before training.
model.eval()
test_images, test_targets, test_lengths = unpack_batch(
    next(iter(train_loader))
)

with torch.no_grad():
    test_logits = model(test_images[:1])

first_target_length = int(test_lengths[0].item())
test_loss = calculate_ctc(
    test_logits,
    test_targets[:first_target_length],
    test_lengths[:1],
)

print("Forward test passed:", tuple(test_logits.shape))
print("CTC test passed:", float(test_loss.item()))


Forward test passed: (1, 128, 169)
CTC test passed: 8.233281135559082


In [62]:

# ============================================================
# PFMS-Net parallel OCR — ONE COMPLETE COLAB CELL
# Swin + REAL MambaVision + Bidirectional Cross-Attention + CTC
# Uses existing train_loader, val_loader, char2idx
# Batch format: (images, flat_targets, target_lengths)
# ============================================================

import os
import time
import csv
import random
import argparse
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from mambavision import create_model as create_mambavision

# ---------------- Configuration ----------------
SEED = 42
EPOCHS = 3
LR = 1e-4
D_MODEL = 256
NUM_HEADS = 8
CTC_BLANK = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR = "/content/PFMS_ABLATION_FINAL/parallel_complete_run"
os.makedirs(OUT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

for required_name in ("train_loader", "val_loader", "char2idx"):
    if required_name not in globals():
        raise RuntimeError(
            f"Missing {required_name}. Run the existing data setup first."
        )

VOCAB_SIZE = len(char2idx)
torch.serialization.add_safe_globals([argparse.Namespace])

print("Device:", DEVICE)
print("Vocabulary:", VOCAB_SIZE)
print("Output:", OUT_DIR)


# ---------------- Bidirectional Cross-Attention ----------------
class BidirectionalCrossAttention(nn.Module):
    def __init__(self, dim=256, heads=8, dropout=0.1):
        super().__init__()
        self.s_to_m = nn.MultiheadAttention(
            dim, heads, dropout=dropout, batch_first=True
        )
        self.m_to_s = nn.MultiheadAttention(
            dim, heads, dropout=dropout, batch_first=True
        )
        self.norm_s = nn.LayerNorm(dim)
        self.norm_m = nn.LayerNorm(dim)
        self.norm_out = nn.LayerNorm(dim)
        self.fusion = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, s, m):
        s_update, _ = self.s_to_m(s, m, m, need_weights=False)
        m_update, _ = self.m_to_s(m, s, s, need_weights=False)
        s = self.norm_s(s + s_update)
        m = self.norm_m(m + m_update)
        return self.norm_out(self.fusion(torch.cat((s, m), dim=-1)))


# ---------------- Parallel OCR model ----------------
class PFMSParallelCTC(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        # Pretrained Swin; preserve wide OCR image dimensions.
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            out_indices=(1,),
            img_size=(64, 1024),
            strict_img_size=False,
        )
        swin_channels = self.swin.feature_info.channels()[0]

        # Actual pretrained MambaVision-T.
        self.mamba = create_mambavision(
            "mamba_vision_T",
            pretrained=True,
        )

        self.swin_proj = nn.Conv1d(swin_channels, D_MODEL, kernel_size=1)
        self.mamba_proj = nn.Conv1d(160, D_MODEL, kernel_size=1)
        self.cross = BidirectionalCrossAttention(D_MODEL, NUM_HEADS)

        self.refine = nn.Sequential(
            nn.Conv1d(D_MODEL, D_MODEL, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Dropout(0.1),
        )
        self.classifier = nn.Linear(D_MODEL, vocab_size)

        # Initial efficient experiment: train fusion and OCR output head.
        for parameter in self.swin.parameters():
            parameter.requires_grad = False
        for parameter in self.mamba.parameters():
            parameter.requires_grad = False

    def forward(self, images):
        with torch.no_grad():
            swin_features = self.swin(images)[0]
            mamba_features = self.mamba.patch_embed(images)
            mamba_features = self.mamba.levels[0](mamba_features)

        if swin_features.ndim != 4 or mamba_features.ndim != 4:
            raise RuntimeError(
                f"Unexpected feature shapes: Swin={tuple(swin_features.shape)}, "
                f"MambaVision={tuple(mamba_features.shape)}"
            )

        # Average feature-map height; retain horizontal OCR sequence.
        s = self.swin_proj(swin_features.mean(dim=2)).transpose(1, 2)
        m = self.mamba_proj(mamba_features.mean(dim=2)).transpose(1, 2)

        # Align sequence lengths.
        if s.size(1) != m.size(1):
            m = F.interpolate(
                m.transpose(1, 2),
                size=s.size(1),
                mode="linear",
                align_corners=False,
            ).transpose(1, 2)

        fused = self.cross(s, m)
        fused = fused + self.refine(fused.transpose(1, 2)).transpose(1, 2)
        return self.classifier(fused)


# ---------------- Batch parsing ----------------
def unpack_batch(batch):
    if not isinstance(batch, (tuple, list)) or len(batch) != 3:
        raise TypeError(
            "Expected each batch as (images, flat_targets, target_lengths)."
        )

    images, flat_targets, target_lengths = batch

    images = torch.as_tensor(
        images, dtype=torch.float32, device=DEVICE
    )
    flat_targets = torch.as_tensor(
        flat_targets, dtype=torch.long, device=DEVICE
    )
    target_lengths = torch.as_tensor(
        target_lengths, dtype=torch.long, device=DEVICE
    )

    if images.ndim != 4:
        raise ValueError(f"Expected images [B,C,H,W], got {tuple(images.shape)}")
    if flat_targets.ndim != 1 or target_lengths.ndim != 1:
        raise ValueError("Targets must be flat and lengths must be one-dimensional.")
    if images.size(0) != target_lengths.numel():
        raise ValueError("Image batch size and target-length count differ.")
    if torch.any(target_lengths <= 0):
        raise ValueError("A target has zero length.")
    if int(target_lengths.sum().item()) != flat_targets.numel():
        raise ValueError(
            f"Target lengths sum to {int(target_lengths.sum().item())}, "
            f"but flattened target count is {flat_targets.numel()}."
        )

    return images, flat_targets, target_lengths


# ---------------- CTC loss ----------------
ctc_loss_fn = nn.CTCLoss(blank=CTC_BLANK, zero_infinity=True)


def calculate_ctc(logits, flat_targets, target_lengths):
    batch_size, time_steps, vocab_size = logits.shape

    if vocab_size != VOCAB_SIZE:
        raise ValueError("Logit vocabulary dimension is incorrect.")
    if target_lengths.numel() != batch_size:
        raise ValueError("Target-length count does not match batch size.")
    if int(target_lengths.sum().item()) != flat_targets.numel():
        raise ValueError("Flattened target count does not match target lengths.")

    offset = 0
    for i in range(batch_size):
        length = int(target_lengths[i].item())
        sequence = flat_targets[offset:offset + length]
        offset += length

        minimum_frames = length
        if length > 1:
            minimum_frames += int(
                (sequence[1:] == sequence[:-1]).sum().item()
            )

        if minimum_frames > time_steps:
            raise ValueError(
                f"Sample {i}: minimum CTC frames={minimum_frames}; "
                f"available frames={time_steps}."
            )

    log_probs = logits.float().log_softmax(dim=-1).transpose(0, 1)
    input_lengths = torch.full(
        (batch_size,),
        time_steps,
        dtype=torch.long,
        device=DEVICE,
    )

    return ctc_loss_fn(
        log_probs,
        flat_targets,
        input_lengths,
        target_lengths,
    )


# ---------------- Build and test before training ----------------
model = PFMSParallelCTC(VOCAB_SIZE).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

first_batch = next(iter(train_loader))
test_images, test_targets, test_lengths = unpack_batch(first_batch)

print("Image batch:", tuple(test_images.shape))
print("Flat target shape:", tuple(test_targets.shape))
print("Target lengths:", test_lengths.tolist())

model.eval()
with torch.no_grad():
    test_logits = model(test_images[:1])

test_loss = calculate_ctc(
    test_logits,
    test_targets[:int(test_lengths[0].item())],
    test_lengths[:1],
)

print("Forward test passed:", tuple(test_logits.shape))
print("CTC test passed; loss =", float(test_loss.item()))


# ---------------- Optimizer ----------------
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=0.01,
)

metrics_path = os.path.join(OUT_DIR, "metrics.csv")
checkpoint_path = os.path.join(OUT_DIR, "best_parallel_checkpoint.pt")
best_val_loss = float("inf")


# ---------------- Validation ----------------
@torch.no_grad()
def validate_model():
    model.eval()
    model.swin.eval()
    model.mamba.eval()

    loss_sum = 0.0
    batch_count = 0

    for batch in val_loader:
        images, flat_targets, target_lengths = unpack_batch(batch)
        logits = model(images)
        loss = calculate_ctc(logits, flat_targets, target_lengths)

        if torch.isfinite(loss):
            loss_sum += float(loss.item())
            batch_count += 1

    if batch_count == 0:
        raise RuntimeError("No valid validation batches.")
    return loss_sum / batch_count


# ---------------- Training ----------------
with open(metrics_path, "w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(["epoch", "train_ctc_loss", "val_ctc_loss", "seconds"])

    for epoch in range(1, EPOCHS + 1):
        model.train()
        model.swin.eval()
        model.mamba.eval()

        running_loss = 0.0
        steps = 0
        start_time = time.time()

        for batch in train_loader:
            images, flat_targets, target_lengths = unpack_batch(batch)
            optimizer.zero_grad(set_to_none=True)

            logits = model(images)
            loss = calculate_ctc(logits, flat_targets, target_lengths)

            if not torch.isfinite(loss):
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                max_norm=1.0,
            )
            optimizer.step()

            running_loss += float(loss.item())
            steps += 1

        if steps == 0:
            raise RuntimeError("No valid training batches.")

        train_loss = running_loss / steps
        val_loss = validate_model()
        elapsed = time.time() - start_time

        print(
            f"Epoch {epoch}/{EPOCHS} | "
            f"train CTC={train_loss:.4f} | "
            f"val CTC={val_loss:.4f} | "
            f"time={elapsed / 60:.1f} min"
        )

        writer.writerow([epoch, train_loss, val_loss, round(elapsed, 2)])
        csv_file.flush()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "train_ctc_loss": train_loss,
                    "val_ctc_loss": val_loss,
                    "vocab_size": VOCAB_SIZE,
                    "blank_id": CTC_BLANK,
                    "seed": SEED,
                    "architecture": (
                        "Swin-B + MambaVision-T + "
                        "Bidirectional Cross-Attention + CTC"
                    ),
                },
                checkpoint_path,
            )

print("\nTraining completed.")
print("Best validation CTC loss:", best_val_loss)
print("Metrics file:", metrics_path)
print("Checkpoint:", checkpoint_path)
print("Do not report CER/WER until the held-out test set is decoded and evaluated.")


Device: cuda
Vocabulary: 169
Output: /content/PFMS_ABLATION_FINAL/parallel_complete_run
Parameters: 34,919,497
Trainable parameters: 1,006,505
Image batch: (4, 3, 64, 1024)
Flat target shape: (340,)
Target lengths: [76, 78, 82, 104]


RuntimeError: Given groups=1, weight of size [256, 256, 1], expected input[1, 8, 256] to have 256 channels, but got 8 channels instead

In [63]:

# FIX: replace the old forward method on the existing model

def fixed_pfms_forward(self, images):
    with torch.no_grad():
        sf = self.swin(images)[0]
        mf = self.mamba.patch_embed(images)
        mf = self.mamba.levels[0](mf)

    print("Raw Swin shape:", tuple(sf.shape))
    print("Raw MambaVision shape:", tuple(mf.shape))

    # Swin features may be channels-last [B,H,W,C].
    # Convert them to Conv1d input [B,C,T].
    if sf.ndim != 4:
        raise RuntimeError(f"Unexpected Swin shape: {tuple(sf.shape)}")

    if sf.shape[-1] == self.swin_proj.in_channels:
        s = sf.mean(dim=1).transpose(1, 2)
    elif sf.shape[1] == self.swin_proj.in_channels:
        s = sf.mean(dim=2)
    else:
        raise RuntimeError(
            f"Cannot identify Swin channel dimension in {tuple(sf.shape)}; "
            f"expected {self.swin_proj.in_channels} channels."
        )

    s = self.swin_proj(s).transpose(1, 2)

    # MambaVision stage output: [B,C,H,W].
    if mf.ndim != 4 or mf.shape[1] != self.mamba_proj.in_channels:
        raise RuntimeError(
            f"Unexpected MambaVision shape: {tuple(mf.shape)}; "
            f"expected channel count {self.mamba_proj.in_channels}."
        )

    m = self.mamba_proj(mf.mean(dim=2)).transpose(1, 2)

    if s.size(1) != m.size(1):
        m = F.interpolate(
            m.transpose(1, 2),
            size=s.size(1),
            mode="linear",
            align_corners=False,
        ).transpose(1, 2)

    z = self.cross(s, m)
    z = z + self.refine(z.transpose(1, 2)).transpose(1, 2)
    return self.classifier(z)


PFMSParallelCTC.forward = fixed_pfms_forward

# Run a forward-only test; do not start training yet.
model.eval()
with torch.no_grad():
    logits = model(test_images[:1])

print("Forward passed:", tuple(logits.shape))


Raw Swin shape: (1, 8, 128, 256)
Raw MambaVision shape: (1, 160, 8, 128)
Forward passed: (1, 128, 169)


In [64]:

# CTC smoke test — run after the successful forward test
model.eval()

test_images, test_targets, test_lengths = unpack_batch(
    next(iter(train_loader))
)

with torch.no_grad():
    logits = model(test_images[:1])

first_length = int(test_lengths[0].item())

loss = calculate_ctc(
    logits,
    test_targets[:first_length],
    test_lengths[:1],
)

print("CTC test passed:", bool(torch.isfinite(loss).item()))
print("CTC loss:", float(loss.item()))


Raw Swin shape: (1, 8, 128, 256)
Raw MambaVision shape: (1, 160, 8, 128)
CTC test passed: True
CTC loss: 8.233281135559082


In [67]:

import os

folder = "/content/PFMS_ABLATION_FINAL/parallel_complete_run"

for filename in ("metrics.csv", "best_parallel_checkpoint.pt"):
    path = os.path.join(folder, filename)
    print(
        filename,
        "| exists:", os.path.exists(path),
        "| bytes:", os.path.getsize(path) if os.path.exists(path) else 0,
    )

metrics = os.path.join(folder, "metrics.csv")
if os.path.exists(metrics):
    with open(metrics, "r", encoding="utf-8") as f:
        print("\n".join(f.readlines()[-5:]))


metrics.csv | exists: True | bytes: 43
best_parallel_checkpoint.pt | exists: False | bytes: 0
epoch,train_ctc_loss,val_ctc_loss,seconds



In [68]:

# Replace the diagnostic forward with a silent version.
def silent_pfms_forward(self, images):
    with torch.no_grad():
        sf = self.swin(images)[0]
        mf = self.mamba.patch_embed(images)
        mf = self.mamba.levels[0](mf)

    if sf.ndim != 4 or mf.ndim != 4:
        raise RuntimeError("Unexpected encoder feature dimensions.")

    if sf.shape[-1] == self.swin_proj.in_channels:
        s = sf.mean(dim=1).transpose(1, 2)
    elif sf.shape[1] == self.swin_proj.in_channels:
        s = sf.mean(dim=2)
    else:
        raise RuntimeError(f"Unexpected Swin shape: {tuple(sf.shape)}")

    s = self.swin_proj(s).transpose(1, 2)
    m = self.mamba_proj(mf.mean(dim=2)).transpose(1, 2)

    if s.size(1) != m.size(1):
        m = F.interpolate(
            m.transpose(1, 2),
            size=s.size(1),
            mode="linear",
            align_corners=False,
        ).transpose(1, 2)

    z = self.cross(s, m)
    z = z + self.refine(z.transpose(1, 2)).transpose(1, 2)
    return self.classifier(z)

PFMSParallelCTC.forward = silent_pfms_forward
print("Diagnostic printing disabled.")


Diagnostic printing disabled.


In [69]:

# Short, measurable training test — no full-epoch run
import os
import time
import torch

assert "model" in globals(), "مدل موجود نیست؛ فعلاً مدل را از ابتدا نسازید."
assert "train_loader" in globals(), "train_loader موجود نیست."
assert "unpack_batch" in globals() and "calculate_ctc" in globals()

pilot_optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4,
    weight_decay=0.01,
)

model.train()
model.swin.eval()
model.mamba.eval()

tracked = next(p for p in model.parameters() if p.requires_grad)
before = tracked.detach().clone()
loss_values = []
start = time.time()

for step, batch in enumerate(train_loader):
    if step >= 5:
        break

    images, flat_targets, target_lengths = unpack_batch(batch)
    pilot_optimizer.zero_grad(set_to_none=True)

    logits = model(images)
    loss = calculate_ctc(logits, flat_targets, target_lengths)

    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite CTC loss at pilot step {step + 1}")

    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad],
        max_norm=1.0,
    )
    pilot_optimizer.step()

    loss_values.append(float(loss.item()))
    print(
        f"Pilot batch {step + 1}/5 | "
        f"CTC loss={loss.item():.4f} | "
        f"elapsed={(time.time() - start) / 60:.2f} min"
    )

after = tracked.detach()
changed = not torch.equal(before, after)

if len(loss_values) != 5:
    raise RuntimeError(f"Only {len(loss_values)} batches completed.")

pilot_path = "/content/PFMS_ABLATION_FINAL/parallel_complete_run/pilot_checkpoint.pt"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": pilot_optimizer.state_dict(),
        "pilot_losses": loss_values,
        "completed_batches": len(loss_values),
        "weights_changed": changed,
    },
    pilot_path,
)

print("\nPilot completed.")
print("Mean CTC loss:", sum(loss_values) / len(loss_values))
print("Trainable weights changed:", changed)
print("Pilot checkpoint:", pilot_path)
print("Elapsed minutes:", round((time.time() - start) / 60, 2))


Pilot batch 1/5 | CTC loss=7.5245 | elapsed=0.00 min
Pilot batch 2/5 | CTC loss=6.8039 | elapsed=0.00 min
Pilot batch 3/5 | CTC loss=6.2829 | elapsed=0.01 min
Pilot batch 4/5 | CTC loss=5.8404 | elapsed=0.01 min
Pilot batch 5/5 | CTC loss=5.2775 | elapsed=0.01 min

Pilot completed.
Mean CTC loss: 6.345823001861572
Trainable weights changed: True
Pilot checkpoint: /content/PFMS_ABLATION_FINAL/parallel_complete_run/pilot_checkpoint.pt
Elapsed minutes: 0.01


In [70]:

# Continue training safely: 3 complete epochs, periodic recovery checkpoints
import os, csv, time, torch

OUT = "/content/PFMS_ABLATION_FINAL/parallel_complete_run"
os.makedirs(OUT, exist_ok=True)
METRICS = os.path.join(OUT, "metrics.csv")
LATEST = os.path.join(OUT, "latest_checkpoint.pt")
BEST = os.path.join(OUT, "best_checkpoint.pt")

assert "model" in globals()
assert "train_loader" in globals() and "val_loader" in globals()
assert "unpack_batch" in globals() and "calculate_ctc" in globals()
assert "pilot_optimizer" in globals()

device = next(model.parameters()).device
best_val = float("inf")
if os.path.exists(METRICS) and os.path.getsize(METRICS) > 50:
    try:
        with open(METRICS, newline="") as f:
            rows = list(csv.DictReader(f))
        previous = [float(r["val_ctc_loss"]) for r in rows
                    if r.get("val_ctc_loss", "").strip()]
        if previous:
            best_val = min(previous)
    except Exception:
        pass

if not os.path.exists(METRICS) or os.path.getsize(METRICS) == 0:
    with open(METRICS, "w", newline="") as f:
        csv.writer(f).writerow(
            ["epoch", "train_ctc_loss", "val_ctc_loss", "seconds"]
        )

for epoch in range(1, 4):
    start = time.time()
    model.train()
    model.swin.eval()
    model.mamba.eval()
    train_sum, train_n = 0.0, 0

    for step, batch in enumerate(train_loader, 1):
        images, targets, target_lengths = unpack_batch(batch)
        images = images.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)

        pilot_optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = calculate_ctc(logits, targets, target_lengths)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite training loss: epoch={epoch}, batch={step}"
            )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], 1.0
        )
        pilot_optimizer.step()

        train_sum += loss.item()
        train_n += 1

        if step % 100 == 0:
            torch.save({
                "epoch": epoch,
                "batch": step,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": pilot_optimizer.state_dict(),
                "train_loss_so_far": train_sum / train_n,
            }, LATEST)
            print(
                f"Epoch {epoch}/3 | batch {step}/{len(train_loader)} "
                f"| mean train CTC={train_sum/train_n:.4f}"
            )

    model.eval()
    val_sum, val_n = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            images, targets, target_lengths = unpack_batch(batch)
            images = images.to(device)
            targets = targets.to(device)
            target_lengths = target_lengths.to(device)
            logits = model(images)
            val_loss = calculate_ctc(logits, targets, target_lengths)

            if torch.isfinite(val_loss):
                val_sum += val_loss.item()
                val_n += 1
            else:
                raise RuntimeError(f"Non-finite validation loss in epoch {epoch}")

    if train_n == 0 or val_n == 0:
        raise RuntimeError("Empty training or validation pass.")

    train_avg = train_sum / train_n
    val_avg = val_sum / val_n
    seconds = time.time() - start

    with open(METRICS, "a", newline="") as f:
        csv.writer(f).writerow(
            [epoch, f"{train_avg:.6f}", f"{val_avg:.6f}", f"{seconds:.1f}"]
        )

    state = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": pilot_optimizer.state_dict(),
        "train_ctc_loss": train_avg,
        "val_ctc_loss": val_avg,
    }
    torch.save(state, LATEST)
    if val_avg < best_val:
        best_val = val_avg
        torch.save(state, BEST)

    print(
        f"\nEPOCH {epoch} COMPLETE | train={train_avg:.4f} "
        f"| val={val_avg:.4f} | time={seconds/60:.1f} min"
    )
    print(f"Latest: {LATEST} | Best: {BEST}\n")

print("3-epoch continuation finished.")
print("Metrics:", METRICS)


Epoch 1/3 | batch 100/5187 | mean train CTC=3.3781
Epoch 1/3 | batch 200/5187 | mean train CTC=3.2309
Epoch 1/3 | batch 300/5187 | mean train CTC=3.1798
Epoch 1/3 | batch 400/5187 | mean train CTC=3.1536
Epoch 1/3 | batch 500/5187 | mean train CTC=3.1353
Epoch 1/3 | batch 600/5187 | mean train CTC=3.1227
Epoch 1/3 | batch 700/5187 | mean train CTC=3.1122
Epoch 1/3 | batch 800/5187 | mean train CTC=3.1070
Epoch 1/3 | batch 900/5187 | mean train CTC=3.0998
Epoch 1/3 | batch 1000/5187 | mean train CTC=3.0961
Epoch 1/3 | batch 1100/5187 | mean train CTC=3.0921
Epoch 1/3 | batch 1200/5187 | mean train CTC=3.0874
Epoch 1/3 | batch 1300/5187 | mean train CTC=3.0863
Epoch 1/3 | batch 1400/5187 | mean train CTC=3.0844
Epoch 1/3 | batch 1500/5187 | mean train CTC=3.0811
Epoch 1/3 | batch 1600/5187 | mean train CTC=3.0784
Epoch 1/3 | batch 1700/5187 | mean train CTC=3.0768
Epoch 1/3 | batch 1800/5187 | mean train CTC=3.0756
Epoch 1/3 | batch 1900/5187 | mean train CTC=3.0740
Epoch 1/3 | batch 200

In [ ]:
import os
import csv
import torch

OUT = "/content/PFMS_ABLATION_FINAL/parallel_complete_run"
METRICS = os.path.join(OUT, "metrics.csv")
LATEST = os.path.join(OUT, "latest_checkpoint.pt")
BEST = os.path.join(OUT, "best_checkpoint.pt")

print("=" * 65)
print("PFMS-Net — READ-ONLY OUTPUT AUDIT")
print("=" * 65)

for path in [METRICS, LATEST, BEST]:
    if os.path.isfile(path):
        print(f"\nFOUND: {path}")
        print(f"Size: {os.path.getsize(path) / (1024**2):.2f} MB")
    else:
        print(f"\nMISSING: {path}")

if os.path.isfile(METRICS):
    with open(METRICS, newline="") as f:
        rows = list(csv.DictReader(f))

    print("\n--- Metrics history ---")
    print("Recorded rows:", len(rows))

    for row in rows:
        print(row)

    if rows and "val_ctc_loss" in rows[0]:
        valid = [
            r for r in rows
            if r.get("val_ctc_loss", "").strip()
        ]
        if valid:
            best_row = min(
                valid, key=lambda r: float(r["val_ctc_loss"])
            )
            print("\nBest recorded validation result:")
            print(best_row)

for path in [LATEST, BEST]:
    if os.path.isfile(path):
        print(f"\n--- Checkpoint inspection: {os.path.basename(path)} ---")
        try:
            checkpoint = torch.load(
                path, map_location="cpu", weights_only=True
            )
            if isinstance(checkpoint, dict):
                for key in checkpoint:
                    value = checkpoint[key]
                    if key == "model_state_dict" and isinstance(value, dict):
                        print(key, "contains", len(value), "state entries")
                    elif key == "optimizer_state_dict":
                        print(key, "present")
                    else:
                        print(f"{key}: {value}")
            else:
                print("Unexpected checkpoint type:", type(checkpoint))
        except Exception as e:
            print("Safe inspection failed:", repr(e))

print("\nAudit complete. No model weights or files were modified.")